In [1]:
library(tidyverse)
library(scales)
library(fs)
library(phylotools)

###
Mode <- function(x) {
  ux <- unique(x)
  ux[which.max(tabulate(match(x, ux)))]
}



── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.4
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘scales’


The following object is masked from ‘package:purrr’:

    discard


The following object is masked from ‘package:readr’:

    col_factor


Loading required package: ape


Attaching package: ‘ape’


The following object is masked from ‘package:dplyr’:

    where




In [2]:
getwd() 

[1] "/hpc/projects/balla_group/sra_experiments/SRA_metadata/dec2025_75k_submitteradded"

In [3]:
#setwd('/hpc/projects/projects/balla_group/sra_experiments/SRA_metadata/june2025_75k')
workingpath <- getwd()
workingpathdash <- str_c(workingpath, "/")


In [4]:
metadata_curationstep0 <- read_tsv("all_zf_datescurated_withGEO.tsv")             

Rows: 77329 Columns: 80
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr  (52): run.accession, study.accession, bioproject, study.title, sample.a...
dbl  (11): run.total_bases, run.total_spots, run.r1_length, run.r2_length, r...
date (17): earliest_date, run.enafirstpublic_date, run.enalastupdate_date, r...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [5]:
## start by removing many of columns from metadata_join5
## remember we use 6 columns for first pass of age stage tissue
## then 20 columns for second & third pass (17 columns second then 3 study columns are third)

## put the primary curation BEFORE run.title

metadata_curationstep1 <- metadata_curationstep0 %>% select(run.accession:earliest_date,experiment.title,experiment.attributes,sample.title,sample.attributes,GEOsample.title,GEOsample.characteristics,run.title,run.alias,run.attributes,experiment.alias,experiment.library_name,experiment.design_description,experiment.library_construction_protocol,sample.description,sample.alias,GEOsample.dataprocessing,GEOsample.source,GEOsample.treatmentprotocol,GEOsample.extractprotocol,GEOsample.growthprotocol,study.abstract,study.attributes)


## Now the dev_stage & tissue calling functions!!


- updated data cleaning function

- updated devstage main function
 
  - but first a more in-depth dev_stage function that searches more terms

- dev_stage_coarse

- tissue

- tissue_coarse


In [6]:
clean_text_columns <- function(df, columns) {
  # Check if all specified columns exist in the dataframe
  if (!all(columns %in% names(df))) {
    stop("Not all specified columns exist in the dataframe")
  }
  
  # Apply cleaning operations to specified columns - late change, convert commas to spaces don't just remove
  df %>%
    ## remove all tildes and parentheses also - & , to spaces (and further down _ to space)
    mutate(across(all_of(columns), ~ gsub("[~()]", "", .))) %>%
    mutate(across(all_of(columns), ~ gsub("-", " ", .))) %>%
    mutate(across(all_of(columns), ~ gsub(",", " ", .))) %>%
    ### adding many mutates from above smaller function for age stage devstage extraction to here as well 50% epi 
    
    # mutate(across(all_of(columns), ~ gsub("(?<=\\d)_(?=\\d)", ".", .))) %>%
    ## for above can't use gsub with lookahead or lookbehind, need to use str_replace
    # Apply the regex to replace underscores surrounded by digits - but first replace numbers after batch also check for day 0 & 0 day
    mutate(across(all_of(columns), ~ gsub("batch1", "batchA", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("batch2", "batchB", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("batch3", "batchC", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("batch4", "batchD", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("batch5", "batchE", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("batch6", "batchF", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("day 0", "day x", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("day0", "day x", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("20 mins post", "0 hours post", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("3 prime", "three prime", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("5 prime", "five prime", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("3'", "three prime", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("5'", "five prime", ., ignore.case = TRUE))) %>%
    # mutate(across(all_of(columns), ~ gsub(" 0 day", "day x", ., ignore.case = TRUE))) %>%
    ## really not sure the next line is a good idea...seems like it breaks a lot of edge cases...
    #mutate(across(all_of(columns), ~ str_replace_all(., "(?<=\\d)_(?=\\d)", "."))) %>% 
    ## first convert underscores before and after numbers to decimal points (?<=\d)_(?=\d) BUT WATCH THINGS LIKE batch2_32hpf_YD Dre_132_6hpf_RPF Dre_155_6hpf_RPF
    mutate(across(all_of(columns), ~ gsub("_", " ", .))) %>%
    mutate(across(all_of(columns), ~ gsub("after", "post", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("BHPF", "BHbisPhenolF", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("heterozygotes", "heterozygous", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("heterozygote", "heterozygous", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("homozygotes", "homozygous", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("homozygote", "homozygous", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("weeks post sham", "wks post sham", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("month F0", "mnth F0", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("post 1 year", "post 1y", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("1 year later", "1y later", ., ignore.case = TRUE))) %>%
    ## note insidious cases where 'injected' or 'exposed' gives additional timepoints that are bad - asking chatGPT to fix these
    ## The regex pattern will capture the phrase "exposed" followed by any number of words and then the word "at" followed by any number, and replace it with "exposed xxx".
    ## modification to keep the words in between str_replace_all(text, "(exposed(?:\\s+\\w+){1,30})\\s+at\\s+\\d+", "\\1 at xxx")
    ## more in august to check "epiboly  hatching  growth" hatching, growth
    ## Aug 6 adding exposed .... from xxx dpf to yyy dpf by adding more terms here (might need to first search for to, then from, also run 2 'to' searches)
    ## check "without head or testis" also "without eyes", "eyes removed" - NOTE REMOVED TERMS ARE FIXED IN TISSUE FUNCTION...also adding checks for multiple numbers at bottom & "xxx post injury"
    mutate(across(all_of(columns), ~ gsub("post hatching", "post hatch", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("50% epi stage", "50% epiboly stage", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("epiboly  hatching  growth", "epibol hatch growth", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("hatching gland", "hatch gland", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("post hatching", "post hatch", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("hatching rate", "hatch rate", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("million year", "million yr", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("juvinile", "juvenile", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("adulthood", "maturity", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("few year", "few yr", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("65 year", "65 yr", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("recent year", "recent yr", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("many year", "many yr", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("without head or testis", "without xxyy", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("\\b\\w+ post injury\\b", "xxx post injury", ., ignore.case = TRUE))) %>%
    # Regex pattern to match any word following "without"
    # pattern_without <- "(?<=without )\\w+"
    # Use str_replace_all to apply the pattern and replace the matched word with "xxx"
    # result1 <- str_replace_all(text1, pattern, "xxx")
    # result2 <- str_replace_all(text2, pattern, "xxx")
    mutate(across(all_of(columns), ~ str_replace_all(., "(?<=without )\\w+", "xxx"))) %>%
    ## check "exposed to a drug from 1dpf to 5dpf" also adding more weeks & months pf below
    ## some new 2025 additions  post fertilization hour 120
    mutate(across(all_of(columns), ~ gsub("days post heart amputation", "dy post heart amputation", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("days after genetic ablation", "dy after genetic ablation", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("days post incubation", "dy post incubation", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("days post amputation", "dy post amputation", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("days post single ablation", "dy post single ablation", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("days post double ablation", "dy post double ablation", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("days after single ablation", "dy after single ablation", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("days after double ablation", "dy after double ablation", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("post fertilization hour 120", "120 hpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ str_replace_all(., "(injection(?:\\s+\\w+){1,30})\\s+at\\s+\\d+", "\\1 at xxx"))) %>%
    ## many possible epiboly - only 30% is blastula rest are gastrula  - so replace these with zfs:0000015 30percent_epiboly 30epiboly  30% epiboly 30%-epiboly 15%-epiboly dome-to-epiboly
    ## also some non numeric epiboly affected epiboly impaired epiboly both epiboly perturbed epiboly at epiboly in epiboly of epiboly on epiboly to epiboly 75epibolu
    ## 1cell stage embryos were injected
    mutate(across(all_of(columns), ~ gsub("dome to epiboly", "dome to epib", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("affected epiboly", "affected epib", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("impaired epiboly", "impaired epib", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("both epiboly", "both epib", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("perturbed epiboly", "perturbed epib", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("at epiboly", "at epib", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("in epiboly", "in epib", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("of epiboly", "of epib", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("on epiboly", "on epib", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("to epiboly", "to epib", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("30percent epiboly", "zfs:0000015", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("30epiboly", "zfs:0000015", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("30% epiboly", "zfs:0000015", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("15% epiboly", "zfs:0000015", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("epibolu", "epiboly", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("90d pfa", "90 days", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("moths", "months", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("1cell stage embryos were injected", "1c embryos were injected", ., ignore.case = TRUE))) %>%
    ## rest from 2024 
    mutate(across(all_of(columns), ~ str_replace_all(., "(exposed(?:\\s+\\w+){1,30})\\s+at\\s+\\d+", "\\1 at xxx"))) %>%
    mutate(across(all_of(columns), ~ str_replace_all(., "(exposed(?:\\s+\\w+){1,30})\\s+for\\s+\\d+", "\\1 for xxx"))) %>%
    mutate(across(all_of(columns), ~ str_replace_all(., "(exposed(?:\\s+\\w+){1,30})\\s+to\\s+\\d+", "\\1 ttwo xxx"))) %>%
    mutate(across(all_of(columns), ~ str_replace_all(., "(exposed(?:\\s+\\w+){1,30})\\s+to\\s+\\d+", "\\1 to xxx"))) %>%
    mutate(across(all_of(columns), ~ str_replace_all(., "(exposed(?:\\s+\\w+){1,30})\\s+from\\s+\\d+", "\\1 from xxx"))) %>%
    mutate(across(all_of(columns), ~ str_replace_all(., "(injected(?:\\s+\\w+){1,30})\\s+at\\s+\\d+", "\\1 at xxx"))) %>%
    mutate(across(all_of(columns), ~ str_replace_all(., "(injected(?:\\s+\\w+){1,30})\\s+for\\s+\\d+", "\\1 for xxx"))) %>%
    mutate(across(all_of(columns), ~ str_replace_all(., "(injected(?:\\s+\\w+){1,30})\\s+to\\s+\\d+", "\\1 ttwo xxx"))) %>%
    mutate(across(all_of(columns), ~ str_replace_all(., "(injected(?:\\s+\\w+){1,30})\\s+to\\s+\\d+", "\\1 to xxx"))) %>%
    mutate(across(all_of(columns), ~ str_replace_all(., "(injected(?:\\s+\\w+){1,30})\\s+from\\s+\\d+", "\\1 from xxx"))) %>%
    mutate(across(all_of(columns), ~ gsub("oocytes microinject", "microinject", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("oocyte microinject", "microinject", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("seven somite", "7 somite", ., ignore.case = TRUE))) %>%
    ## stressed 21 days
    mutate(across(all_of(columns), ~ gsub("stressed 21 days", "stressed", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("mothes", "months", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("age:6 mouths", "age:6 months", ., ignore.case = TRUE))) %>%
    ## ArrayExpress Age:2 days
    mutate(across(all_of(columns), ~ gsub("ArrayExpress Age:2 days", "age:2 days", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("injected at one", "injected at a", ., ignore.case = TRUE))) %>%
    # mutate(across(all_of(columns), ~ gsub("injected at 1", "injected at a", ., ignore.case = TRUE))) %>% # previous line should catch 
    mutate(across(all_of(columns), ~ gsub("hour old", "hpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("hours old", "hpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("day old", "dpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("days old", "dpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("week old", "wpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("weeks old", "wpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("month old", "mpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("months old", "mpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("post post", "post", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("d post fert[^ ]*", "dpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("day post fert[^ ]*", "dpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("days post fert[^ ]*", "dpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("w post fert[^ ]*", "wpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("week post fert[^ ]*", "wpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("weeks post fert[^ ]*", "wpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("wk post fert[^ ]*", "wpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("wks post fert[^ ]*", "wpf", ., ignore.case = TRUE))) %>% 
    mutate(across(all_of(columns), ~ gsub("m post fert[^ ]*", "mpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("month post fert[^ ]*", "mpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("months post fert[^ ]*", "mpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("w postfert[^ ]*", "wpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("week postfert[^ ]*", "wpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("weeks postfert[^ ]*", "wpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("wk postfert[^ ]*", "wpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("wks postfert[^ ]*", "wpf", ., ignore.case = TRUE))) %>% 
    mutate(across(all_of(columns), ~ gsub("m postfert[^ ]*", "mpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("month postfert[^ ]*", "mpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("months postfert[^ ]*", "mpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("d postfert[^ ]*", "dpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("day postfert[^ ]*", "dpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("days postfert[^ ]*", "dpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("h post fert[^ ]*", "hpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("hour post fert[^ ]*", "hpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("hours post fert[^ ]*", "hpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("hr post fert[^ ]*", "hpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("hrs post fert[^ ]*", "hpf", ., ignore.case = TRUE))) %>% 
    mutate(across(all_of(columns), ~ gsub("h postfert[^ ]*", "hpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("hour postfert[^ ]*", "hpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("hours postfert[^ ]*", "hpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("hr postfert[^ ]*", "hpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("hrs postfert[^ ]*", "hpf", ., ignore.case = TRUE))) %>% 
    mutate(across(all_of(columns), ~ str_replace_all(., "(\\d+) to (\\d+) (hpf|hfp|wpf|dfp|dpf|month|mpf)", "\\1 \\3 to \\2 \\3"))) %>%
    mutate(across(all_of(columns), ~ str_replace_all(., "(\\d+)(?=(?: \\d+)*(?: and \\d+ (hpf|hfp|wpf|dfp|dpf|month|mpf)))", "\\1 \\2"))) %>%
    mutate(across(all_of(columns), ~ str_replace_all(., "(\\d+)(?=(?: \\d+)* (hpf|hfp|wpf|dfp|dpf|month|mpf))", "\\1 \\2"))) %>% 
    mutate(across(all_of(columns), ~ gsub("hpf hpf hpf", "hpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("hpf hpf", "hpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("dpf dpf", "dpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("wpf wpf", "wpf", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("month months", "month", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("month month", "month", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("hfp hfp", "hfp", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("dfp dfp", "dfp", ., ignore.case = TRUE))) %>%
    mutate(across(all_of(columns), ~ gsub("mpf mpf", "mpf", ., ignore.case = TRUE)))
  
  # mutate(across(all_of(columns), tolower))
}



In [7]:
classify_stage2025first <- function(entry) {
  classifications <- c()
  # First - Replace dashes with spaces in the entry - instead doing this & rest in separate function clean_text_columns (but keeping basic dash replace just in case)-
  entry <- gsub("-", " ", entry)
  # ## fix post-fertilization - use agrep?? post fertilization - instead simpler fix (but add \s ??)
  # Regex pattern to match a number immediately before or after "hpf", "hfp", "week", or "wpf"
  ## if I add h at end need to also add it here
    ## for first stage ONLY adding both singlular and plural
  # plus h only when followed by | or end "(hpf|hfp|week|wpf|day|dfp|dpf|epiboly|hour|hrs|somite|month|mpf|h(?=\\||$))", also adding d at very end
#  pattern <- "(\\d+(\\.\\d+)?\\s*(hpf|hfp|week|wpf|day|dfp|dpf|epiboly|hour|hrs|somite|month|mpf))|((hpf|hfp|week|wpf|day|dfp|dpf|epiboly|hour|hrs|somite|month|mpf)\\s*\\d+(\\.\\d+)?)"
  pattern <- "(\\d+(\\.\\d+)?\\s*(hpf|hfp|week|wpf|day|dfp|dpf|pdf|epiboly|hours|hrs|somite|month|weeks|days|hour|months|mpf|d(?=\\||$)|h(?=\\||$)))|((hpf|hfp|week|wpf|day|dfp|dpf|pdf|epiboly|hours|hrs|somite|month|weeks|days|hour|months|mpf|d(?=\\||$)|h(?=\\||$))\\s*\\d+(\\.\\d+)?)"
  
  # Use regmatches and regexpr to find and extract the matched pattern - moved to further down
  # matches <- regmatches(entry, regexpr(pattern, entry, ignore.case = TRUE, perl = TRUE))
  

  ## first more basic grep searches - CHECK SOME OF THESE, MAY NOT WANT TO INCLUDE MORE GENERAL KEYWORDS...   15% epiboly 30% epiboly also move adult first, most likely to be additional term
  ## remove adult here, only adding to coarse category if not otherwise found
  # if (grepl("adult", entry, ignore.case = TRUE) == TRUE | grepl("mpf", entry, ignore.case = TRUE) == TRUE | grepl("month", entry, ignore.case = TRUE) == TRUE | grepl("year", entry, ignore.case = TRUE) == TRUE) {
  #   return("Adult Period")
  # }
  ## similarly add month & year to coarse category - also at this point 6/11 removing 'day' from keywords...could be
  ### moving mpf to numbers because 1-3 is juvenile, only leaving if undetermined in coarse
  ### also do same for juvenile & larval larva larvae - also removing 'cleavage', can be general term
  ## fixing 00044 to adult, removig 00045 00046 00047 which are only used with earlier stages
  ## aug 1 changing all else if to if - 'shield' and 'oblong' were not being caught??
  if (grepl("zfs:0000044", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Adult")
  }
  if (grepl("zygote", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000001", entry, ignore.case = TRUE) == TRUE | grepl("unfertilized", entry, ignore.case = TRUE) == TRUE | grepl("sperm ", entry, ignore.case = TRUE) == TRUE | grepl("spermatozoa", entry, ignore.case = TRUE) == TRUE | grepl("zygote", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Zygote")
  }
  if (grepl("zfs:0000002", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000003", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000004", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000005", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000006", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000007", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Cleavage")
  }
  if (grepl("blastula", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000008", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000009", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000010", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000011", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000012", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000013", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000014", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000015", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000016", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Blastula")
  }
  if (grepl("gastrula", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000017", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000018", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000019", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000020", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000021", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000022", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Gastrula")
  }
  if (grepl("somite", entry, ignore.case = TRUE) == TRUE | grepl("segmentation", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000049", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000023", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000024", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000025", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000026", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000027", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000028", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Segmentation")
  }
  if (grepl("pharyngula", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000050", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000029", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000030", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000031", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000032", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Pharyngula")
  }
  if (grepl("hatching", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0007000", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000033", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000034", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Hatching")
  }
  if (grepl("protruding mouth", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000048", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000035", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000036", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000037", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000038", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000039", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000040", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000041", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Larval")
  }
  if (grepl("zfs:0000051", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000042", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000043", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Juvenile")
  }
  # if (grepl("10% epiboly", entry, ignore.case = TRUE) == TRUE | grepl("15% epiboly", entry, ignore.case = TRUE) == TRUE | grepl("20% epiboly", entry, ignore.case = TRUE) == TRUE | grepl("25% epiboly", entry, ignore.case = TRUE) == TRUE | grepl("30% epiboly", entry, ignore.case = TRUE) == TRUE) {
  #   classifications <- c(classifications, "Blastula")
  # }
  ## after above new gsubs, any remaining epiboly is Gastrula
  if (grepl("epiboly", entry, ignore.case = TRUE) == TRUE | grepl("germ ring", entry, ignore.case = TRUE) == TRUE | grepl("shield stage", entry, ignore.case = TRUE) == TRUE | grepl("bud stage", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Gastrula")
  }
  # if (grepl("50% epiboly", entry, ignore.case = TRUE) == TRUE | grepl("55% epiboly", entry, ignore.case = TRUE) == TRUE | grepl("60% epiboly", entry, ignore.case = TRUE) == TRUE | grepl("65% epiboly", entry, ignore.case = TRUE) == TRUE | grepl("70% epiboly", entry, ignore.case = TRUE) == TRUE | grepl("75% epiboly", entry, ignore.case = TRUE) == TRUE | grepl("80% epiboly", entry, ignore.case = TRUE) == TRUE | grepl("85% epiboly", entry, ignore.case = TRUE) == TRUE | grepl("90% epiboly", entry, ignore.case = TRUE) == TRUE | grepl("germ ring", entry, ignore.case = TRUE) == TRUE | grepl("shield stage", entry, ignore.case = TRUE) == TRUE | grepl("bud stage", entry, ignore.case = TRUE) == TRUE) {
  #   classifications <- c(classifications, "Gastrula")
  # }
  
  ## also adding prim and pec
  if (grepl("prim 5", entry, ignore.case = TRUE) == TRUE | grepl("prim 15", entry, ignore.case = TRUE) == TRUE | grepl("prim 25", entry, ignore.case = TRUE) == TRUE | 
      grepl("prim5", entry, ignore.case = TRUE) == TRUE  | grepl("prim15", entry, ignore.case = TRUE) == TRUE  | grepl("prim25", entry, ignore.case = TRUE) == TRUE  | grepl("high pec", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Pharyngula")
  }
  if (grepl("long pec", entry, ignore.case = TRUE) == TRUE | grepl("pec fin", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Hatching")
  }
  
  # # Use regmatches and regexpr to find and extract the matched pattern - moved to further down
  # matches <- regmatches(entry, regexpr(pattern, entry, ignore.case = TRUE, perl = TRUE))
  
  # JUNE 2025 UPDATE - TO FIND ALL OF THE MATCHES
#  3) Extract *all* the matches
  #all_matches <- regmatches(entry,gregexpr(pattern, entry, ignore.case = TRUE, perl = TRUE))[[1]]
  all_matches <- regmatches(entry,gregexpr(pattern, entry, ignore.case = TRUE, perl = TRUE))[[1]]
  
  

  # Check if any match was found if (length(all_matches) > 0 && nzchar(all_matches[1]))
#  if (length(matches) > 0 && nzchar(matches)) {
#  if (length(matches) > 0 && nzchar(matches[1])) {
    # Extract the number from the match
    # num <- as.numeric(unlist(regmatches(matches, gregexpr("\\d+\\.?\\d*", matches))))
    # 
    # # Classification based on the number and keywords
    # if (!is.na(num)) {
    #   # Determine the keyword involved for specific classification
    #   keyword <- tolower(unlist(regmatches(matches, gregexpr("(hpf|hfp|week|wpf|day|dfp|dpf|epiboly|hour|hrs|somite|month|mpf)", matches, ignore.case = TRUE))))

  if (length(all_matches) > 0 && nzchar(all_matches[1])) {
    for (m in all_matches) {
      # extract the numeric part
      num <- as.numeric(
        unlist(regmatches(m, gregexpr("\\d+\\.?\\d*", m)))
      )
      if (is.na(num)) next
      
      # extract the keyword
      ## plus h only when followed by | or end "(hpf|hfp|week|wpf|day|dfp|dpf|epiboly|hour|hrs|somite|month|mpf|h(?=\\||$))",
#      keyword <- tolower(unlist(regmatches(m,gregexpr("(hpf|hfp|week|wpf|day|dfp|dpf|epiboly|hour|hrs|somite|month|mpf)",m, ignore.case = TRUE))))
      ## Both 96 h and 7h will result in keyword == "h". d(?=\\||$)|
        ## for first stage ONLY adding both singlular and plural
      keyword <- tolower(unlist(regmatches(m,gregexpr("(hpf|hfp|week|wpf|day|dfp|dpf|pdf|epiboly|hours|hrs|somite|month|weeks|days|hour|months|mpf|d(?=\\||$)|h(?=\\||$))",m, ignore.case = TRUE, perl = TRUE))))
      
      # Classify based on hours - but don't use just hour or hours, see above for combining if 'post-fert' is seen
      ## NEED TO FIX ALL LIKE if (num < 2.25 && num > 0.75) {
      ### NOTE 2025 MAKE 2 VERSIONS OF THIS - ONE WITH MORE KEYWORDS IN NEXT CHUNKS - include hour hrs maybe also h$
      if (any(keyword %in% c("hpf", "hfp", "hours", "hour", "h"))) {
        if (num < 0.75) classifications <- c(classifications, "Zygote")
        if (num < 2.25 && num >= 0.75) classifications <- c(classifications, "Cleavage")
        if (num < 5.25 && num >= 2.25) classifications <- c(classifications, "Blastula")
        if (num < 10.33 && num >= 5.25) classifications <- c(classifications, "Gastrula")
        if (num < 24 && num >= 10.33) classifications <- c(classifications, "Segmentation")
        if (num < 48 && num >= 24) classifications <- c(classifications, "Pharyngula")
        if (num < 72 && num >= 48) classifications <- c(classifications, "Hatching")
        if (num < 720 && num >= 72) classifications <- c(classifications, "Larval")
        #classifications <- c(classifications, "Adult")
      }
      
      # Classify based on days
      if (any(keyword %in% c("dpf", "dfp", "day", "days", "pdf", "d"))) {
        if (num < 0.2) classifications <- c(classifications, "Blastula")
        if (num < 0.4 && num >= 0.2) classifications <- c(classifications, "Gastrula")
        if (num < 1 && num >= 0.4) classifications <- c(classifications, "Segmentation")
        if (num < 2 && num >= 1) classifications <- c(classifications, "Pharyngula")
        if (num < 3 && num >= 2) classifications <- c(classifications, "Hatching")
        if (num < 30 && num >= 3) classifications <- c(classifications, "Larval")
        if (num < 90 && num >= 30) classifications <- c(classifications, "Juvenile")
        if (num < 990 && num >= 90) classifications <- c(classifications, "Adult")
        #classifications <- c(classifications, "Adult")
      }
      
      # Classify based on "week" or "wpf"
      if (any(keyword %in% c("week", "weeks", "wpf"))) {
        if (num < 4.29) classifications <- c(classifications, "Larval")
        if (num < 12.86 && num >= 4.29) classifications <- c(classifications, "Juvenile")
        if (num < 199 && num >= 12.86) classifications <- c(classifications, "Adult")
        #classifications <- c(classifications, "Adult")
      }
      # moving month to numbers because 1-3 is juvenile, only leaving if undetermined in coarse
      if (any(keyword %in% c("month", "months", "mpf"))) {
        if (num < 1) classifications <- c(classifications, "Larval")
        if (num < 3 && num >= 1) classifications <- c(classifications, "Juvenile")
        if (num < 99 && num >= 3) classifications <- c(classifications, "Adult")
        #classifications <- c(classifications, "Adult")
      }
      
    }
  }
  
  ## bringing back a few of these 7/16/24 ADDING LAST!!, also oocyte adding last here - many from PRJNA529241
  ## aug 1 adding back dome, shield, tailbud, some cells be careful with zf4
  if (grepl("128/256 cell", entry, ignore.case = TRUE) == TRUE | grepl("\\bdome\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b512 cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b512cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b128 cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b128cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b256 cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b256cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b512 cells\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b512cells\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b128 cells\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b128cells\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b256 cells\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b256cells\\b", entry, ignore.case = TRUE) == TRUE | grepl("1k cell", entry, ignore.case = TRUE) == TRUE | grepl("\\b1000 cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b1000cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("oblong", entry, ignore.case = TRUE) == TRUE | grepl("\\bsphere\\b", entry, ignore.case = TRUE) == TRUE | grepl(" sphere", entry, ignore.case = TRUE) == TRUE | grepl("WT 1Kc", entry, ignore.case = TRUE) == TRUE | grepl("1Kc p300", entry, ignore.case = TRUE) == TRUE | grepl("1Kc trip", entry, ignore.case = TRUE) == TRUE | grepl("a Am 1Kc", entry, ignore.case = TRUE) == TRUE | grepl("TinyLNA430 1Kc", entry, ignore.case = TRUE) == TRUE | grepl("a Am 256c", entry, ignore.case = TRUE) == TRUE | grepl("2N 1Kc", entry, ignore.case = TRUE) == TRUE | grepl("a Am 512c", entry, ignore.case = TRUE) == TRUE | grepl("1N 512c", entry, ignore.case = TRUE) == TRUE | grepl("WT 256c", entry, ignore.case = TRUE) == TRUE | grepl("WT dome", entry, ignore.case = TRUE) == TRUE | grepl("1Kcell", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Blastula")
  }
  if (grepl("a Am 75%epi", entry, ignore.case = TRUE) == TRUE | grepl("shield", entry, ignore.case = TRUE) == TRUE | grepl("WT 50%epi", entry, ignore.case = TRUE) == TRUE | grepl("WT 75%epi", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Gastrula")
  }
  ## adding shield as a full word here almost last for blastula "\\bshield\\b" just bfore zygote
  if (grepl("\\bshield\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\btailbud\\b", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Gastrula")
  }
  if (grepl("\\b1cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("oocyte", entry, ignore.case = TRUE) == TRUE | grepl("WT 1c", entry, ignore.case = TRUE) == TRUE | grepl("a Am 1c", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Zygote")
  }
  ## note "2 cell" has to be after "512 cell" - FIXING ALL IN AUGUST WITH \\B, 4 can get both 4 and 64 though (not 4 because zf4 also will ping), also for zygote remvoing "1 cell" only keeping 1cell
  if (grepl("\\b4 cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b4cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b2 cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b2cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b32 cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b32cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b8 cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b8cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b4 cells\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b4cells\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b2 cells\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b2cells\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b32 cells\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b32cells\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b8 cells\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b8cells\\b", entry, ignore.case = TRUE) == TRUE | grepl("16/32 cell", entry, ignore.case = TRUE) == TRUE | grepl("16 cell", entry, ignore.case = TRUE) == TRUE | grepl("64 cell", entry, ignore.case = TRUE) == TRUE | grepl("16cell", entry, ignore.case = TRUE) == TRUE | grepl("64cell", entry, ignore.case = TRUE) == TRUE | grepl("WT 2c", entry, ignore.case = TRUE) == TRUE | grepl("WT 4c", entry, ignore.case = TRUE) == TRUE | grepl("a Am 16c", entry, ignore.case = TRUE) == TRUE | grepl("WT 64c", entry, ignore.case = TRUE) == TRUE | grepl("WT 32c", entry, ignore.case = TRUE) == TRUE | grepl("TinyLNA430 64c", entry, ignore.case = TRUE) == TRUE | grepl("a Am 64c", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Cleavage")
  }
  ## also adding adult larva juvenile terms from coarse function here, only embryo will be added at coarse now
  ## move larvae to ONLY in coarse, hundreds of cases where it gets added even though hpf shows it as embryo - BETTER IDEA, HAVE THESE INSIDE A  if (length(classifications) == 0) { STATEMENT!!
  if (length(classifications) == 0) {
    if (grepl("adult", entry, ignore.case = TRUE) == TRUE | grepl("age:1y", entry, ignore.case = TRUE) == TRUE | grepl("three month", entry, ignore.case = TRUE) == TRUE | grepl("three mpf", entry, ignore.case = TRUE) == TRUE | grepl("four month", entry, ignore.case = TRUE) == TRUE | grepl("four mpf", entry, ignore.case = TRUE) == TRUE | grepl("five month", entry, ignore.case = TRUE) == TRUE | grepl("five mpf", entry, ignore.case = TRUE) == TRUE | grepl("six month", entry, ignore.case = TRUE) == TRUE | grepl("six mpf", entry, ignore.case = TRUE) == TRUE | grepl("seven month", entry, ignore.case = TRUE) == TRUE | grepl("seven mpf", entry, ignore.case = TRUE) == TRUE | grepl("eight month", entry, ignore.case = TRUE) == TRUE | grepl("eight mpf", entry, ignore.case = TRUE) == TRUE | grepl("natal month three", entry, ignore.case = TRUE) == TRUE | grepl("year", entry, ignore.case = TRUE) == TRUE) {
      classifications <- c(classifications, "Adult")
    }
    if (grepl("larva", entry, ignore.case = TRUE) == TRUE | grepl("larval", entry, ignore.case = TRUE) == TRUE | grepl("larvae", entry, ignore.case = TRUE) == TRUE | grepl("eleutherolarva", entry, ignore.case = TRUE) == TRUE) {
      classifications <- c(classifications, "Larval")
    }
    if (grepl("juvenile", entry, ignore.case = TRUE) == TRUE | grepl("one month", entry, ignore.case = TRUE) == TRUE | grepl("two months", entry, ignore.case = TRUE) == TRUE) {
      classifications <- c(classifications, "Juvenile")
    }
  }
  
  if (length(classifications) == 0) {
    return("Undetermined")
  } else {
    # Split the string into words
    words <- unlist(strsplit(classifications, " "))
    # Remove duplicate words
    unique_words <- unique(words)
    # Combine the unique words back into a single string
    classifications <- paste(unique_words, collapse = ", ")
    return(paste(classifications, collapse = ", "))
  }
}


In [8]:
classify_stage2025 <- function(entry) {
  classifications <- c()
  # First - Replace dashes with spaces in the entry - instead doing this & rest in separate function clean_text_columns (but keeping basic dash replace just in case)-
  entry <- gsub("-", " ", entry)
  # ## fix post-fertilization - use agrep?? post fertilization - instead simpler fix (but add \s ??)
  # Regex pattern to match a number immediately before or after "hpf", "hfp", "week", or "wpf"
  ## if I add h at end need to also add it here
  # plus h only when followed by | or end "(hpf|hfp|week|wpf|day|dfp|dpf|epiboly|hour|hrs|somite|month|mpf|h(?=\\||$))",
  #  pattern <- "(\\d+(\\.\\d+)?\\s*(hpf|hfp|week|wpf|day|dfp|dpf|epiboly|hour|hrs|somite|month|mpf))|((hpf|hfp|week|wpf|day|dfp|dpf|epiboly|hour|hrs|somite|month|mpf)\\s*\\d+(\\.\\d+)?)"
  pattern <- "(\\d+(\\.\\d+)?\\s*(hpf|hfp|week|wpf|day|dfp|dpf|epiboly|hours|hrs|somite|month|mpf|h(?=\\||$)))|((hpf|hfp|week|wpf|day|dfp|dpf|epiboly|hours|hrs|somite|month|mpf|h(?=\\||$))\\s*\\d+(\\.\\d+)?)"
  
  # Use regmatches and regexpr to find and extract the matched pattern - moved to further down
  # matches <- regmatches(entry, regexpr(pattern, entry, ignore.case = TRUE, perl = TRUE))
  
  
  ## first more basic grep searches - CHECK SOME OF THESE, MAY NOT WANT TO INCLUDE MORE GENERAL KEYWORDS...   15% epiboly 30% epiboly also move adult first, most likely to be additional term
  ## remove adult here, only adding to coarse category if not otherwise found
  # if (grepl("adult", entry, ignore.case = TRUE) == TRUE | grepl("mpf", entry, ignore.case = TRUE) == TRUE | grepl("month", entry, ignore.case = TRUE) == TRUE | grepl("year", entry, ignore.case = TRUE) == TRUE) {
  #   return("Adult Period")
  # }
  ## similarly add month & year to coarse category - also at this point 6/11 removing 'day' from keywords...could be
  ### moving mpf to numbers because 1-3 is juvenile, only leaving if undetermined in coarse
  ### also do same for juvenile & larval larva larvae - also removing 'cleavage', can be general term
  ## fixing 00044 to adult, removig 00045 00046 00047 which are only used with earlier stages
  ## aug 1 changing all else if to if - 'shield' and 'oblong' were not being caught??
  if (grepl("zfs:0000044", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Adult")
  }
  if (grepl("zygote", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000001", entry, ignore.case = TRUE) == TRUE | grepl("unfertilized", entry, ignore.case = TRUE) == TRUE | grepl("sperm ", entry, ignore.case = TRUE) == TRUE | grepl("spermatozoa", entry, ignore.case = TRUE) == TRUE | grepl("zygote", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Zygote")
  }
  if (grepl("zfs:0000002", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000003", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000004", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000005", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000006", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000007", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Cleavage")
  }
  if (grepl("blastula", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000008", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000009", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000010", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000011", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000012", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000013", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000014", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000015", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000016", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Blastula")
  }
  if (grepl("gastrula", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000017", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000018", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000019", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000020", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000021", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000022", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Gastrula")
  }
  if (grepl("somite", entry, ignore.case = TRUE) == TRUE | grepl("segmentation", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000049", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000023", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000024", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000025", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000026", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000027", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000028", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Segmentation")
  }
  if (grepl("pharyngula", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000050", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000029", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000030", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000031", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000032", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Pharyngula")
  }
  if (grepl("hatching", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0007000", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000033", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000034", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Hatching")
  }
  if (grepl("protruding mouth", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000048", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000035", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000036", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000037", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000038", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000039", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000040", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000041", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Larval")
  }
  if (grepl("zfs:0000051", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000042", entry, ignore.case = TRUE) == TRUE | grepl("zfs:0000043", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Juvenile")
  }
  # if (grepl("10% epiboly", entry, ignore.case = TRUE) == TRUE | grepl("15% epiboly", entry, ignore.case = TRUE) == TRUE | grepl("20% epiboly", entry, ignore.case = TRUE) == TRUE | grepl("25% epiboly", entry, ignore.case = TRUE) == TRUE | grepl("30% epiboly", entry, ignore.case = TRUE) == TRUE) {
  #   classifications <- c(classifications, "Blastula")
  # }
  ## after above new gsubs, any remaining epiboly is Gastrula
  if (grepl("epiboly", entry, ignore.case = TRUE) == TRUE | grepl("germ ring", entry, ignore.case = TRUE) == TRUE | grepl("shield stage", entry, ignore.case = TRUE) == TRUE | grepl("bud stage", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Gastrula")
  }
  # if (grepl("50% epiboly", entry, ignore.case = TRUE) == TRUE | grepl("55% epiboly", entry, ignore.case = TRUE) == TRUE | grepl("60% epiboly", entry, ignore.case = TRUE) == TRUE | grepl("65% epiboly", entry, ignore.case = TRUE) == TRUE | grepl("70% epiboly", entry, ignore.case = TRUE) == TRUE | grepl("75% epiboly", entry, ignore.case = TRUE) == TRUE | grepl("80% epiboly", entry, ignore.case = TRUE) == TRUE | grepl("85% epiboly", entry, ignore.case = TRUE) == TRUE | grepl("90% epiboly", entry, ignore.case = TRUE) == TRUE | grepl("germ ring", entry, ignore.case = TRUE) == TRUE | grepl("shield stage", entry, ignore.case = TRUE) == TRUE | grepl("bud stage", entry, ignore.case = TRUE) == TRUE) {
  #   classifications <- c(classifications, "Gastrula")
  # }
  
  ## also adding prim and pec
  if (grepl("prim 5", entry, ignore.case = TRUE) == TRUE | grepl("prim 15", entry, ignore.case = TRUE) == TRUE | grepl("prim 25", entry, ignore.case = TRUE) == TRUE | 
      grepl("prim5", entry, ignore.case = TRUE) == TRUE  | grepl("prim15", entry, ignore.case = TRUE) == TRUE  | grepl("prim25", entry, ignore.case = TRUE) == TRUE  | grepl("high pec", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Pharyngula")
  }
  if (grepl("long pec", entry, ignore.case = TRUE) == TRUE | grepl("pec fin", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Hatching")
  }
  
  # # Use regmatches and regexpr to find and extract the matched pattern - moved to further down
  # matches <- regmatches(entry, regexpr(pattern, entry, ignore.case = TRUE, perl = TRUE))
  
  # JUNE 2025 UPDATE - TO FIND ALL OF THE MATCHES
  #  3) Extract *all* the matches
  #all_matches <- regmatches(entry,gregexpr(pattern, entry, ignore.case = TRUE, perl = TRUE))[[1]]
  all_matches <- regmatches(entry,gregexpr(pattern, entry, ignore.case = TRUE, perl = TRUE))[[1]]
  
  
  
  # Check if any match was found if (length(all_matches) > 0 && nzchar(all_matches[1]))
  #  if (length(matches) > 0 && nzchar(matches)) {
  #  if (length(matches) > 0 && nzchar(matches[1])) {
  # Extract the number from the match
  # num <- as.numeric(unlist(regmatches(matches, gregexpr("\\d+\\.?\\d*", matches))))
  # 
  # # Classification based on the number and keywords
  # if (!is.na(num)) {
  #   # Determine the keyword involved for specific classification
  #   keyword <- tolower(unlist(regmatches(matches, gregexpr("(hpf|hfp|week|wpf|day|dfp|dpf|epiboly|hour|hrs|somite|month|mpf)", matches, ignore.case = TRUE))))
  
  if (length(all_matches) > 0 && nzchar(all_matches[1])) {
    for (m in all_matches) {
      # extract the numeric part
      num <- as.numeric(
        unlist(regmatches(m, gregexpr("\\d+\\.?\\d*", m)))
      )
      if (is.na(num)) next
      
      # extract the keyword
      ## plus h only when followed by | or end "(hpf|hfp|week|wpf|day|dfp|dpf|epiboly|hour|hrs|somite|month|mpf|h(?=\\||$))",
      #      keyword <- tolower(unlist(regmatches(m,gregexpr("(hpf|hfp|week|wpf|day|dfp|dpf|epiboly|hour|hrs|somite|month|mpf)",m, ignore.case = TRUE))))
      ## Both 96 h and 7h will result in keyword == "h".
      keyword <- tolower(unlist(regmatches(m,gregexpr("(hpf|hfp|week|wpf|day|dfp|dpf|epiboly|hours|hrs|somite|month|mpf|h(?=\\||$))",m, ignore.case = TRUE, perl = TRUE))))
      
      # Classify based on hours - but don't use just hour or hours, see above for combining if 'post-fert' is seen
      ## NEED TO FIX ALL LIKE if (num < 2.25 && num > 0.75) {
      ### NOTE 2025 MAKE 2 VERSIONS OF THIS - ONE WITH MORE KEYWORDS IN NEXT CHUNKS - include hour hrs maybe also h$
      if (any(keyword %in% c("hpf", "hfp"))) {
        if (num < 0.75) classifications <- c(classifications, "Zygote")
        if (num < 2.25 && num >= 0.75) classifications <- c(classifications, "Cleavage")
        if (num < 5.25 && num >= 2.25) classifications <- c(classifications, "Blastula")
        if (num < 10.33 && num >= 5.25) classifications <- c(classifications, "Gastrula")
        if (num < 24 && num >= 10.33) classifications <- c(classifications, "Segmentation")
        if (num < 48 && num >= 24) classifications <- c(classifications, "Pharyngula")
        if (num < 72 && num >= 48) classifications <- c(classifications, "Hatching")
        if (num < 720 && num >= 72) classifications <- c(classifications, "Larval")
        #classifications <- c(classifications, "Adult")
      }
      
      # Classify based on days
      if (any(keyword %in% c("dpf", "dfp"))) {
        if (num < 0.2) classifications <- c(classifications, "Blastula")
        if (num < 0.4 && num >= 0.2) classifications <- c(classifications, "Gastrula")
        if (num < 1 && num >= 0.4) classifications <- c(classifications, "Segmentation")
        if (num < 2 && num >= 1) classifications <- c(classifications, "Pharyngula")
        if (num < 3 && num >= 2) classifications <- c(classifications, "Hatching")
        if (num < 30 && num >= 3) classifications <- c(classifications, "Larval")
        if (num < 90 && num >= 30) classifications <- c(classifications, "Juvenile")
        if (num < 990 && num >= 90) classifications <- c(classifications, "Adult")
        #classifications <- c(classifications, "Adult")
      }
      
      # Classify based on "week" or "wpf"
      if (any(keyword %in% c("week", "wpf"))) {
        if (num < 4.29) classifications <- c(classifications, "Larval")
        if (num < 12.86 && num >= 4.29) classifications <- c(classifications, "Juvenile")
        if (num < 199 && num >= 12.86) classifications <- c(classifications, "Adult")
        #classifications <- c(classifications, "Adult")
      }
      # moving month to numbers because 1-3 is juvenile, only leaving if undetermined in coarse
      if (any(keyword %in% c("month", "mpf"))) {
        if (num < 1) classifications <- c(classifications, "Larval")
        if (num < 3 && num >= 1) classifications <- c(classifications, "Juvenile")
        if (num < 99 && num >= 3) classifications <- c(classifications, "Adult")
        #classifications <- c(classifications, "Adult")
      }
      
    }
  }
  
  ## bringing back a few of these 7/16/24 ADDING LAST!!, also oocyte adding last here
  ## aug 1 adding back dome, shield, tailbud, some cells be careful with zf4
  if (grepl("128/256 cell", entry, ignore.case = TRUE) == TRUE | grepl("\\bdome\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b512 cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b512cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b128 cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b128cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b256 cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b256cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b512 cells\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b512cells\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b128 cells\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b128cells\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b256 cells\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b256cells\\b", entry, ignore.case = TRUE) == TRUE | grepl("1k cell", entry, ignore.case = TRUE) == TRUE | grepl("\\b1000 cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b1000cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("oblong", entry, ignore.case = TRUE) == TRUE | grepl("\\bsphere\\b", entry, ignore.case = TRUE) == TRUE | grepl(" sphere", entry, ignore.case = TRUE) == TRUE | grepl("WT 1Kc", entry, ignore.case = TRUE) == TRUE | grepl("1Kc p300", entry, ignore.case = TRUE) == TRUE | grepl("1Kc trip", entry, ignore.case = TRUE) == TRUE | grepl("a Am 1Kc", entry, ignore.case = TRUE) == TRUE | grepl("TinyLNA430 1Kc", entry, ignore.case = TRUE) == TRUE | grepl("a Am 256c", entry, ignore.case = TRUE) == TRUE | grepl("2N 1Kc", entry, ignore.case = TRUE) == TRUE | grepl("a Am 512c", entry, ignore.case = TRUE) == TRUE | grepl("1N 512c", entry, ignore.case = TRUE) == TRUE | grepl("WT 256c", entry, ignore.case = TRUE) == TRUE | grepl("WT dome", entry, ignore.case = TRUE) == TRUE | grepl("1Kcell", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Blastula")
  }
  if (grepl("a Am 75%epi", entry, ignore.case = TRUE) == TRUE | grepl("shield", entry, ignore.case = TRUE) == TRUE | grepl("WT 50%epi", entry, ignore.case = TRUE) == TRUE | grepl("WT 75%epi", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Gastrula")
  }
  ## adding shield as a full word here almost last for blastula "\\bshield\\b" just bfore zygote
  if (grepl("\\bshield\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\btailbud\\b", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Gastrula")
  }
  if (grepl("\\b1cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("oocyte", entry, ignore.case = TRUE) == TRUE | grepl("WT 1c", entry, ignore.case = TRUE) == TRUE | grepl("a Am 1c", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Zygote")
  }
  ## note "2 cell" has to be after "512 cell" - FIXING ALL IN AUGUST WITH \\B, 4 can get both 4 and 64 though (not 4 because zf4 also will ping), also for zygote remvoing "1 cell" only keeping 1cell
  if (grepl("\\b4 cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b4cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b2 cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b2cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b32 cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b32cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b8 cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b8cell\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b4 cells\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b4cells\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b2 cells\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b2cells\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b32 cells\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b32cells\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b8 cells\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\b8cells\\b", entry, ignore.case = TRUE) == TRUE | grepl("16/32 cell", entry, ignore.case = TRUE) == TRUE | grepl("16 cell", entry, ignore.case = TRUE) == TRUE | grepl("64 cell", entry, ignore.case = TRUE) == TRUE | grepl("16cell", entry, ignore.case = TRUE) == TRUE | grepl("64cell", entry, ignore.case = TRUE) == TRUE | grepl("WT 2c", entry, ignore.case = TRUE) == TRUE | grepl("WT 4c", entry, ignore.case = TRUE) == TRUE | grepl("a Am 16c", entry, ignore.case = TRUE) == TRUE | grepl("WT 64c", entry, ignore.case = TRUE) == TRUE | grepl("WT 32c", entry, ignore.case = TRUE) == TRUE | grepl("TinyLNA430 64c", entry, ignore.case = TRUE) == TRUE | grepl("a Am 64c", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Cleavage")
  }
  ## also adding adult larva juvenile terms from coarse function here, only embryo will be added at coarse now
  ## move larvae to ONLY in coarse, hundreds of cases where it gets added even though hpf shows it as embryo - BETTER IDEA, HAVE THESE INSIDE A  if (length(classifications) == 0) { STATEMENT!!
  if (length(classifications) == 0) {
    if (grepl("adult", entry, ignore.case = TRUE) == TRUE | grepl("three month", entry, ignore.case = TRUE) == TRUE | grepl("three mpf", entry, ignore.case = TRUE) == TRUE | grepl("four month", entry, ignore.case = TRUE) == TRUE | grepl("four mpf", entry, ignore.case = TRUE) == TRUE | grepl("five month", entry, ignore.case = TRUE) == TRUE | grepl("five mpf", entry, ignore.case = TRUE) == TRUE | grepl("six month", entry, ignore.case = TRUE) == TRUE | grepl("six mpf", entry, ignore.case = TRUE) == TRUE | grepl("seven month", entry, ignore.case = TRUE) == TRUE | grepl("seven mpf", entry, ignore.case = TRUE) == TRUE | grepl("eight month", entry, ignore.case = TRUE) == TRUE | grepl("eight mpf", entry, ignore.case = TRUE) == TRUE | grepl("natal month three", entry, ignore.case = TRUE) == TRUE | grepl("year", entry, ignore.case = TRUE) == TRUE) {
      classifications <- c(classifications, "Adult")
    }
    if (grepl("larva", entry, ignore.case = TRUE) == TRUE | grepl("larval", entry, ignore.case = TRUE) == TRUE | grepl("larvae", entry, ignore.case = TRUE) == TRUE | grepl("eleutherolarva", entry, ignore.case = TRUE) == TRUE) {
      classifications <- c(classifications, "Larval")
    }
    if (grepl("juvenile", entry, ignore.case = TRUE) == TRUE | grepl("one month", entry, ignore.case = TRUE) == TRUE | grepl("two months", entry, ignore.case = TRUE) == TRUE) {
      classifications <- c(classifications, "Juvenile")
    }
  }
  
  if (length(classifications) == 0) {
    return("Undetermined")
  } else {
    # Split the string into words
    words <- unlist(strsplit(classifications, " "))
    # Remove duplicate words
    unique_words <- unique(words)
    # Combine the unique words back into a single string
    classifications <- paste(unique_words, collapse = ", ")
    return(paste(classifications, collapse = ", "))
  }
}





In [9]:
classify_stage_coarse2025 <- function(entry, devstage_full1) {
  classifications <- c()
  if (grepl("Undetermined", devstage_full1, ignore.case = TRUE) == TRUE) {
    ## moved these except for embryo into main devstage function, but also want to leave here for tertiary coarse-only
    ## but for adult, leave out year, even "adult", use instead "adult zebrafish" and "adult fish" ADDING 8/21 "adult stage" adding all mpf to adult...ONLY TO COARSE BTW
    # if (grepl("adult", entry, ignore.case = TRUE) == TRUE) {
    if (grepl("adult fish", entry, ignore.case = TRUE) == TRUE | grepl("adult zebrafish", entry, ignore.case = TRUE) == TRUE | grepl("adult stage", entry, ignore.case = TRUE) == TRUE | grepl("three month", entry, ignore.case = TRUE) == TRUE | grepl("mpf", entry, ignore.case = TRUE) == TRUE | grepl("four month", entry, ignore.case = TRUE) == TRUE | grepl("four mpf", entry, ignore.case = TRUE) == TRUE | grepl("five month", entry, ignore.case = TRUE) == TRUE | grepl("five mpf", entry, ignore.case = TRUE) == TRUE | grepl("six month", entry, ignore.case = TRUE) == TRUE | grepl("six mpf", entry, ignore.case = TRUE) == TRUE | grepl("seven month", entry, ignore.case = TRUE) == TRUE | grepl("seven mpf", entry, ignore.case = TRUE) == TRUE | grepl("eight month", entry, ignore.case = TRUE) == TRUE | grepl("eight mpf", entry, ignore.case = TRUE) == TRUE | grepl("natal month three", entry, ignore.case = TRUE) == TRUE) {
      classifications <- c(classifications, "Adult")
    }
    if (grepl("embryo", entry, ignore.case = TRUE) == TRUE | grepl("mzt", entry, ignore.case = TRUE) == TRUE | grepl("embyro", entry, ignore.case = TRUE) == TRUE | grepl("embryro", entry, ignore.case = TRUE) == TRUE | grepl("embroy", entry, ignore.case = TRUE) == TRUE) {
      classifications <- c(classifications, "Embryo")
    }
    if (grepl("larva", entry, ignore.case = TRUE) == TRUE | grepl("larval", entry, ignore.case = TRUE) == TRUE | grepl("larvae", entry, ignore.case = TRUE) == TRUE | grepl("eleutherolarva", entry, ignore.case = TRUE) == TRUE) {
      classifications <- c(classifications, "Larval")
    }
    if (grepl("juvenile", entry, ignore.case = TRUE) == TRUE | grepl("one month", entry, ignore.case = TRUE) == TRUE | grepl("two months", entry, ignore.case = TRUE) == TRUE) {
      classifications <- c(classifications, "Juvenile")
    }
    if (length(classifications) == 0) {
      return("Undetermined")
    } else {
      # Split the string into words
      words <- unlist(strsplit(classifications, " "))
      # Remove duplicate words
      unique_words <- unique(words)
      # Combine the unique words back into a single string
      classifications <- paste(unique_words, collapse = ", ")
      return(paste(classifications, collapse = ", "))
    }
  }
  if (grepl("Adult", devstage_full1, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Adult")
  }
  if (grepl("Juvenile", devstage_full1, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Juvenile")
  }
  if (grepl("Larval", devstage_full1, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Larval")
  }
  if (grepl("Zygote", devstage_full1, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Embryo")
  }
  if (grepl("Cleavage", devstage_full1, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Embryo")
  }
  if (grepl("Blastula", devstage_full1, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Embryo")
  }
  if (grepl("Gastrula", devstage_full1, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Embryo")
  }
  if (grepl("Segmentation", devstage_full1, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Embryo")
  }
  if (grepl("Pharyngula", devstage_full1, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Embryo")
  }
  if (grepl("Hatching", devstage_full1, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Embryo")
  }
  if (grepl("Embryo", devstage_full1, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Embryo")
  }
  if (length(classifications) == 0) {
    return("Undetermined")
  } else {
    # Split the string into words
    words <- unlist(strsplit(classifications, " "))
    # Remove duplicate words
    unique_words <- unique(words)
    # Combine the unique words back into a single string
    classifications <- paste(unique_words, collapse = ", ")
    return(paste(classifications, collapse = ", "))
  }
  
}




In [10]:
## tissue function here
## first a version for tissue-specific columns that doesn't require word boundaries

classify_tissue2025first <- function(entry) {
  classifications <- c()
  # First - Replace dashes with spaces in the entry - instead doing this & rest in separate function clean_text_columns (but keeping basic dash replace just in case) Oocyte microinjection
  removed1 <- "\\b\\w+ removed\\b"
  removed2 <- "\\b\\w+ was removed\\b"
  # Use gsub to apply the pattern
  entry <- gsub(removed1, "removed", entry)
  entry <- gsub(removed2, "removed", entry)
  entry <- gsub("cardiomyocyte ablation", "card cell ablation", entry)
  entry <- gsub("Oocyte microinjection", "egg microinjection", entry)
  entry <- gsub("except the heads", "except the fore", entry)
  entry <- gsub("pec fin", "pecfin", entry)
  entry <- gsub("thyroid hormone", "thyr hormone", entry)
  entry <- gsub("head kidney", "kidney", entry)
  entry <- gsub("headkidney", "kidney", entry)
  entry <- gsub("whole zebrafish brain", "whole zfish brain", entry)
  entry <- gsub("Paris Brain Institute", "Paris B Institute", entry)
  entry <- gsub("kidney marrow", "kidney", entry)
  entry <- gsub("head removed", "hd was removed", entry)
  entry <- gsub("tailfin cut", "xx cut", entry)
  entry <- gsub("head region cut", "xx cut", entry)
  entry <- gsub("removing gill  fin and tail tissue", "removing xx", entry)
  entry <- gsub("poly A tail", "poly A tai", entry)
  entry <- gsub("-", " ", entry)
  if (grepl("pituitary", entry, ignore.case = TRUE) == TRUE | grepl("hypophysis", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Pituitary Gland")
  }
  if (grepl("testis", entry, ignore.case = TRUE) == TRUE | grepl("testes", entry, ignore.case = TRUE) == TRUE | grepl("testicle", entry, ignore.case = TRUE) == TRUE | grepl("follicle", entry, ignore.case = TRUE) == TRUE | grepl("follicular", entry, ignore.case = TRUE) == TRUE | grepl("germ cell", entry, ignore.case = TRUE) == TRUE | grepl("gonad", entry, ignore.case = TRUE) == TRUE | grepl("ovary", entry, ignore.case = TRUE) == TRUE | grepl("ovaries", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Gonad")
  }
  if (grepl("sperm ", entry, ignore.case = TRUE) == TRUE | grepl("spermatozoa", entry, ignore.case = TRUE) == TRUE | grepl("zygote", entry, ignore.case = TRUE) == TRUE | grepl("unfertilized", entry, ignore.case = TRUE) == TRUE | grepl("oocyte", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Oocyte")
  }
  if (grepl("hepatocyte", entry, ignore.case = TRUE) == TRUE | grepl("liver", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Liver")
  }
  if (grepl("spinal cord", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Spinal Cord")
  }
  if (grepl("thyroid", entry, ignore.case = TRUE) == TRUE | grepl("thyrocyte", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Thyroid")
  }
  if (grepl("melanoma", entry, ignore.case = TRUE) == TRUE | grepl("\\bcancer\\b", entry, ignore.case = TRUE) == TRUE | grepl("oncogene", entry, ignore.case = TRUE) == TRUE | grepl("malignanc", entry, ignore.case = TRUE) == TRUE | grepl("neuroblastoma", entry, ignore.case = TRUE) == TRUE | grepl("leukemia", entry, ignore.case = TRUE) == TRUE | grepl("carcinoma", entry, ignore.case = TRUE) == TRUE | grepl("tumour", entry, ignore.case = TRUE) == TRUE | grepl("tumor", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Cancer or Tumor") ## for firstpass only adding "cancer" and only full word to avoid precancerous
  }
  if (grepl("bone marrow", entry, ignore.case = TRUE) == TRUE | grepl("marrow", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Marrow")
  }
  if (grepl("pancreas", entry, ignore.case = TRUE) == TRUE | grepl("beta cells", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Pancreas")
  }
  if (grepl("spleen", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Spleen")
  }
  if (grepl("heart", entry, ignore.case = TRUE) == TRUE | grepl("cardiomyocyte", entry, ignore.case = TRUE) == TRUE | grepl("cardium", entry, ignore.case = TRUE) == TRUE | grepl("aorta", entry, ignore.case = TRUE) == TRUE | grepl("atrium", entry, ignore.case = TRUE) == TRUE | grepl("cardiac", entry, ignore.case = TRUE) == TRUE | grepl("bulbus arteriosus", entry, ignore.case = TRUE) == TRUE | grepl("ventricle", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Heart")
  }
  if (grepl("muscle", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Muscle")
  }
  if (grepl("intestin", entry, ignore.case = TRUE) == TRUE | grepl("digestive", entry, ignore.case = TRUE) == TRUE | grepl("gastrointestinal", entry, ignore.case = TRUE) == TRUE | grepl("\\bgut\\b", entry, ignore.case = TRUE) == TRUE | grepl("gut ", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Gut") ## first pass only changing to "intestin" and adding whole word "gut"
  }
  if (grepl("neuromast", entry, ignore.case = TRUE) == TRUE | grepl("zebrafish neuromast hair cells", entry, ignore.case = TRUE) == TRUE | grepl("lateral line", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Lateral Line")
  }
  if (grepl("blood cell", entry, ignore.case = TRUE) == TRUE | grepl("matopoie", entry, ignore.case = TRUE) == TRUE | grepl("blood", entry, ignore.case = TRUE) == TRUE | grepl("macrophage", entry, ignore.case = TRUE) == TRUE | grepl("granuloma", entry, ignore.case = TRUE) == TRUE | grepl("granulocyte", entry, ignore.case = TRUE) == TRUE | grepl("phagocyte", entry, ignore.case = TRUE) == TRUE | grepl(" mast cell", entry, ignore.case = TRUE) == TRUE | grepl("eosinophil", entry, ignore.case = TRUE) == TRUE | grepl("basophil", entry, ignore.case = TRUE) == TRUE | grepl("neutrophil", entry, ignore.case = TRUE) == TRUE | grepl("erythroblast", entry, ignore.case = TRUE) == TRUE | grepl("erythroid cell", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Blood")
  }
  # if (grepl("macrophage", entry, ignore.case = TRUE) == TRUE | grepl("microglia", entry, ignore.case = TRUE) == TRUE | grepl("granuloma", entry, ignore.case = TRUE) == TRUE | grepl("granulocyte", entry, ignore.case = TRUE) == TRUE | grepl("phagocyte", entry, ignore.case = TRUE) == TRUE | grepl(" mast cell", entry, ignore.case = TRUE) == TRUE | grepl("eosinophil", entry, ignore.case = TRUE) == TRUE | grepl("basophil", entry, ignore.case = TRUE) == TRUE | grepl("neutrophil", entry, ignore.case = TRUE) == TRUE | grepl("erythroblast", entry, ignore.case = TRUE) == TRUE | grepl("erythroid cell", entry, ignore.case = TRUE) == TRUE) {
  #   classifications <- c(classifications, "Monocytes")
  # }
  if (grepl("kidney marrow", entry, ignore.case = TRUE) == TRUE | grepl("pronephros", entry, ignore.case = TRUE) == TRUE | grepl("mesonephros", entry, ignore.case = TRUE) == TRUE | grepl("DL segment", entry, ignore.case = TRUE) == TRUE | grepl("podocyte", entry, ignore.case = TRUE) == TRUE | grepl("nephrocyte", entry, ignore.case = TRUE) == TRUE | grepl("kidney", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Kidney")
  }
  if (grepl("pharyngeal arch", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Pharyngeal Arch")
  }
  if (grepl("swim bladder", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Swim Bladder")
  }
  if (grepl("pineal gland", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Pineal Gland")
  }
  if (grepl("cartilage", entry, ignore.case = TRUE) == TRUE | grepl("osteoblast", entry, ignore.case = TRUE) == TRUE | grepl("chondrocyte", entry, ignore.case = TRUE) == TRUE | grepl("osteoclast", entry, ignore.case = TRUE) == TRUE | grepl("bone", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Bone or Cartilage")
    ## terms chondrocyte & osteoclast do not pick up any additional rows - adding anyway!
  }
  if (grepl("jaw", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Jaw")
  }
  if (grepl("elasmoid", entry, ignore.case = TRUE) == TRUE | grepl("scale", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Scale") ## for firstpass only adding "scale"
  }
  if (grepl("melanocyte", entry, ignore.case = TRUE) == TRUE | grepl("keratinocyte", entry, ignore.case = TRUE) == TRUE | grepl("epidermal", entry, ignore.case = TRUE) == TRUE | grepl("epidermis", entry, ignore.case = TRUE) == TRUE | grepl("skin", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Skin")
  }
  if (grepl("adipose", entry, ignore.case = TRUE) == TRUE | grepl("adipocyte", entry, ignore.case = TRUE) == TRUE | grepl("fat ", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Adipose Tissue")
  }
  if (grepl("otolith", entry, ignore.case = TRUE) == TRUE | grepl("utricule", entry, ignore.case = TRUE) == TRUE | grepl("saccule", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Ear")
  }
  if ( grepl("lymphatic", entry, ignore.case = TRUE) == TRUE | grepl("lymph", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Lymphatic System")
  }
  if (grepl("thymus", entry, ignore.case = TRUE) == TRUE | grepl("thymocyte", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Thymus")
  }
  if (grepl("endothelium", entry, ignore.case = TRUE) == TRUE | grepl("endothelial cells", entry, ignore.case = TRUE) == TRUE | grepl("perivascular", entry, ignore.case = TRUE) == TRUE | grepl("endothelial", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Endothelium")
  }
  if (grepl("BCR", entry, ignore.case = TRUE) == TRUE | grepl("TCR", entry, ignore.case = TRUE) == TRUE | grepl("immunoglobulin", entry, ignore.case = TRUE) == TRUE | grepl(" IgH ", entry, ignore.case = FALSE) == TRUE) {
    classifications <- c(classifications, "BCR TCR repertoire")
  }
  if (grepl("telencephalon", entry, ignore.case = TRUE) == TRUE | grepl("microglia", entry, ignore.case = TRUE) == TRUE | grepl("optic nerve", entry, ignore.case = TRUE) == TRUE | grepl("optic tectum", entry, ignore.case = TRUE) == TRUE | grepl("meninge", entry, ignore.case = TRUE) == TRUE | grepl("ependymoglial", entry, ignore.case = TRUE) == TRUE | grepl("cerebellum", entry, ignore.case = TRUE) == TRUE | grepl("cerebrum", entry, ignore.case = TRUE) == TRUE | grepl("notochord", entry, ignore.case = TRUE) == TRUE | grepl("olfactory", entry, ignore.case = TRUE) == TRUE | grepl("neuron", entry, ignore.case = TRUE) == TRUE | grepl("neural crest", entry, ignore.case = TRUE) == TRUE | grepl("brain", entry, ignore.case = TRUE) == TRUE | grepl("hypothalamus", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Brain") ## adding optic nerve optic tectum meninge only to first
  }
  ## only using word boundaries for these sets - note for first pass removing word boundaries!
  if (grepl("trunk", entry, ignore.case = TRUE) == TRUE | grepl("abdomen", entry, ignore.case = TRUE) == TRUE | grepl("posterior half", entry, ignore.case = TRUE) == TRUE | grepl("abdominal", entry, ignore.case = TRUE) == TRUE | grepl("body", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Trunk")
  }
  if (grepl("fin", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Fin")
  }
  if (grepl("gill", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Gill")
  }
  if (grepl("tail", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Tail")
  }
  if (grepl("head", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Head")
  }
  if (grepl("retina", entry, ignore.case = TRUE) == TRUE | grepl("eye", entry, ignore.case = TRUE) == TRUE | grepl("photoreceptors", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Eye")
  }
  ### adding whole organism ONLY IF NOTHING ELSE FOUND
  if (length(classifications) == 0) {
    if (grepl("clutch", entry, ignore.case = TRUE) == TRUE | grepl("whole body", entry, ignore.case = TRUE) == TRUE | grepl("whole individual", entry, ignore.case = TRUE) == TRUE | grepl("whole tissue", entry, ignore.case = TRUE) == TRUE | grepl("whole eleutherolarvae", entry, ignore.case = TRUE) == TRUE | grepl("total fish", entry, ignore.case = TRUE) == TRUE | grepl("whole organism", entry, ignore.case = TRUE) == TRUE) {
      classifications <- c(classifications, "Whole Organism")
    }
    if (grepl("whole animal", entry, ignore.case = TRUE) == TRUE | grepl("whole fish", entry, ignore.case = TRUE) == TRUE | grepl("entire embryo", entry, ignore.case = TRUE) == TRUE | grepl("whole embryo", entry, ignore.case = TRUE) == TRUE | grepl("entire larvae", entry, ignore.case = TRUE) == TRUE | grepl("entire larva", entry, ignore.case = TRUE) == TRUE | grepl("whole larvae", entry, ignore.case = TRUE) == TRUE | grepl("complete larva", entry, ignore.case = TRUE) == TRUE | grepl("whole larva", entry, ignore.case = TRUE) == TRUE) {
      classifications <- c(classifications, "Whole Organism") ## only first pass changing "whole larva " to "whole larva" and adding "complete larva"
    }
    if (grepl("pooled male and female", entry, ignore.case = TRUE) == TRUE | grepl("pooled embryos", entry, ignore.case = TRUE) == TRUE | grepl("pooled larva", entry, ignore.case = TRUE) == TRUE | grepl("larvae total", entry, ignore.case = TRUE) == TRUE | grepl("pooled fish", entry, ignore.case = TRUE) == TRUE | grepl("pooled extract", entry, ignore.case = TRUE) == TRUE | grepl("pool of", entry, ignore.case = TRUE) == TRUE | grepl("whole adult", entry, ignore.case = TRUE) == TRUE | grepl("whole zebrafish", entry, ignore.case = TRUE) == TRUE) {
      classifications <- c(classifications, "Whole Organism")
    }
    ## moving cell line to only if nothing else found
    if (grepl("zf4", entry, ignore.case = TRUE) == TRUE | grepl("zfl", entry, ignore.case = TRUE) == TRUE |  grepl("AB9", entry, ignore.case = FALSE) == TRUE |  grepl("sjd", entry, ignore.case = TRUE) == TRUE | grepl("cell line", entry, ignore.case = TRUE) == TRUE) {
      classifications <- c(classifications, "Cell Line")
    }  
    ## moving imprecise embryo down to here without word boundaries
    if (grepl("embryo", entry, ignore.case = TRUE) == TRUE | grepl("embyro", entry, ignore.case = TRUE) == TRUE | grepl("embryro", entry, ignore.case = TRUE) == TRUE | grepl("embroy", entry, ignore.case = TRUE) == TRUE) {
      classifications <- c(classifications, "Embryo Imprecise")
    }
    
  }
  ## now if any whole organism is found, it removes other calls
  if ("Whole Organism" %in% classifications) {
    return("Whole Organism")
  }
  if (length(classifications) == 0) {
    return("Undetermined")
  } else {
    return(paste(classifications, collapse = ", "))
  }
}



In [11]:
## also OG version - updated for 2025 though with a few new gsub

classify_tissue2025 <- function(entry) {
  classifications <- c()
  # First - Replace dashes with spaces in the entry - instead doing this & rest in separate function clean_text_columns (but keeping basic dash replace just in case) Oocyte microinjection
  removed1 <- "\\b\\w+ removed\\b"
  removed2 <- "\\b\\w+ was removed\\b"
  # Use gsub to apply the pattern
  entry <- gsub(removed1, "removed", entry)
  entry <- gsub(removed2, "removed", entry)
  entry <- gsub("cardiomyocyte ablation", "card cell ablation", entry)
  entry <- gsub("Oocyte microinjection", "egg microinjection", entry)
  entry <- gsub("except the heads", "except the fore", entry)
  entry <- gsub("pec fin", "pecfin", entry)
  entry <- gsub("thyroid hormone", "thyr hormone", entry)
  entry <- gsub("head kidney", "kidney", entry)
  entry <- gsub("headkidney", "kidney", entry)
  entry <- gsub("whole zebrafish brain", "whole zfish brain", entry)
  entry <- gsub("Paris Brain Institute", "Paris B Institute", entry)
  entry <- gsub("kidney marrow", "kidney", entry)
  entry <- gsub("head removed", "hd was removed", entry)
  entry <- gsub("tailfin cut", "xx cut", entry)
  entry <- gsub("head region cut", "xx cut", entry)
  entry <- gsub("removing gill  fin and tail tissue", "removing xx", entry)
  entry <- gsub("poly A tail", "poly A tai", entry)
  entry <- gsub("-", " ", entry)
  if (grepl("pituitary", entry, ignore.case = TRUE) == TRUE | grepl("hypophysis", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Pituitary Gland")
  }
  if (grepl("testis", entry, ignore.case = TRUE) == TRUE | grepl("testes", entry, ignore.case = TRUE) == TRUE | grepl("testicle", entry, ignore.case = TRUE) == TRUE | grepl("follicle", entry, ignore.case = TRUE) == TRUE | grepl("follicular", entry, ignore.case = TRUE) == TRUE | grepl("germ cell", entry, ignore.case = TRUE) == TRUE | grepl("gonad", entry, ignore.case = TRUE) == TRUE | grepl("ovary", entry, ignore.case = TRUE) == TRUE | grepl("ovaries", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Gonad")
  }
  if (grepl("sperm ", entry, ignore.case = TRUE) == TRUE | grepl("spermatozoa", entry, ignore.case = TRUE) == TRUE | grepl("zygote", entry, ignore.case = TRUE) == TRUE | grepl("unfertilized", entry, ignore.case = TRUE) == TRUE | grepl("oocyte", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Oocyte")
  }
  if (grepl("hepatocyte", entry, ignore.case = TRUE) == TRUE | grepl("liver", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Liver")
  }
  if (grepl("spinal cord", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Spinal Cord")
  }
  if (grepl("thyroid", entry, ignore.case = TRUE) == TRUE | grepl("thyrocyte", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Thyroid")
  }
  if (grepl("zf4", entry, ignore.case = TRUE) == TRUE | grepl("zfl", entry, ignore.case = TRUE) == TRUE |  grepl("AB9", entry, ignore.case = FALSE) == TRUE |  grepl("sjd", entry, ignore.case = TRUE) == TRUE | grepl("cell line", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Cell Line")
  }
  if (grepl("melanoma", entry, ignore.case = TRUE) == TRUE | grepl("oncogene", entry, ignore.case = TRUE) == TRUE | grepl("malignanc", entry, ignore.case = TRUE) == TRUE | grepl("neuroblastoma", entry, ignore.case = TRUE) == TRUE | grepl("leukemia", entry, ignore.case = TRUE) == TRUE | grepl("carcinoma", entry, ignore.case = TRUE) == TRUE | grepl("tumour", entry, ignore.case = TRUE) == TRUE | grepl("tumor", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Cancer or Tumor")
  }
  if (grepl("bone marrow", entry, ignore.case = TRUE) == TRUE | grepl("marrow", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Marrow")
  }
  if (grepl("pancreas", entry, ignore.case = TRUE) == TRUE | grepl("beta cells", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Pancreas")
  }
  if (grepl("spleen", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Spleen")
  }
  if (grepl("heart", entry, ignore.case = TRUE) == TRUE | grepl("cardiomyocyte", entry, ignore.case = TRUE) == TRUE | grepl("cardium", entry, ignore.case = TRUE) == TRUE | grepl("aorta", entry, ignore.case = TRUE) == TRUE | grepl("atrium", entry, ignore.case = TRUE) == TRUE | grepl("cardiac", entry, ignore.case = TRUE) == TRUE | grepl("bulbus arteriosus", entry, ignore.case = TRUE) == TRUE | grepl("ventricle", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Heart")
  }
  if (grepl("muscle", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Muscle")
  }
  if (grepl("intestine", entry, ignore.case = TRUE) == TRUE | grepl("digestive", entry, ignore.case = TRUE) == TRUE | grepl("gastrointestinal", entry, ignore.case = TRUE) == TRUE | grepl("gut ", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Gut")
  }
  if (grepl("neuromast", entry, ignore.case = TRUE) == TRUE | grepl("zebrafish neuromast hair cells", entry, ignore.case = TRUE) == TRUE | grepl("lateral line", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Lateral Line")
  }
  if (grepl("blood cell", entry, ignore.case = TRUE) == TRUE | grepl("matopoie", entry, ignore.case = TRUE) == TRUE | grepl("blood", entry, ignore.case = TRUE) == TRUE | grepl("macrophage", entry, ignore.case = TRUE) == TRUE | grepl("granuloma", entry, ignore.case = TRUE) == TRUE | grepl("granulocyte", entry, ignore.case = TRUE) == TRUE | grepl("phagocyte", entry, ignore.case = TRUE) == TRUE | grepl(" mast cell", entry, ignore.case = TRUE) == TRUE | grepl("eosinophil", entry, ignore.case = TRUE) == TRUE | grepl("basophil", entry, ignore.case = TRUE) == TRUE | grepl("neutrophil", entry, ignore.case = TRUE) == TRUE | grepl("erythroblast", entry, ignore.case = TRUE) == TRUE | grepl("erythroid cell", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Blood")
  }
  # if (grepl("macrophage", entry, ignore.case = TRUE) == TRUE | grepl("microglia", entry, ignore.case = TRUE) == TRUE | grepl("granuloma", entry, ignore.case = TRUE) == TRUE | grepl("granulocyte", entry, ignore.case = TRUE) == TRUE | grepl("phagocyte", entry, ignore.case = TRUE) == TRUE | grepl(" mast cell", entry, ignore.case = TRUE) == TRUE | grepl("eosinophil", entry, ignore.case = TRUE) == TRUE | grepl("basophil", entry, ignore.case = TRUE) == TRUE | grepl("neutrophil", entry, ignore.case = TRUE) == TRUE | grepl("erythroblast", entry, ignore.case = TRUE) == TRUE | grepl("erythroid cell", entry, ignore.case = TRUE) == TRUE) {
  #   classifications <- c(classifications, "Monocytes")
  # }
  if (grepl("kidney marrow", entry, ignore.case = TRUE) == TRUE | grepl("pronephros", entry, ignore.case = TRUE) == TRUE | grepl("mesonephros", entry, ignore.case = TRUE) == TRUE | grepl("DL segment", entry, ignore.case = TRUE) == TRUE | grepl("podocyte", entry, ignore.case = TRUE) == TRUE | grepl("nephrocyte", entry, ignore.case = TRUE) == TRUE | grepl("kidney", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Kidney")
  }
  if (grepl("pharyngeal arch", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Pharyngeal Arch")
  }
  if (grepl("swim bladder", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Swim Bladder")
  }
  if (grepl("pineal gland", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Pineal Gland")
  }
  if (grepl("cartilage", entry, ignore.case = TRUE) == TRUE | grepl("osteoblast", entry, ignore.case = TRUE) == TRUE | grepl("bone", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Bone or Cartilage")
    ## terms chondrocyte & osteoclast do not pick up any additional rows
  }
  if (grepl("jaw", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Jaw")
  }
  if (grepl("elasmoid", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Scale")
  }
  if (grepl("melanocyte", entry, ignore.case = TRUE) == TRUE | grepl("keratinocyte", entry, ignore.case = TRUE) == TRUE | grepl("epidermal", entry, ignore.case = TRUE) == TRUE | grepl("epidermis", entry, ignore.case = TRUE) == TRUE | grepl("skin", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Skin")
  }
  if (grepl("adipose", entry, ignore.case = TRUE) == TRUE | grepl("adipocyte", entry, ignore.case = TRUE) == TRUE | grepl("fat ", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Adipose Tissue")
  }
  if (grepl("otolith", entry, ignore.case = TRUE) == TRUE | grepl("utricule", entry, ignore.case = TRUE) == TRUE | grepl("saccule", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Ear")
  }
  if ( grepl("lymphatic", entry, ignore.case = TRUE) == TRUE | grepl("lymph", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Lymphatic System")
  }
  if (grepl("thymus", entry, ignore.case = TRUE) == TRUE | grepl("thymocyte", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Thymus")
  }
  if (grepl("endothelium", entry, ignore.case = TRUE) == TRUE | grepl("endothelial cells", entry, ignore.case = TRUE) == TRUE | grepl("perivascular", entry, ignore.case = TRUE) == TRUE | grepl("endothelial", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Endothelium")
  }
  if (grepl("BCR", entry, ignore.case = TRUE) == TRUE | grepl("TCR", entry, ignore.case = TRUE) == TRUE | grepl("immunoglobulin", entry, ignore.case = TRUE) == TRUE | grepl(" IgH ", entry, ignore.case = FALSE) == TRUE) {
    classifications <- c(classifications, "BCR TCR repertoire")
  }
  if (grepl("telencephalon", entry, ignore.case = TRUE) == TRUE | grepl("microglia", entry, ignore.case = TRUE) == TRUE | grepl("optic nerve", entry, ignore.case = TRUE) == TRUE | grepl("optic tectum", entry, ignore.case = TRUE) == TRUE | grepl("meninge", entry, ignore.case = TRUE) == TRUE | grepl("ependymoglial", entry, ignore.case = TRUE) == TRUE | grepl("cerebellum", entry, ignore.case = TRUE) == TRUE | grepl("cerebrum", entry, ignore.case = TRUE) == TRUE | grepl("notochord", entry, ignore.case = TRUE) == TRUE | grepl("olfactory", entry, ignore.case = TRUE) == TRUE | grepl("\\bneuron\\b", entry, ignore.case = TRUE) == TRUE | grepl("neural crest", entry, ignore.case = TRUE) == TRUE | grepl("brain", entry, ignore.case = TRUE) == TRUE | grepl("hypothalamus", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Brain") ## adding optic nerve optic tectum meninge
  }
  ## only using word boundaries for these sets
  if (grepl("\\btrunk\\b", entry, ignore.case = TRUE) == TRUE | grepl("abdomen", entry, ignore.case = TRUE) == TRUE | grepl("posterior half", entry, ignore.case = TRUE) == TRUE | grepl("abdominal", entry, ignore.case = TRUE) == TRUE | grepl("\\bbody\\b", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Trunk")
  }
  if (grepl("\\bfin\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\bfins\\b", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Fin")
  }
  if (grepl("\\bgill\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\bgills\\b", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Gill")
  }
  if (grepl("\\btail\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\btails\\b", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Tail")
  }
  if (grepl("\\bhead\\b", entry, ignore.case = TRUE) == TRUE | grepl("whole head", entry, ignore.case = TRUE) == TRUE | grepl("head  without eyes", entry, ignore.case = TRUE) == TRUE | grepl("\\bheads\\b", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Head")
  }
  if (grepl("retina", entry, ignore.case = TRUE) == TRUE | grepl("\\beye\\b", entry, ignore.case = TRUE) == TRUE | grepl("\\beyes\\b", entry, ignore.case = TRUE) == TRUE | grepl("photoreceptors", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Eye")
  }
  ## moving whole fish up, but adding whole zebrafish here - 2025 update moving back down...
  # if (grepl("embryo", entry, ignore.case = TRUE) == TRUE | grepl("embyro", entry, ignore.case = TRUE) == TRUE | grepl("embryro", entry, ignore.case = TRUE) == TRUE | grepl("embroy", entry, ignore.case = TRUE) == TRUE) {
  #   classifications <- c(classifications, "Embryo Imprecise")
  # }
  ### adding whole organism ONLY IF NOTHING ELSE FOUND
  if (length(classifications) == 0) {
    if (grepl("clutch", entry, ignore.case = TRUE) == TRUE | grepl("whole body", entry, ignore.case = TRUE) == TRUE | grepl("whole individual", entry, ignore.case = TRUE) == TRUE | grepl("whole tissue", entry, ignore.case = TRUE) == TRUE | grepl("whole eleutherolarvae", entry, ignore.case = TRUE) == TRUE | grepl("total fish", entry, ignore.case = TRUE) == TRUE | grepl("whole organism", entry, ignore.case = TRUE) == TRUE) {
      classifications <- c(classifications, "Whole Organism")
    }
    if (grepl("whole animal", entry, ignore.case = TRUE) == TRUE | grepl("whole fish", entry, ignore.case = TRUE) == TRUE | grepl("entire embryo", entry, ignore.case = TRUE) == TRUE | grepl("whole embryo", entry, ignore.case = TRUE) == TRUE | grepl("entire larvae", entry, ignore.case = TRUE) == TRUE | grepl("entire larva", entry, ignore.case = TRUE) == TRUE | grepl("whole larvae", entry, ignore.case = TRUE) == TRUE | grepl("whole larva ", entry, ignore.case = TRUE) == TRUE) {
      classifications <- c(classifications, "Whole Organism")
    }
    if (grepl("pooled male and female", entry, ignore.case = TRUE) == TRUE | grepl("pooled embryos", entry, ignore.case = TRUE) == TRUE | grepl("pooled larva", entry, ignore.case = TRUE) == TRUE | grepl("larvae total", entry, ignore.case = TRUE) == TRUE | grepl("pooled fish", entry, ignore.case = TRUE) == TRUE | grepl("pooled extract", entry, ignore.case = TRUE) == TRUE | grepl("pool of", entry, ignore.case = TRUE) == TRUE | grepl("whole adult", entry, ignore.case = TRUE) == TRUE | grepl("whole zebrafish", entry, ignore.case = TRUE) == TRUE) {
      classifications <- c(classifications, "Whole Organism")
    }
   ## moving cell line & imprecise embryo to only if nothing else found
    if (grepl("zf4", entry, ignore.case = TRUE) == TRUE | grepl("zfl", entry, ignore.case = TRUE) == TRUE |  grepl("AB9", entry, ignore.case = FALSE) == TRUE |  grepl("sjd", entry, ignore.case = TRUE) == TRUE | grepl("cell line", entry, ignore.case = TRUE) == TRUE) {
      classifications <- c(classifications, "Cell Line")
    }  
    if (grepl("embryo", entry, ignore.case = TRUE) == TRUE | grepl("embyro", entry, ignore.case = TRUE) == TRUE | grepl("embryro", entry, ignore.case = TRUE) == TRUE | grepl("embroy", entry, ignore.case = TRUE) == TRUE) {
      classifications <- c(classifications, "Embryo Imprecise")
    }
  }
  ## now if any whole organism is found, it removes other calls
  if ("Whole Organism" %in% classifications) {
    return("Whole Organism")
  }
  if (length(classifications) == 0) {
    return("Undetermined")
  } else {
    return(paste(classifications, collapse = ", "))
  }
}


In [12]:
## note a few get changed to Multi-tissue, instead lets skip ahead and go straight to Multi-system for coarse!!
classify_tissue_coarse2025 <- function(entry, devstage_full) {
  classifications <- c()
  if (grepl("Undetermined", entry, ignore.case = TRUE) == TRUE) {
    if (grepl("Embryo", devstage_full, ignore.case = TRUE) == TRUE) {
      classifications <- c(classifications, "Embryo Imprecise")
    }
  }
  if (grepl("Gonad", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Reproductive System")
  }
  if (grepl("Head", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Nervous System")
  }
  if (grepl("Trunk", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Surface Structure")
  }
  if (grepl("Swim Bladder", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Swim Bladder")
  }
  if (grepl("Brain", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Nervous System")
  }
  if (grepl("Heart", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Cardiovascular System")
  }
  if (grepl("Kidney", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Renal System")
  }
  if (grepl("Oocyte", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Reproductive System")
  }
  if (grepl("BCR TCR repertoire", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Hematopoietic System")
  }
  if (grepl("Liver", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Liver and Biliary System")
  }
  if (grepl("Eye", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Sensory System")
  }
  if (grepl("Skin", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Surface Structure")
  }
  if (grepl("Tail", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Multi-system")
  }
  if (grepl("Muscle", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Muscular System")
  }
  if (grepl("Pineal Gland", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Endocrine System")
  }
  if (grepl("Lymphatic System", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Cardiovascular System")
  }
  if (grepl("Thymus", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Hematopoietic System")
  }
  if (grepl("Endothelium", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Cardiovascular System")
  }
  if (grepl("Fin", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Surface Structure")
  }
  if (grepl("Blood", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Hematopoietic System")
  }
  if (grepl("Gut", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Digestive System")
  }
  if (grepl("Gill", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Respiratory System")
  }
  if (grepl("Bone or Cartilage", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Skeletal Element")
  }
  if (grepl("Jaw", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Surface Structure")
  }
  if (grepl("Scale", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Surface Structure")
  }
  if (grepl("Pharyngeal Arch", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Multi-system")
  }
  if (grepl("Spleen", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Hematopoietic System")
  }
  if (grepl("Thyroid", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Endocrine System")
  }
  if (grepl("Pancreas", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Endocrine System")
  }
  if (grepl("Spinal Cord", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Nervous System")
  }
  if (grepl("Marrow", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Renal System")
  }
  if (grepl("\\bEar\\b", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Sensory System")
  }
  if (grepl("Pituitary Gland", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Endocrine System")
  }
  if (grepl("Lateral Line", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Sensory System")
  }
  if (grepl("Adipose Tissue", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Adipose Tissue")
  }
  if (grepl("Multi-tissue", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Multi-system")
  }
  if (grepl("Cancer or Tumor", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Cancer or Tumor")
  }
  if (grepl("Cell Line", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "Cell Line")
  }
  if (grepl("Whole Organism", entry, ignore.case = TRUE) == TRUE) {
    classifications <- c(classifications, "All anatomical structures")
  }
  ## idea - we want to only add this if the entry is ONLY embryo imprecise!!
  if (entry == "Embryo Imprecise") {
    classifications <- c(classifications, "All anatomical structures")
  }
  if (length(classifications) == 0) {
    return("Undetermined")
  } else {
    return(paste(classifications, collapse = ", "))
  }
}



In [13]:
## also add a step to convert multiple systems to "Multiple Systems" - this is a separate code chunk but also a new chunk in coarse tissue function
## also want to simplify - if Embryo Imprecise found anywhere, make the coarse label just Embryo Imprecise
## similarly if All anatomical structures found anywhere, make coarse label just All anatomical structures

# Dev stage functions




## First run the _clean_text_columns_ function



In [14]:
metadata_curationstep1

run.accession,study.accession,bioproject,study.title,sample.accession,experiment.accession,earliest_date,experiment.title,experiment.attributes,sample.title,⋯,experiment.library_construction_protocol,sample.description,sample.alias,GEOsample.dataprocessing,GEOsample.source,GEOsample.treatmentprotocol,GEOsample.extractprotocol,GEOsample.growthprotocol,study.abstract,study.attributes
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
DRR314108,DRP008373,PRJDB12134,Comparison of expression profile between banp mutant and wildtype sibling.,DRS233566,DRX303511,2022-04-01,Illumina NovaSeq 6000 paired end sequencing of SAMD00399013,NA,RNA-seq of wild-type sibling sample3,⋯,1,NA,SAMD00399013,NA,NA,NA,NA,NA,"To characterize the physiological function of Banp, the expression profile of banp mutant and wild-type sibling was obtained by ATAC sequencing and RNA sequencing. All data were obtained using embryo heads at 48 hpf.",NA
DRR314107,DRP008373,PRJDB12134,Comparison of expression profile between banp mutant and wildtype sibling.,DRS233565,DRX303510,2022-04-01,Illumina NovaSeq 6000 paired end sequencing of SAMD00399012,NA,RNA-seq of wild-type sibling sample2,⋯,1,NA,SAMD00399012,NA,NA,NA,NA,NA,"To characterize the physiological function of Banp, the expression profile of banp mutant and wild-type sibling was obtained by ATAC sequencing and RNA sequencing. All data were obtained using embryo heads at 48 hpf.",NA
DRR314106,DRP008373,PRJDB12134,Comparison of expression profile between banp mutant and wildtype sibling.,DRS233564,DRX303509,2022-04-01,Illumina NovaSeq 6000 paired end sequencing of SAMD00399011,NA,RNA-seq of wild-type sibling sample1,⋯,1,NA,SAMD00399011,NA,NA,NA,NA,NA,"To characterize the physiological function of Banp, the expression profile of banp mutant and wild-type sibling was obtained by ATAC sequencing and RNA sequencing. All data were obtained using embryo heads at 48 hpf.",NA
DRR314105,DRP008373,PRJDB12134,Comparison of expression profile between banp mutant and wildtype sibling.,DRS233563,DRX303508,2022-04-01,Illumina NovaSeq 6000 paired end sequencing of SAMD00399010,NA,RNA-seq of banp mutant sample3,⋯,1,NA,SAMD00399010,NA,NA,NA,NA,NA,"To characterize the physiological function of Banp, the expression profile of banp mutant and wild-type sibling was obtained by ATAC sequencing and RNA sequencing. All data were obtained using embryo heads at 48 hpf.",NA
DRR314104,DRP008373,PRJDB12134,Comparison of expression profile between banp mutant and wildtype sibling.,DRS233562,DRX303507,2022-04-01,Illumina NovaSeq 6000 paired end sequencing of SAMD00399009,NA,RNA-seq of banp mutant sample2,⋯,1,NA,SAMD00399009,NA,NA,NA,NA,NA,"To characterize the physiological function of Banp, the expression profile of banp mutant and wild-type sibling was obtained by ATAC sequencing and RNA sequencing. All data were obtained using embryo heads at 48 hpf.",NA
DRR314103,DRP008373,PRJDB12134,Comparison of expression profile between banp mutant and wildtype sibling.,DRS233561,DRX303506,2022-04-01,Illumina NovaSeq 6000 paired end sequencing of SAMD00399008,NA,RNA-seq of banp mutant sample1,⋯,1,NA,SAMD00399008,NA,NA,NA,NA,NA,"To characterize the physiological function of Banp, the expression profile of banp mutant and wild-type sibling was obtained by ATAC sequencing and RNA sequencing. All data were obtained using embryo heads at 48 hpf.",NA
DRR315802,DRP008318,PRJDB12206,Transcriptome analysis of strip1 mutant and wildtype zebrafish eyes,DRS231989,DRX305194,2022-03-16,Illumina NovaSeq 6000 paired end sequencing of SAMD00400823,NA,wildtype sibling sample4,⋯,1,NA,SAMD00400823,NA,NA,NA,NA,NA,"Strip1 plays essential roles in the developing zebrafish retinal neural circuit. To identify the underlying molecular mechanisms at the transcriptomic level, transcriptome of strip1 mutant ""rw147"" eye cups at 2.5 days post fertilization was compared to that of wild-type siblings using bulk

In [15]:
### first run clean columns also add "study.title"
metadata_curationstep1clean <- clean_text_columns(metadata_curationstep1, c("study.title","experiment.title","experiment.attributes", "sample.title", "sample.attributes", "GEOsample.title", "GEOsample.characteristics", "run.title", "run.alias", "run.attributes", "experiment.alias", "experiment.library_name", "experiment.design_description", "experiment.library_construction_protocol", "sample.description", "sample.alias", "GEOsample.dataprocessing", "GEOsample.source", "GEOsample.treatmentprotocol", "GEOsample.extractprotocol", "GEOsample.growthprotocol", "study.abstract", "study.attributes"))



In [16]:
metadata_curationstep1clean

run.accession,study.accession,bioproject,study.title,sample.accession,experiment.accession,earliest_date,experiment.title,experiment.attributes,sample.title,⋯,experiment.library_construction_protocol,sample.description,sample.alias,GEOsample.dataprocessing,GEOsample.source,GEOsample.treatmentprotocol,GEOsample.extractprotocol,GEOsample.growthprotocol,study.abstract,study.attributes
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
DRR314108,DRP008373,PRJDB12134,Comparison of expression profile between banp mutant and wildtype sibling.,DRS233566,DRX303511,2022-04-01,Illumina NovaSeq 6000 paired end sequencing of SAMD00399013,NA,RNA seq of wild type sibling sample3,⋯,1,NA,SAMD00399013,NA,NA,NA,NA,NA,To characterize the physiological function of Banp the expression profile of banp mutant and wild type sibling was obtained by ATAC sequencing and RNA sequencing. All data were obtained using embryo heads at 48 hpf.,NA
DRR314107,DRP008373,PRJDB12134,Comparison of expression profile between banp mutant and wildtype sibling.,DRS233565,DRX303510,2022-04-01,Illumina NovaSeq 6000 paired end sequencing of SAMD00399012,NA,RNA seq of wild type sibling sample2,⋯,1,NA,SAMD00399012,NA,NA,NA,NA,NA,To characterize the physiological function of Banp the expression profile of banp mutant and wild type sibling was obtained by ATAC sequencing and RNA sequencing. All data were obtained using embryo heads at 48 hpf.,NA
DRR314106,DRP008373,PRJDB12134,Comparison of expression profile between banp mutant and wildtype sibling.,DRS233564,DRX303509,2022-04-01,Illumina NovaSeq 6000 paired end sequencing of SAMD00399011,NA,RNA seq of wild type sibling sample1,⋯,1,NA,SAMD00399011,NA,NA,NA,NA,NA,To characterize the physiological function of Banp the expression profile of banp mutant and wild type sibling was obtained by ATAC sequencing and RNA sequencing. All data were obtained using embryo heads at 48 hpf.,NA
DRR314105,DRP008373,PRJDB12134,Comparison of expression profile between banp mutant and wildtype sibling.,DRS233563,DRX303508,2022-04-01,Illumina NovaSeq 6000 paired end sequencing of SAMD00399010,NA,RNA seq of banp mutant sample3,⋯,1,NA,SAMD00399010,NA,NA,NA,NA,NA,To characterize the physiological function of Banp the expression profile of banp mutant and wild type sibling was obtained by ATAC sequencing and RNA sequencing. All data were obtained using embryo heads at 48 hpf.,NA
DRR314104,DRP008373,PRJDB12134,Comparison of expression profile between banp mutant and wildtype sibling.,DRS233562,DRX303507,2022-04-01,Illumina NovaSeq 6000 paired end sequencing of SAMD00399009,NA,RNA seq of banp mutant sample2,⋯,1,NA,SAMD00399009,NA,NA,NA,NA,NA,To characterize the physiological function of Banp the expression profile of banp mutant and wild type sibling was obtained by ATAC sequencing and RNA sequencing. All data were obtained using embryo heads at 48 hpf.,NA
DRR314103,DRP008373,PRJDB12134,Comparison of expression profile between banp mutant and wildtype sibling.,DRS233561,DRX303506,2022-04-01,Illumina NovaSeq 6000 paired end sequencing of SAMD00399008,NA,RNA seq of banp mutant sample1,⋯,1,NA,SAMD00399008,NA,NA,NA,NA,NA,To characterize the physiological function of Banp the expression profile of banp mutant and wild type sibling was obtained by ATAC sequencing and RNA sequencing. All data were obtained using embryo heads at 48 hpf.,NA
DRR315802,DRP008318,PRJDB12206,Transcriptome analysis of strip1 mutant and wildtype zebrafish eyes,DRS231989,DRX305194,2022-03-16,Illumina NovaSeq 6000 paired end sequencing of SAMD00400823,NA,wildtype sibling sample4,⋯,1,NA,SAMD00400823,NA,NA,NA,NA,NA,"Strip1 plays essential roles in the developing zebrafish retinal neural circuit. To identify the underlying molecular mechanisms at the transcriptomic level transcriptome of strip1 mutant ""rw147"" eye cups at 2.5 dpf was compared to that of wild type siblings using bulk RNA sequencing analysis.",NA
DRR315801

## Extracting terms from 'devstage:' 'age:' etc.

Not a function per se, but commands:

Note we are also pulling 'tissue:' for tissue step...

In [17]:
## next get _age _stage _tissue for the primary 6 columns
## adding hours post fertilization:97 conversion in replace string portion
metadata_curationstep2 <- metadata_curationstep1clean %>%
  mutate(
    across(
      c(experiment.title, experiment.attributes, sample.title, sample.attributes, GEOsample.title, GEOsample.characteristics),
      ~ .x %>%
        ## remove all tildes and parentheses also - & , to spaces (and further down _ to space)
        str_replace_all("[~()]", "") %>% 
        str_replace_all("-", " ") %>%
        str_replace_all(",", " ") %>% 
        str_replace_all("_", " ") %>% 
        str_replace_all(regex("thirty two", ignore_case = TRUE), "32") %>%
        str_replace_all(regex("thirty", ignore_case = TRUE), "30") %>%
        str_replace_all(regex("forty eight", ignore_case = TRUE), "48") %>%
        str_replace_all(regex("fourteen", ignore_case = TRUE), "14") %>%
        str_replace_all(regex("twenty nine", ignore_case = TRUE), "29") %>%
        str_replace_all(regex("twenty two", ignore_case = TRUE), "22") %>%
        str_replace_all(regex("one", ignore_case = TRUE), "1") %>%
        str_replace_all(regex("two", ignore_case = TRUE), "2") %>%
        str_replace_all(regex("three", ignore_case = TRUE), "3") %>%
        str_replace_all(regex("four", ignore_case = TRUE), "4") %>%
        str_replace_all(regex("five", ignore_case = TRUE), "5") %>%
        str_replace_all(regex("six", ignore_case = TRUE), "6") %>%
        str_replace_all(regex("seven", ignore_case = TRUE), "7") %>%
        str_replace_all(regex("eight", ignore_case = TRUE), "8") %>%
        str_replace_all(regex("nine", ignore_case = TRUE), "9") %>%
        # your existing “lesion stage” → “lesion stg”
        str_replace_all(regex("lesion stage", ignore_case = TRUE), "lesion stg") %>%
        # you can add more str_replace_all() lines here as needed disease_stage
        str_replace_all(regex("regeneration stage", ignore_case = TRUE), "regeneration stg") %>%
        str_replace_all(regex("at larval stage", ignore_case = TRUE), "at larval stg") %>%
        str_replace_all(regex("treatment /stage", ignore_case = TRUE), "treatment stg") %>%
        str_replace_all(regex("tumor stage", ignore_case = TRUE), "tumor stg") %>%
        str_replace_all(regex("age:4.5 MO", ignore_case = TRUE), "age:4.5 months") %>%
        str_replace_all(regex("age:4.6 MO", ignore_case = TRUE), "age:4.6 months") %>%
        str_replace_all(regex("age:4.7 MO", ignore_case = TRUE), "age:4.7 months") %>%
        str_replace_all(regex("age:5.0 MO", ignore_case = TRUE), "age:5.0 months") %>%
        str_replace_all(regex("age:5.O MO", ignore_case = TRUE), "age:5.0 months") %>%
        str_replace_all(regex("age:7.5 MO", ignore_case = TRUE), "age:7.5 months") %>%
        str_replace_all(regex("age:8.4 MO", ignore_case = TRUE), "age:8.4 months") %>%
        str_replace_all(regex("age:10.1 MO", ignore_case = TRUE), "age:10.1 months") %>%
        ## also 1-cell| with 1cell| note have to add \\ before pipe and only in these columns 1 cell embryos| and just fertilized 1 cell embryos also 1 cell embryo 
        str_replace_all(regex(":1 cell\\|", ignore_case = TRUE), ":1cell\\|") %>%
        str_replace_all(regex(":1 cell embryos\\|", ignore_case = TRUE), ":1cell embryos\\|") %>%
        str_replace_all(regex("just fertilized 1 cell embryos", ignore_case = TRUE), "just fertilized 1cell embryos") %>%
        str_replace_all(regex("1 cell embryo ", ignore_case = TRUE), "1cell embryo ") %>%
        # str_replace_all(regex(" MO\\|", ignore_case = FALSE), "months") %>% - hmm only replace in _age columns? better if adding a pipe? no best to skip
        ## note disease_stage is changed to disease stage above
        str_replace_all(regex("disease stage", ignore_case = TRUE), "disease stg") %>%
        # new rule: capture the digits after "hours post fertilization:" and append "hpf"
        #### match “hours post fertilization:”, capture digits, then a literal “|”
        #str_replace_all(regex("hours post fertilization:(\\d+)", ignore_case = TRUE), "\\1hpf") %>% 
        str_replace_all(regex("hours post fertilization:(\\d+)\\|", ignore_case = TRUE),
          # replace with “stage:<digits>hpf|”
          "stage:\\1hpf|") %>% 
        # new rule: “injection at 1-cell stage” → “injection at 1-cell stg”
        str_replace_all(regex("injection at 1-cell stage", ignore_case = TRUE), "injection at 1-cell stg")
      
    )
  ) %>%
# 1b) normalize "source name:" and "cell type:" to "tissue:" when "tissue:" is not present
## update - only should replace one term and cell type is always redundant with source name
  mutate(
    across(
      c(experiment.title, experiment.attributes, sample.title, sample.attributes, GEOsample.title, GEOsample.characteristics),
      ~ if_else(
        !str_detect(., regex("tissue:", ignore_case = TRUE)),
        str_replace_all(., regex("source name:", ignore_case = TRUE), "tissue:"),
        .
      )
    )
  ) %>%
  # 2) extract *all* tage: and age: patterns, collapsing with "|"
  mutate(
    across(
      c(experiment.title, experiment.attributes, sample.title, sample.attributes, GEOsample.title, GEOsample.characteristics),
      .fns = list(
        # 1) match "tage:" followed by anything up to | or end
        devstage = ~ map_chr(
          str_extract_all(., "(?<=tage:)[^|]+"),
          ~ if (length(.x)>0) paste(.x, collapse="|") else NA_character_
        ),
        # 2) match "age:" at start OR after a pipe 
        age = ~ map_chr(
          str_extract_all(., "(?<=^age:|\\|age:)[^|]+"),
          ~ if (length(.x)>0) paste(.x, collapse="|") else NA_character_
        ),
        # 3) match "tissue:" followed by anything up to | or end
        tissue = ~ map_chr(
          str_extract_all(., "(?<=tissue:)[^|]+"),
          ~ if (length(.x)>0) paste(.x, collapse="|") else NA_character_
        )
      ),
      .names = "{.col}_{.fn}"
    )
  )


In [18]:
## move the non six primary columns and their new versions to the end...
metadata_curationstep2 <- metadata_curationstep2 %>% relocate(run.title,run.alias,run.attributes,experiment.alias,experiment.library_name,experiment.design_description,experiment.library_construction_protocol,sample.description,sample.alias,GEOsample.dataprocessing,GEOsample.source,GEOsample.treatmentprotocol,GEOsample.extractprotocol,GEOsample.growthprotocol,study.abstract,study.attributes, .after = last_col())


In [19]:
## saving intermediate step
# update - saving the writing to the very end after renaming of columns...
#write.table(metadata_curationstep2, file = paste0("zf_metadata_curationstep2clean.tsv"), sep = "\t", row.names = FALSE, quote = FALSE)


## First dev stage pass - searching these _devstage and _age columns

Using the classify_stage2025first function

In [20]:
metadata_curationstep3 <- metadata_curationstep2

In [21]:
## iterate through many of the columns
# classify_stage2025first function on 6 x _devstage & _age columns
## work through the list c("experiment.title","experiment.attributes", "sample.title", "sample.attributes", "GEOsample.title", "GEOsample.characteristics", "run.title", "run.alias", "run.attributes", "experiment.alias", "experiment.library_name", "experiment.design_description", "experiment.library_construction_protocol", "sample.description", "sample.alias", "GEOsample.dataprocessing", "GEOsample.source", "GEOsample.treatmentprotocol", "GEOsample.extractprotocol", "GEOsample.growthprotocol", "study.abstract", "study.attributes"))

metadata_curationstep3$sample.attributes_devstage_curated <- sapply(metadata_curationstep3$sample.attributes_devstage, classify_stage2025first)
metadata_curationstep3$sample.attributes_age_curated <- sapply(metadata_curationstep3$sample.attributes_age, classify_stage2025first)
metadata_curationstep3$sample.title_devstage_curated <- sapply(metadata_curationstep3$sample.title_devstage, classify_stage2025first)
metadata_curationstep3$sample.title_age_curated <- sapply(metadata_curationstep3$sample.title_age, classify_stage2025first)


metadata_curationstep3$experiment.attributes_devstage_curated <- sapply(metadata_curationstep3$experiment.attributes_devstage, classify_stage2025first)
metadata_curationstep3$experiment.attributes_age_curated <- sapply(metadata_curationstep3$experiment.attributes_age, classify_stage2025first)
metadata_curationstep3$experiment.title_devstage_curated <- sapply(metadata_curationstep3$experiment.title_devstage, classify_stage2025first)
metadata_curationstep3$experiment.title_age_curated <- sapply(metadata_curationstep3$experiment.title_age, classify_stage2025first)

metadata_curationstep3$GEOsample.title_devstage_curated <- sapply(metadata_curationstep3$GEOsample.title_devstage, classify_stage2025first)
metadata_curationstep3$GEOsample.title_age_curated <- sapply(metadata_curationstep3$GEOsample.title_age, classify_stage2025first)
metadata_curationstep3$GEOsample.characteristics_devstage_curated <- sapply(metadata_curationstep3$GEOsample.characteristics_devstage, classify_stage2025first)
metadata_curationstep3$GEOsample.characteristics_age_curated <- sapply(metadata_curationstep3$GEOsample.characteristics_age, classify_stage2025first)


In [22]:
## now combine all of these terms into one column

metadata_curationstep3c <- metadata_curationstep3 %>% unite("runexpsample_devstage_curated_first0", ends_with("_curated"), sep = ",", remove = TRUE, na.rm = TRUE)
metadata_curationstep3c <- metadata_curationstep3c %>% arrange(runexpsample_devstage_curated_first0)




In [23]:
# Clean labels
# Function to clean a single label string
cleanlabels1 <- function(label_string) {
  parts <- str_split(label_string, ",")[[1]] %>%
    str_trim()
  deduped <- parts[!duplicated(parts)]
  cleaned <- deduped[deduped != "Undetermined"]
  if (length(cleaned) == 0) {
    return("Undetermined")
  } else {
    sorted <- sort(cleaned)  # <-- this makes the order consistent
    return(paste(sorted, collapse = ","))
  }
}


In [24]:
# Apply the cleaning function to the column
metadata_curationstep4 <- metadata_curationstep3c %>% mutate(runexpsample_devstage_curated_first = map_chr(runexpsample_devstage_curated_first0, cleanlabels1))


In [25]:
## check
metadata_curationstep4 %>% group_by(runexpsample_devstage_curated_first) %>% summarize(n = n()) %>% arrange(desc(n)) %>% print(n=35)


# A tibble: 29 × 2
   runexpsample_devstage_curated_first     n
   <chr>                               <int>
 1 Undetermined                        24676
 2 Larval                              15953
 3 Adult                               12571
 4 Pharyngula                           6708
 5 Hatching                             5179
 6 Gastrula                             4459
 7 Segmentation                         3471
 8 Blastula                             2163
 9 Juvenile                              984
10 Cleavage                              532
11 Zygote                                299
12 Juvenile,Larval                        79
13 Hatching,Larval                        76
14 Pharyngula,Segmentation                46
15 Adult,Juvenile                         39
16 Blastula,Gastrula                      15
17 Larval,Pharyngula                      15
18 Adult,Pharyngula                       12
19 Adult,Hatching                          8
20 Adult,Larval                     

## Second dev stage pass - searching all run/experiment/sample/GEOsample columns

Using the classify_stage2025 function

In [26]:
metadata_curationstep4$experiment.title_curated <- sapply(metadata_curationstep4$experiment.title, classify_stage2025)
metadata_curationstep4$experiment.attributes_curated <- sapply(metadata_curationstep4$experiment.attributes, classify_stage2025)
metadata_curationstep4$sample.title_curated <- sapply(metadata_curationstep4$sample.title, classify_stage2025)
metadata_curationstep4$sample.attributes_curated <- sapply(metadata_curationstep4$sample.attributes, classify_stage2025)
metadata_curationstep4$GEOsample.title_curated <- sapply(metadata_curationstep4$GEOsample.title, classify_stage2025)
metadata_curationstep4$GEOsample.characteristics_curated <- sapply(metadata_curationstep4$GEOsample.characteristics, classify_stage2025)

#metadata_curationstep4o <- metadata_curationstep4

metadata_curationstep4$run.title_curated <- sapply(metadata_curationstep4$run.title, classify_stage2025)
metadata_curationstep4$run.alias_curated <- sapply(metadata_curationstep4$run.alias, classify_stage2025)
metadata_curationstep4$run.attributes_curated <- sapply(metadata_curationstep4$run.attributes, classify_stage2025)
metadata_curationstep4$experiment.alias_curated <- sapply(metadata_curationstep4$experiment.alias, classify_stage2025)
metadata_curationstep4$experiment.library_name_curated <- sapply(metadata_curationstep4$experiment.library_name, classify_stage2025)
metadata_curationstep4$experiment.design_description_curated <- sapply(metadata_curationstep4$experiment.design_description, classify_stage2025)
metadata_curationstep4$experiment.library_construction_protocol_curated <- sapply(metadata_curationstep4$experiment.library_construction_protocol, classify_stage2025)


metadata_curationstep4$sample.description_curated <- sapply(metadata_curationstep4$sample.description, classify_stage2025)
metadata_curationstep4$sample.alias_curated <- sapply(metadata_curationstep4$sample.alias, classify_stage2025)
metadata_curationstep4$GEOsample.dataprocessing_curated <- sapply(metadata_curationstep4$GEOsample.dataprocessing, classify_stage2025)
metadata_curationstep4$GEOsample.source_curated <- sapply(metadata_curationstep4$GEOsample.source, classify_stage2025)
metadata_curationstep4$GEOsample.treatmentprotocol_curated <- sapply(metadata_curationstep4$GEOsample.treatmentprotocol, classify_stage2025)
metadata_curationstep4$GEOsample.extractprotocol_curated <- sapply(metadata_curationstep4$GEOsample.extractprotocol, classify_stage2025)
metadata_curationstep4$GEOsample.growthprotocol_curated <- sapply(metadata_curationstep4$GEOsample.growthprotocol, classify_stage2025)


In [27]:
## another round of uniting note only newest columns still end in _curated

metadata_curationstep4c <- metadata_curationstep4 %>% unite("runexpsample_devstage_curated_second0", ends_with("_curated"), sep = ",", remove = TRUE, na.rm = TRUE)


In [28]:
# Apply the cleaning function to the column
metadata_curationstep5 <- metadata_curationstep4c %>% mutate(runexpsample_devstage_curated_second = map_chr(runexpsample_devstage_curated_second0, cleanlabels1))


In [29]:
## now coalesce only after creating NAs from curated_first when Unclassified
metadata_curationstep5 <- metadata_curationstep5 %>% mutate(runexpsample_devstage_curated_first = na_if(runexpsample_devstage_curated_first, "Undetermined"))
metadata_curationstep5 <- metadata_curationstep5 %>% mutate(runexpsample_devstage_curated_firstandsecond = coalesce(runexpsample_devstage_curated_first,runexpsample_devstage_curated_second))

## moving some columns to back
metadata_curationstep5 <- metadata_curationstep5 %>% relocate(runexpsample_devstage_curated_first0, .after = last_col())
metadata_curationstep5 <- metadata_curationstep5 %>% relocate(runexpsample_devstage_curated_second0, .after = last_col())


In [30]:
## add undetermined back in 
metadata_curationstep5$runexpsample_devstage_curated_first <- metadata_curationstep5$runexpsample_devstage_curated_first %>% replace_na("Undetermined")


In [31]:
## check this result
metadata_curationstep5 %>% group_by(runexpsample_devstage_curated_firstandsecond) %>% summarize(n = n()) %>% arrange(desc(n)) %>% print(n=35)


# A tibble: 67 × 2
   runexpsample_devstage_curated_firstandsecond     n
   <chr>                                        <int>
 1 Larval                                       19709
 2 Adult                                        15670
 3 Undetermined                                 10064
 4 Pharyngula                                    7444
 5 Hatching                                      5782
 6 Gastrula                                      4676
 7 Segmentation                                  3626
 8 Juvenile                                      2710
 9 Blastula                                      2320
10 Blastula,Cleavage,Gastrula                    1157
11 Cleavage                                       676
12 Zygote                                         538
13 Larval,Pharyngula                              267
14 Adult,Larval                                   266
15 Gastrula,Larval                                204
16 Hatching,Larval                                199
17 Blastu

## Third dev stage pass - running coarse function on the same run/experiment/sample/GEOsample columns curation columns

Using the classify_stage_coarse2025 function

In [32]:
metadata_curationstep6 <- metadata_curationstep5


In [33]:
## note the coarse curation function looks at runexpsample_devstage_curated_firstandsecond column and both creates
## coarse category based on it, it also searches again for keywords if it is Undetermined

In [34]:
# coarse function on the 20 main run/experiment/sample/GEOsample columns
metadata_curationstep6$experiment.title_coarsecurated <- mapply(classify_stage_coarse2025, metadata_curationstep6$experiment.title, metadata_curationstep6$runexpsample_devstage_curated_firstandsecond)
metadata_curationstep6$experiment.attributes_coarsecurated <- mapply(classify_stage_coarse2025, metadata_curationstep6$experiment.attributes, metadata_curationstep6$runexpsample_devstage_curated_firstandsecond)
metadata_curationstep6$sample.title_coarsecurated <- mapply(classify_stage_coarse2025, metadata_curationstep6$sample.title, metadata_curationstep6$runexpsample_devstage_curated_firstandsecond)

metadata_curationstep6$sample.attributes_coarsecurated <- mapply(classify_stage_coarse2025, metadata_curationstep6$sample.attributes, metadata_curationstep6$runexpsample_devstage_curated_firstandsecond)
metadata_curationstep6$GEOsample.title_coarsecurated <- mapply(classify_stage_coarse2025, metadata_curationstep6$GEOsample.title, metadata_curationstep6$runexpsample_devstage_curated_firstandsecond)
metadata_curationstep6$GEOsample.characteristics_coarsecurated <- mapply(classify_stage_coarse2025, metadata_curationstep6$GEOsample.characteristics, metadata_curationstep6$runexpsample_devstage_curated_firstandsecond)


metadata_curationstep6$run.title_coarsecurated <- mapply(classify_stage_coarse2025, metadata_curationstep6$run.title, metadata_curationstep6$runexpsample_devstage_curated_firstandsecond)
metadata_curationstep6$run.alias_coarsecurated <- mapply(classify_stage_coarse2025, metadata_curationstep6$run.alias, metadata_curationstep6$runexpsample_devstage_curated_firstandsecond)
metadata_curationstep6$run.attributes_coarsecurated <- mapply(classify_stage_coarse2025, metadata_curationstep6$run.attributes, metadata_curationstep6$runexpsample_devstage_curated_firstandsecond)
metadata_curationstep6$experiment.alias_coarsecurated <- mapply(classify_stage_coarse2025, metadata_curationstep6$experiment.alias, metadata_curationstep6$runexpsample_devstage_curated_firstandsecond)
metadata_curationstep6$experiment.library_name_coarsecurated <- mapply(classify_stage_coarse2025, metadata_curationstep6$experiment.library_name, metadata_curationstep6$runexpsample_devstage_curated_firstandsecond)
metadata_curationstep6$experiment.design_description_coarsecurated <- mapply(classify_stage_coarse2025, metadata_curationstep6$experiment.design_description, metadata_curationstep6$runexpsample_devstage_curated_firstandsecond)
metadata_curationstep6$experiment.library_construction_protocol_coarsecurated <- mapply(classify_stage_coarse2025, metadata_curationstep6$experiment.library_construction_protocol, metadata_curationstep6$runexpsample_devstage_curated_firstandsecond)


metadata_curationstep6$sample.description_coarsecurated <- mapply(classify_stage_coarse2025, metadata_curationstep6$sample.description, metadata_curationstep6$runexpsample_devstage_curated_firstandsecond)
metadata_curationstep6$sample.alias_coarsecurated <- mapply(classify_stage_coarse2025, metadata_curationstep6$sample.alias, metadata_curationstep6$runexpsample_devstage_curated_firstandsecond)
metadata_curationstep6$GEOsample.dataprocessing_coarsecurated <- mapply(classify_stage_coarse2025, metadata_curationstep6$GEOsample.dataprocessing, metadata_curationstep6$runexpsample_devstage_curated_firstandsecond)
metadata_curationstep6$GEOsample.source_coarsecurated <- mapply(classify_stage_coarse2025, metadata_curationstep6$GEOsample.source, metadata_curationstep6$runexpsample_devstage_curated_firstandsecond)
metadata_curationstep6$GEOsample.treatmentprotocol_coarsecurated <- mapply(classify_stage_coarse2025, metadata_curationstep6$GEOsample.treatmentprotocol, metadata_curationstep6$runexpsample_devstage_curated_firstandsecond)
metadata_curationstep6$GEOsample.extractprotocol_coarsecurated <- mapply(classify_stage_coarse2025, metadata_curationstep6$GEOsample.extractprotocol, metadata_curationstep6$runexpsample_devstage_curated_firstandsecond)
metadata_curationstep6$GEOsample.growthprotocol_coarsecurated <- mapply(classify_stage_coarse2025, metadata_curationstep6$GEOsample.growthprotocol, metadata_curationstep6$runexpsample_devstage_curated_firstandsecond)


In [35]:
## another round of uniting this time on _coarsecurated columns

metadata_curationstep6c <- metadata_curationstep6 %>% unite("runexpsample_devstage_coarsecurated0", ends_with("_coarsecurated"), sep = ",", remove = TRUE, na.rm = TRUE)


In [36]:
# Apply the cleaning function to the column
metadata_curationstep7 <- metadata_curationstep6c %>% mutate(runexpsample_devstage_coarsecurated = map_chr(runexpsample_devstage_coarsecurated0, cleanlabels1))
metadata_curationstep7 %>% group_by(runexpsample_devstage_coarsecurated) %>% summarize(n = n()) %>% arrange(desc(n)) %>% print(n=35)

## relocate some columns to the end
metadata_curationstep7 <- metadata_curationstep7 %>% relocate(runexpsample_devstage_curated_second, .after = last_col())
metadata_curationstep7 <- metadata_curationstep7 %>% relocate(runexpsample_devstage_curated_first0, .after = last_col())
metadata_curationstep7 <- metadata_curationstep7 %>% relocate(runexpsample_devstage_curated_second0, .after = last_col())
metadata_curationstep7 <- metadata_curationstep7 %>% relocate(runexpsample_devstage_coarsecurated0, .after = last_col())


# A tibble: 15 × 2
   runexpsample_devstage_coarsecurated     n
   <chr>                               <int>
 1 Embryo                              29716
 2 Larval                              19709
 3 Adult                               15684
 4 Undetermined                         7486
 5 Juvenile                             2710
 6 Embryo,Larval                        1082
 7 Adult,Larval                          266
 8 Adult,Embryo                          186
 9 Juvenile,Larval                       151
10 Adult,Juvenile                         89
11 Adult,Embryo,Larval                    88
12 Adult,Juvenile,Larval                  86
13 Embryo,Juvenile                        26
14 Embryo,Juvenile,Larval                 26
15 Adult,Embryo,Juvenile                  24


## Only now look at study-level columns

Also using the classify_stage_coarse2025 function

In [37]:
## coarse function on study columns
metadata_curationstep7$study.title_coarselastcurated <- mapply(classify_stage_coarse2025, metadata_curationstep7$study.title, metadata_curationstep7$runexpsample_devstage_curated_firstandsecond)
metadata_curationstep7$study.abstract_coarselastcurated <- mapply(classify_stage_coarse2025, metadata_curationstep7$study.abstract, metadata_curationstep7$runexpsample_devstage_curated_firstandsecond)
metadata_curationstep7$study.attributes_coarselastcurated <- mapply(classify_stage_coarse2025, metadata_curationstep7$study.attributes, metadata_curationstep7$runexpsample_devstage_curated_firstandsecond)


In [38]:
metadata_curationstep7c <- metadata_curationstep7 %>% unite("runexpsample_devstage_coarselastcurated0", ends_with("_coarselastcurated"), sep = ",", remove = TRUE, na.rm = TRUE)


In [39]:
# Apply the cleaning function to the column
metadata_curationstep8 <- metadata_curationstep7c %>% mutate(runexpsample_devstage_coarselastcurated = map_chr(runexpsample_devstage_coarselastcurated0, cleanlabels1))
metadata_curationstep8 %>% group_by(runexpsample_devstage_coarselastcurated) %>% summarize(n = n()) %>% arrange(desc(n)) %>% print(n=35)


# A tibble: 15 × 2
   runexpsample_devstage_coarselastcurated     n
   <chr>                                   <int>
 1 Embryo                                  29609
 2 Larval                                  21149
 3 Adult                                   16923
 4 Undetermined                             4493
 5 Juvenile                                 2716
 6 Embryo,Larval                            1189
 7 Adult,Embryo                              412
 8 Adult,Larval                              320
 9 Juvenile,Larval                           159
10 Adult,Juvenile                            101
11 Adult,Embryo,Larval                        88
12 Adult,Juvenile,Larval                      86
13 Embryo,Juvenile                            34
14 Embryo,Juvenile,Larval                     26
15 Adult,Embryo,Juvenile                      24


In [40]:
## now coalesce only after creating NAs from runexpsample_devstage_coarsecurated when Unclassified
metadata_curationstep8 <- metadata_curationstep8 %>% mutate(runexpsample_devstage_coarsecurated = na_if(runexpsample_devstage_coarsecurated, "Undetermined"))
metadata_curationstep8 <- metadata_curationstep8 %>% mutate(runexpsample_devstage_coarsecurated_alsostudy = coalesce(runexpsample_devstage_coarsecurated,runexpsample_devstage_coarselastcurated))


In [41]:
## add undetermined back in 
metadata_curationstep8$runexpsample_devstage_coarsecurated <- metadata_curationstep8$runexpsample_devstage_coarsecurated %>% replace_na("Undetermined")
metadata_curationstep8 %>% group_by(runexpsample_devstage_coarsecurated_alsostudy) %>% summarize(n = n()) %>% arrange(desc(n)) %>% print(n=35)


# A tibble: 15 × 2
   runexpsample_devstage_coarsecurated_alsostudy     n
   <chr>                                         <int>
 1 Embryo                                        30056
 2 Larval                                        21006
 3 Adult                                         16849
 4 Undetermined                                   4327
 5 Juvenile                                       2716
 6 Embryo,Larval                                  1127
 7 Adult,Embryo                                    412
 8 Adult,Larval                                    318
 9 Juvenile,Larval                                 159
10 Adult,Juvenile                                  101
11 Adult,Embryo,Larval                              88
12 Adult,Juvenile,Larval                            86
13 Embryo,Juvenile                                  34
14 Embryo,Juvenile,Larval                           26
15 Adult,Embryo,Juvenile                            24


In [42]:
## now make versions of main curation columns with multi-stage
#    runexpsample_devstage_curated_first        runexpsample_devstage_curated_firstandsecond        runexpsample_devstage_coarsecurated        runexpsample_devstage_coarsecurated_alsostudy     
## first remove some extra columns for last version

## only remove in separate step just for saving - will keep full column set for raw summary saving at very end...
# metadata_curationstep9 <- metadata_curationstep8 %>% select(-runexpsample_devstage_curated_second)
# metadata_curationstep9 <- metadata_curationstep9 %>% select(-runexpsample_devstage_curated_first0)
# metadata_curationstep9 <- metadata_curationstep9 %>% select(-runexpsample_devstage_curated_second0)
# metadata_curationstep9 <- metadata_curationstep9 %>% select(-runexpsample_devstage_coarsecurated0)
# metadata_curationstep9 <- metadata_curationstep9 %>% select(-runexpsample_devstage_coarselastcurated0)
# metadata_curationstep9 <- metadata_curationstep9 %>% select(-runexpsample_devstage_coarselastcurated)
metadata_curationstep9 <- metadata_curationstep8



In [43]:
metadata_curationstep9

run.accession,study.accession,bioproject,study.title,sample.accession,experiment.accession,earliest_date,experiment.title,experiment.attributes,sample.title,⋯,runexpsample_devstage_curated_first,runexpsample_devstage_curated_firstandsecond,runexpsample_devstage_coarsecurated,runexpsample_devstage_curated_second,runexpsample_devstage_curated_first0,runexpsample_devstage_curated_second0,runexpsample_devstage_coarsecurated0,runexpsample_devstage_coarselastcurated0,runexpsample_devstage_coarselastcurated,runexpsample_devstage_coarsecurated_alsostudy
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
ERR1706522,ERP016002,PRJEB14364,Transcriptome profiling of zebrafish actin prl3 1 transgenic embryos,ERS1226910,ERX1776756,2016-06-28,Illumina HiSeq 2000 paired end sequencing,NA,zmp ph260 B6,⋯,"Adult,Pharyngula,Segmentation","Adult,Pharyngula,Segmentation","Adult,Embryo","Adult,Pharyngula,Segmentation","Adult, Segmentation, Pharyngula,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined","Undetermined,Undetermined,Undetermined,Adult, Segmentation, Pharyngula,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined","Adult, Embryo,Adult, Embryo,Adult, Embryo,Adult, Embryo,Adult, Embryo,Adult, Embryo,Adult, Embryo,Adult, Embryo,Adult, Embryo,Adult, Embryo,Adult, Embryo,Adult, Embryo,Adult, Embryo,Adult, Embryo,Adult, Embryo,Adult, Embryo,Adult, Embryo,Adult, Embryo,Adult, Embryo,Adult, Embryo","Adult, Embryo,Adult, Embryo,Adult, Embryo","Adult,Embryo","Adult,Embryo"
SRR1873571,SRP056014,PRJNA277780,Identification of small non coding RNAs in zebrafish,SRS870223,SRX915251,2015-03-09,GSM1630515: fins; Danio rerio; miRNA Seq,GEO Accession:GSM1630515,fins,⋯,Adult,Adult,Adult,"Adult,Juvenile,Larval,Pharyngula","Adult,Adult,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Adult,Adult","Undetermined,Undetermined,Undetermined,Adult,Undetermined,Adult,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Pharyngula, Larval, Juvenile,Undetermined,Undetermined,Adult,Adult,Undetermined,Pharyngula, Larval, Juvenile,Undetermined","Adult,Adult,Adult,Adult,Adult,Adult,Adult,Adult,Adult,Adult,Adult,Adult,Adult,Adult,Adult,Adult,Adult,Adult,Adult,Adult","Adult,Adult,Adult",Adult,Adult
SRR1873570,SRP056014,PRJNA277780,Identification of small non coding RNAs in zebrafish,SRS870224,SRX915250,2015-03-09,GSM1630514: skin; Danio rerio; miRNA Seq,GEO Accession:GSM1630514,skin,⋯,Adult,Adult,Adult,"Adult,Juvenile,Larval,Pharyngula","Adult,Adult,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Adult,Adult","Undetermined,Undetermined,Undetermined,Adult,Undetermined,Adult,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Pharyngula, Larval, Juvenile,Undetermined,Undetermined,Adult,Adult,Undetermined,Pharyngula, Larval, Juvenile,Undetermined","Adult,Adult,Adult,Adult,Adult,Adult,Adult,Adult,Adult,Adult,Adult,Adult,Adult,Adult,Adult,Adult,Adult,Adult,Adult,Adult","Adult,Adult,Adult",Adult,Adult
SRR1873569,SRP056014,PRJNA277780,Identification of small non coding RNAs in zebrafish,SRS870225,SRX915249,2015-03-09,GSM1630513: heart; Danio rerio; miRNA Seq,GEO Accession:GSM1630513,heart,⋯,Adult,Adult,Adult,"Adult,Juvenile,Larval,Pharyngula","Adult,Adult,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Adult,Adult","Undetermined,Undetermined,Undetermined,Adult,Undetermined,Adult,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Pharyngula, Larval, Juvenile,Undetermined,Undetermined,Adult,Adult,Undetermined,Pharyngula, Larval, Ju

In [44]:
## replace any string with a comma with "multi-stage UNLESS IT HAS TUMOR
# metadata_curationstep9 <- metadata_curationstep9 %>%
#   mutate(runexpsample_devstage_curated_first2 = if_else(grepl(",", runexpsample_devstage_curated_first), "Multi-stage", runexpsample_devstage_curated_first))

# metadata_curationstep9 <- metadata_curationstep9 %>%
#   mutate(runexpsample_devstage_curated_firstandsecond2 = if_else(grepl(",", runexpsample_devstage_curated_firstandsecond), "Multi-stage", runexpsample_devstage_curated_firstandsecond))

# metadata_curationstep9 <- metadata_curationstep9 %>%
#   mutate(runexpsample_devstage_coarsecurated2 = if_else(grepl(",", runexpsample_devstage_coarsecurated), "Multi-stage", runexpsample_devstage_coarsecurated))

# metadata_curationstep9 <- metadata_curationstep9 %>%
#   mutate(runexpsample_devstage_coarsecurated_alsostudy2 = if_else(grepl(",", runexpsample_devstage_coarsecurated_alsostudy), "Multi-stage", runexpsample_devstage_coarsecurated_alsostudy))


## replace any string with a comma with "multi-stage", except if it contains "Cancer or Tumor"
metadata_curationstep9 <- metadata_curationstep9 %>%
  mutate(runexpsample_devstage_curated_first2 = case_when(
    grepl("Cancer or Tumor", runexpsample_devstage_curated_first, ignore.case = TRUE) ~ "Cancer or Tumor",
    grepl(",", runexpsample_devstage_curated_first) ~ "Multi-stage",
    TRUE ~ runexpsample_devstage_curated_first
  ))

metadata_curationstep9 <- metadata_curationstep9 %>%
  mutate(runexpsample_devstage_curated_firstandsecond2 = case_when(
    grepl("Cancer or Tumor", runexpsample_devstage_curated_firstandsecond, ignore.case = TRUE) ~ "Cancer or Tumor",
    grepl(",", runexpsample_devstage_curated_firstandsecond) ~ "Multi-stage",
    TRUE ~ runexpsample_devstage_curated_firstandsecond
  ))

metadata_curationstep9 <- metadata_curationstep9 %>%
  mutate(runexpsample_devstage_coarsecurated2 = case_when(
    grepl("Cancer or Tumor", runexpsample_devstage_coarsecurated, ignore.case = TRUE) ~ "Cancer or Tumor",
    grepl(",", runexpsample_devstage_coarsecurated) ~ "Multi-stage",
    TRUE ~ runexpsample_devstage_coarsecurated
  ))

metadata_curationstep9 <- metadata_curationstep9 %>%
  mutate(runexpsample_devstage_coarsecurated_alsostudy2 = case_when(
    grepl("Cancer or Tumor", runexpsample_devstage_coarsecurated_alsostudy, ignore.case = TRUE) ~ "Cancer or Tumor",
    grepl(",", runexpsample_devstage_coarsecurated_alsostudy) ~ "Multi-stage",
    TRUE ~ runexpsample_devstage_coarsecurated_alsostudy
  ))

In [45]:
## last column relocates - this version has 55 columns so moving these 5 to end for viewing in RStudio

metadata_curationstep9 <- metadata_curationstep9 %>% relocate(study.abstract, .after = last_col())
metadata_curationstep9 <- metadata_curationstep9 %>% relocate(study.attributes, .after = last_col())
metadata_curationstep9 <- metadata_curationstep9 %>% relocate(run.alias, .after = last_col())
metadata_curationstep9 <- metadata_curationstep9 %>% relocate(experiment.alias, .after = last_col())
metadata_curationstep9 <- metadata_curationstep9 %>% relocate(sample.alias, .after = last_col())


In [46]:
## final numbers
metadata_curationstep9 %>% group_by(runexpsample_devstage_curated_first2) %>% summarize(n = n()) %>% arrange(desc(n)) %>% print(n=35)
metadata_curationstep9 %>% group_by(runexpsample_devstage_curated_firstandsecond2) %>% summarize(n = n()) %>% arrange(desc(n)) %>% print(n=35)
metadata_curationstep9 %>% group_by(runexpsample_devstage_coarsecurated2) %>% summarize(n = n()) %>% arrange(desc(n)) %>% print(n=35)
metadata_curationstep9 %>% group_by(runexpsample_devstage_coarsecurated_alsostudy2) %>% summarize(n = n()) %>% arrange(desc(n)) %>% print(n=35)


# A tibble: 12 × 2
   runexpsample_devstage_curated_first2     n
   <chr>                                <int>
 1 Undetermined                         24676
 2 Larval                               15953
 3 Adult                                12571
 4 Pharyngula                            6708
 5 Hatching                              5179
 6 Gastrula                              4459
 7 Segmentation                          3471
 8 Blastula                              2163
 9 Juvenile                               984
10 Cleavage                               532
11 Multi-stage                            334
12 Zygote                                 299
# A tibble: 12 × 2
   runexpsample_devstage_curated_firstandsecond2     n
   <chr>                                         <int>
 1 Larval                                        19709
 2 Adult                                         15670
 3 Undetermined                                  10064
 4 Pharyngula                              

In [47]:
## save devstage curation
# ## moving removing of some columns here just for this saving step:
# metadata_curationstep9fewer <- metadata_curationstep9
# metadata_curationstep9fewer <- metadata_curationstep9fewer %>% select(-runexpsample_devstage_curated_second)
# metadata_curationstep9fewer <- metadata_curationstep9fewer %>% select(-runexpsample_devstage_curated_first0)
# metadata_curationstep9fewer <- metadata_curationstep9fewer %>% select(-runexpsample_devstage_curated_second0)
# metadata_curationstep9fewer <- metadata_curationstep9fewer %>% select(-runexpsample_devstage_coarsecurated0)
# metadata_curationstep9fewer <- metadata_curationstep9fewer %>% select(-runexpsample_devstage_coarselastcurated0)
# metadata_curationstep9fewer <- metadata_curationstep9fewer %>% select(-runexpsample_devstage_coarselastcurated)

# # update - saving the writing to the very end after renaming of columns...
# #write.table(metadata_curationstep9fewer, file = paste0("all_zf_dates_devstage_curated_withGEO.tsv"), sep = "\t", row.names = FALSE, quote = FALSE)

# #write.table(metadata_curationstep9, file = paste0("all_zf_dates_devstage_curated_withGEO.tsv"), sep = "\t", row.names = FALSE, quote = FALSE)


## Tissue curation



## First pass - using those curated columns with _tissue 


In [48]:
#metadata_curationstep9 <- read_tsv("all_zf_dates_devstage_curated_withGEO.tsv")             

In [49]:
metadata_curationstep9

run.accession,study.accession,bioproject,study.title,sample.accession,experiment.accession,earliest_date,experiment.title,experiment.attributes,sample.title,⋯,runexpsample_devstage_coarsecurated_alsostudy,runexpsample_devstage_curated_first2,runexpsample_devstage_curated_firstandsecond2,runexpsample_devstage_coarsecurated2,runexpsample_devstage_coarsecurated_alsostudy2,study.abstract,study.attributes,run.alias,experiment.alias,sample.alias
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
ERR1706522,ERP016002,PRJEB14364,Transcriptome profiling of zebrafish actin prl3 1 transgenic embryos,ERS1226910,ERX1776756,2016-06-28,Illumina HiSeq 2000 paired end sequencing,NA,zmp ph260 B6,⋯,"Adult,Embryo",Multi-stage,Multi-stage,Multi-stage,Multi-stage,RNA Seq data was generated from RNA of zebrafish actin:prl3.1 transgenic embryos for transcriptional profiling.,ArrayExpress:E ERAD 507,SC RUN 20417 1#16,SC EXP 20417 1#16,SAMEA4055800
SRR1873571,SRP056014,PRJNA277780,Identification of small non coding RNAs in zebrafish,SRS870223,SRX915251,2015-03-09,GSM1630515: fins; Danio rerio; miRNA Seq,GEO Accession:GSM1630515,fins,⋯,Adult,Adult,Adult,Adult,Adult,MicroRNAs miRNAs are a new class of small RNAs of approximately 22 nucleotides in length that control eukaryotic gene expression by fine tuning mRNA translation. They regulate a wide variety of biological processes namely developmental timing cell differentiation cell proliferation immune response and infection. For this reason their identification is essential to understand eukaryotic biology. Their small size low abundance and high instability complicated early identification; however cloning/Sanger sequencing and new generation genome sequencing approaches overcame most technical hurdles and are being used for rapid miRNA identification in many eukaryotes. We have applied 454 DNA pyrosequencing technology to miRNA discovery in zebrafish Danio rerio. For this a series of cDNA libraries were prepared from small non coding RNAs isolated at different embryonic time points and from fully developed organs. Each cDNA library was tagged with specific sequences and was sequenced using the Roche FLX genome sequencer. This approach retrieved 90% of the 192 miRNAs previously identified by cloning/Sanger sequencing and bioinformatics and 25 novel miRNAs were predicted. Overall design: Small RNA libraries were prepared from different zebrafish developmental stages namely 24 hpf 72 hpf 96 hpf 5 dpf dpf 45 dpf young adult and from adult brain eyes gills heart skin and fins.,NA,GSM1630515 r1,GSM1630515,GSM1630515
SRR1873570,SRP056014,PRJNA277780,Identification of small non coding RNAs in zebrafish,SRS870224,SRX915250,2015-03-09,GSM1630514: skin; Danio rerio; miRNA Seq,GEO Accession:GSM1630514,skin,⋯,Adult,Adult,Adult,Adult,Adult,MicroRNAs miRNAs are a new class of small RNAs of approximately 22 nucleotides in length that control eukaryotic gene expression by fine tuning mRNA translation. They regulate a wide variety of biological processes namely developmental timing cell differentiation cell proliferation immune response and infection. For this reason their identification is essential to understand eukaryotic biology. Their small size low abundance and high instability complicated early identification; however cloning/Sanger sequencing and new generation genome sequencing approaches overcame most technical hurdles and are being used for rapid miRNA identification in many eukaryotes. We have applied 454 DNA pyrosequencing technology to miRNA discovery in zebrafish Danio rerio. For this a series of cDNA libraries were prepared from small non coding RNAs isolated at different embryonic time points and from fully developed organs. Each cDNA library was tagged with specific sequences and was sequenced using the Roche FLX genome sequencer. This approach retrieved 90% of the 192 miRNAs previously identified by cloning/Sanger sequencing and b

In [50]:
metadata_curationstep10 <- metadata_curationstep9

In [51]:
############ now somewhat similar strategy for tissue
## run the classify_tissue2025first function for the 6 _tissue columns separately
## then the classify_tissue2025 'regular' function for the other 20
## then coarse tissue function
## lastly for 3 study columns run coarse function


metadata_curationstep10$sample.attributes_tissue_curated <- sapply(metadata_curationstep10$sample.attributes_tissue, classify_tissue2025first)
metadata_curationstep10$sample.title_tissue_curated <- sapply(metadata_curationstep10$sample.title_tissue, classify_tissue2025first)

metadata_curationstep10$experiment.attributes_tissue_curated <- sapply(metadata_curationstep10$experiment.attributes_tissue, classify_tissue2025first)
metadata_curationstep10$experiment.title_tissue_curated <- sapply(metadata_curationstep10$experiment.title_tissue, classify_tissue2025first)

metadata_curationstep10$GEOsample.title_tissue_curated <- sapply(metadata_curationstep10$GEOsample.title_tissue, classify_tissue2025first)
metadata_curationstep10$GEOsample.characteristics_tissue_curated <- sapply(metadata_curationstep10$GEOsample.characteristics_tissue, classify_tissue2025first)


In [52]:
## for first pass unite & clean


metadata_curationstep10c <- metadata_curationstep10 %>% unite("runexpsample_tissue_curated_first0", ends_with("_curated"), sep = ",", remove = TRUE, na.rm = TRUE)
metadata_curationstep10c <- metadata_curationstep10c %>% arrange(runexpsample_tissue_curated_first0)

# Apply the cleaning function to the column
metadata_curationstep11 <- metadata_curationstep10c %>% mutate(runexpsample_tissue_curated_first = map_chr(runexpsample_tissue_curated_first0, cleanlabels1))

metadata_curationstep11 %>% group_by(runexpsample_tissue_curated_first) %>% summarize(n = n()) %>% arrange(desc(n)) %>% print(n=35)


# A tibble: 79 × 2
   runexpsample_tissue_curated_first     n
   <chr>                             <int>
 1 Undetermined                      28976
 2 Whole Organism                    11202
 3 Embryo Imprecise                   6234
 4 Brain                              6122
 5 Trunk                              3480
 6 Heart                              3091
 7 Pancreas                           2507
 8 Spinal Cord                        2221
 9 Liver                              1776
10 Eye                                1435
11 Cancer or Tumor                    1032
12 Gonad                              1021
13 Kidney                              873
14 Gut                                 833
15 Tail                                802
16 Spleen                              748
17 Fin                                 627
18 Muscle                              574
19 Blood                               547
20 Head                                475
21 Head,Swim Bladder               

## Second pass - all main run/experiment/sample/GEOsample columns


In [53]:
getwd()

[1] "/hpc/projects/balla_group/sra_experiments/SRA_metadata/dec2025_75k_submitteradded"

In [54]:
metadata_curationstep11o <- metadata_curationstep11
metadata_curationstep11o

run.accession,study.accession,bioproject,study.title,sample.accession,experiment.accession,earliest_date,experiment.title,experiment.attributes,sample.title,⋯,runexpsample_devstage_curated_firstandsecond2,runexpsample_devstage_coarsecurated2,runexpsample_devstage_coarsecurated_alsostudy2,study.abstract,study.attributes,run.alias,experiment.alias,sample.alias,runexpsample_tissue_curated_first0,runexpsample_tissue_curated_first
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<chr>,<chr>,<chr>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
SRR9735192,SRP216059,PRJNA556239,Adipocyte serine uptake curbs ROS generation and visceral adiposity [Zebrafish],SRS5140748,SRX6492152,2019-07-23,GSM3964281: Adipose KO A13; Danio rerio; RNA Seq,GEO Accession:GSM3964281,Adipose KO A13,⋯,Adult,Adult,Adult,Obesity and visceral adiposity in particular increases the risk of common metabolic diseases including type 2 diabetes cardiovascular disease and several forms of cancer. However the molecular mechanisms responsible for regional fat storage remain poorly characterized preventing therapeutic innovation. We here applied a systematic genome wide screen and translational approach and discovered a novel role for the adipocyte expressed neutral amino acid transporter SLC7A10/ASC 1 in the regulation of visceral adiposity. Among 65 genes showing both adipose depot dependent and fat loss dependent expression 27 genes further showed significant correlations with waist to hip WHR ratio adjusted for BMI. Among these ASC 1 was expressed at the highest level in isolated visceral adipocytes. Further we found decreased ASC 1 mRNA in visceral and not subcutaneous adipose tissue in carriers of the KLF14 type 2 diabetes risk allele compared to the protective allele. By profiling amino acid fluxes during adipocyte differentiation in vitro we found that ASC 1 inhibition by a selective inhibitor decreased adipocyte uptake particularly of serine in mature adipocytes. Interestingly radiometric amino acid uptake assays showed ASC 1 dependent uptake of the serine D enantiomere. Using primary human and murine adipocyte models we uncovered marked effects of inhibiting ASC 1 on mitochondrial respiratory capacity within hours and lipid accumulation within days. Finally Asc 1 knockout KO zebrafish had increased body weight and adipocyte enlargement upon eight week overfeeding compared to wild type WT fish. RNA sequencing data from zebrafish adipose tissue showed up regulation of genes involved in fatty acid and lipid metabolism in the ASC 1 KOs consistent with the increased lipid accumulation in the inhibitor treated cell models. Additionally duox an enzyme involved in ROS generation showed higher expression in the KOs compared to the WTs. Importantly we confirmed increased reactive oxygen species ROS generation within minutes and within hours when inhibiting ASC 1 in our in vitro cell models. Our study points to increased ROS generation and reduced mitochondrial respiratory capacity as central early mechanisms in development of visceral adiposity and a role for adipocyte D serine transport via ASC 1 in these processes. Enhancing ASC 1 expression and/or activity in adipocytes likely through primary effects on one carbon metabolism and redox balance is a promising therapeutic strategy for reducing visceral adiposity and related diseases. Overall design: Total RNA Seq of Zebrafish adipose tissue were sequenced on the Illumina HiSeq4000 platform,parent bioproject:PRJNA557596,GSM3964281 r1,GSM3964281,GSM3964281,"Adipose Tissue,Undetermined,Undetermined,Undetermined,Undetermined,Adipose Tissue",Adipose Tissue
SRR9735191,SRP216059,PRJNA556239,Adipocyte serine uptake curbs ROS generation and visceral adiposity [Zebrafish],SRS5140747,SRX6492151,2019-07-23,GSM3964280: Adipose KO A12; Danio rerio; RNA Seq,GEO Accession:GSM3964280,Adipose KO A12,⋯,Adult,Adult,Adult,Obesity and visceral adiposity in particular increases the risk of common metabolic diseases including type 2 di

In [55]:
## then all 20 for main tissue curation...classify_tissue2025
#metadata_curationstep11 <- metadata_curationstep11o

## changing column end to _tcurated

metadata_curationstep11$experiment.title_tcurated <- sapply(metadata_curationstep11$experiment.title, classify_tissue2025)
metadata_curationstep11$experiment.attributes_tcurated <- sapply(metadata_curationstep11$experiment.attributes, classify_tissue2025)
metadata_curationstep11$sample.title_tcurated <- sapply(metadata_curationstep11$sample.title, classify_tissue2025)
metadata_curationstep11$sample.attributes_tcurated <- sapply(metadata_curationstep11$sample.attributes, classify_tissue2025)
metadata_curationstep11$GEOsample.title_tcurated <- sapply(metadata_curationstep11$GEOsample.title, classify_tissue2025)
metadata_curationstep11$GEOsample.characteristics_tcurated <- sapply(metadata_curationstep11$GEOsample.characteristics, classify_tissue2025)

metadata_curationstep11$run.title_tcurated <- sapply(metadata_curationstep11$run.title, classify_tissue2025)
metadata_curationstep11$run.alias_tcurated <- sapply(metadata_curationstep11$run.alias, classify_tissue2025)
metadata_curationstep11$run.attributes_tcurated <- sapply(metadata_curationstep11$run.attributes, classify_tissue2025)
metadata_curationstep11$experiment.alias_tcurated <- sapply(metadata_curationstep11$experiment.alias, classify_tissue2025)
metadata_curationstep11$experiment.library_name_tcurated <- sapply(metadata_curationstep11$experiment.library_name, classify_tissue2025)
metadata_curationstep11$experiment.design_description_tcurated <- sapply(metadata_curationstep11$experiment.design_description, classify_tissue2025)
metadata_curationstep11$experiment.library_construction_protocol_tcurated <- sapply(metadata_curationstep11$experiment.library_construction_protocol, classify_tissue2025)

metadata_curationstep11$sample.description_tcurated <- sapply(metadata_curationstep11$sample.description, classify_tissue2025)
metadata_curationstep11$sample.alias_tcurated <- sapply(metadata_curationstep11$sample.alias, classify_tissue2025)
metadata_curationstep11$GEOsample.dataprocessing_tcurated <- sapply(metadata_curationstep11$GEOsample.dataprocessing, classify_tissue2025)
metadata_curationstep11$GEOsample.source_tcurated <- sapply(metadata_curationstep11$GEOsample.source, classify_tissue2025)
metadata_curationstep11$GEOsample.treatmentprotocol_tcurated <- sapply(metadata_curationstep11$GEOsample.treatmentprotocol, classify_tissue2025)
metadata_curationstep11$GEOsample.extractprotocol_tcurated <- sapply(metadata_curationstep11$GEOsample.extractprotocol, classify_tissue2025)
metadata_curationstep11$GEOsample.growthprotocol_tcurated <- sapply(metadata_curationstep11$GEOsample.growthprotocol, classify_tissue2025)



In [56]:
## unite and clean

## another round of uniting

metadata_curationstep11c <- metadata_curationstep11 %>% unite("runexpsample_tissue_curated_second0", ends_with("_tcurated"), sep = ",", remove = TRUE, na.rm = TRUE)
#metadata_curationstep4c <- metadata_curationstep4c %>% arrange(runexpsample_tissue_curated_second0)

# Apply the cleaning function to the column
metadata_curationstep12 <- metadata_curationstep11c %>% mutate(runexpsample_tissue_curated_second = map_chr(runexpsample_tissue_curated_second0, cleanlabels1))

## now coalesce only after creating NAs from curated_first when Unclassified
metadata_curationstep12 <- metadata_curationstep12 %>% mutate(runexpsample_tissue_curated_first = na_if(runexpsample_tissue_curated_first, "Undetermined"))
metadata_curationstep12 <- metadata_curationstep12 %>% mutate(runexpsample_tissue_curated_firstandsecond = coalesce(runexpsample_tissue_curated_first,runexpsample_tissue_curated_second))


metadata_curationstep12 <- metadata_curationstep12 %>% relocate(runexpsample_tissue_curated_first0, .after = last_col())
metadata_curationstep12 <- metadata_curationstep12 %>% relocate(runexpsample_tissue_curated_second0, .after = last_col())

## add undetermined back in 
metadata_curationstep12$runexpsample_tissue_curated_first <- metadata_curationstep12$runexpsample_tissue_curated_first %>% replace_na("Undetermined")

metadata_curationstep12 %>% group_by(runexpsample_tissue_curated_firstandsecond) %>% summarize(n = n()) %>% arrange(desc(n)) %>% print(n=35)


# A tibble: 165 × 2
   runexpsample_tissue_curated_firstandsecond               n
   <chr>                                                <int>
 1 Whole Organism                                       24803
 2 Embryo Imprecise                                      8312
 3 Brain                                                 6454
 4 Undetermined                                          4325
 5 Trunk                                                 3555
 6 Heart                                                 3173
 7 Pancreas                                              2522
 8 Spinal Cord                                           2275
 9 Blood,Fin,Gonad,Kidney                                2015
10 Liver                                                 1826
11 Eye                                                   1527
12 Fin                                                   1280
13 Gonad                                                 1126
14 Cancer or Tumor                                

### Code to convert to multi-tissue also embryo imprecise and also whole organism taking priority (and not multi-tissue)


In [57]:
metadata_curationstep12b <- metadata_curationstep12

## first two replace firstandsecond column!!
## let's first replicate firstandsecond
metadata_curationstep12b$runexpsample_tissue_curated_firstandsecond2 <- metadata_curationstep12b$runexpsample_tissue_curated_firstandsecond


In [58]:
metadata_curationstep12b <- metadata_curationstep12b %>%
  mutate(runexpsample_tissue_curated_firstandsecond2 = case_when(
    grepl("Embryo Imprecise", runexpsample_tissue_curated_firstandsecond) & grepl("Whole Organism", runexpsample_tissue_curated_firstandsecond) ~ "Whole Organism",
    grepl("Embryo Imprecise", runexpsample_tissue_curated_firstandsecond) ~ "Embryo Imprecise",
    grepl("Whole Organism", runexpsample_tissue_curated_firstandsecond) ~ "Whole Organism",
    TRUE ~ runexpsample_tissue_curated_firstandsecond
  ))

## now this command will replace all other firstandsecond2
metadata_curationstep12b <- metadata_curationstep12b %>%
  mutate(runexpsample_tissue_curated_firstandsecond2 = if_else(grepl(",", runexpsample_tissue_curated_firstandsecond2), "Multi-tissue", runexpsample_tissue_curated_firstandsecond2))

## now summarize
metadata_curationstep12b %>% group_by(runexpsample_tissue_curated_firstandsecond2) %>% summarize(n = n()) %>% arrange(desc(n)) %>% print(n=45)


# A tibble: 41 × 2
   runexpsample_tissue_curated_firstandsecond2     n
   <chr>                                       <int>
 1 Whole Organism                              25682
 2 Embryo Imprecise                             8973
 3 Brain                                        6454
 4 Multi-tissue                                 5171
 5 Undetermined                                 4325
 6 Trunk                                        3555
 7 Heart                                        3173
 8 Pancreas                                     2522
 9 Spinal Cord                                  2275
10 Liver                                        1826
11 Eye                                          1527
12 Fin                                          1280
13 Gonad                                        1126
14 Cancer or Tumor                              1116
15 Kidney                                       1056
16 Spleen                                        948
17 Gut                     

In [59]:

## repeat for runexpsample_tissue_curated_first
metadata_curationstep12c <- metadata_curationstep12b
metadata_curationstep12c$runexpsample_tissue_curated_first2 <- metadata_curationstep12c$runexpsample_tissue_curated_first

metadata_curationstep12c <- metadata_curationstep12c %>%
  mutate(runexpsample_tissue_curated_first2 = case_when(
    grepl("Embryo Imprecise", runexpsample_tissue_curated_first) & grepl("Whole Organism", runexpsample_tissue_curated_first) ~ "Whole Organism",
    grepl("Embryo Imprecise", runexpsample_tissue_curated_first) ~ "Embryo Imprecise",
    grepl("Whole Organism", runexpsample_tissue_curated_first) ~ "Whole Organism",
    TRUE ~ runexpsample_tissue_curated_first
  ))

## now this command will replace all other first2
metadata_curationstep12c <- metadata_curationstep12c %>%
  mutate(runexpsample_tissue_curated_first2 = if_else(grepl(",", runexpsample_tissue_curated_first2), "Multi-tissue", runexpsample_tissue_curated_first2))

## now summarize
metadata_curationstep12c %>% group_by(runexpsample_tissue_curated_first2) %>% summarize(n = n()) %>% arrange(desc(n)) %>% print(n=45)


# A tibble: 40 × 2
   runexpsample_tissue_curated_first2     n
   <chr>                              <int>
 1 Undetermined                       28976
 2 Whole Organism                     11202
 3 Embryo Imprecise                    6240
 4 Brain                               6122
 5 Trunk                               3480
 6 Heart                               3091
 7 Pancreas                            2507
 8 Spinal Cord                         2221
 9 Liver                               1776
10 Eye                                 1435
11 Cancer or Tumor                     1032
12 Gonad                               1021
13 Multi-tissue                         967
14 Kidney                               873
15 Gut                                  833
16 Tail                                 802
17 Spleen                               748
18 Fin                                  627
19 Muscle                               574
20 Blood                                547
21 Head      

## Coarse functions - this time just on curated _curated_firstandsecond

But with different tissue coarse columns to check for embryo imprecise

In [60]:
## coarse function 
## devstage_full in this case is runexpsample_devstage_coarsecurated2
metadata_curationstep13 <- metadata_curationstep12c
#metadata_curationstep13 <- metadata_curationstep12b
metadata_curationstep13$runexpsample_tissue_coarsecurated <- mapply(classify_tissue_coarse2025, metadata_curationstep13$runexpsample_tissue_curated_firstandsecond2, metadata_curationstep13$runexpsample_devstage_coarsecurated2)


In [61]:
### then repeat with study included runexpsample_devstage_coarsecurated_alsostudy2

metadata_curationstep13$runexpsample_tissue_coarsecurated_alsostudy <- mapply(classify_tissue_coarse2025, metadata_curationstep13$runexpsample_tissue_curated_firstandsecond2, metadata_curationstep13$runexpsample_devstage_coarsecurated_alsostudy2)



In [62]:

## final summaries - might not need to remove commas now!


metadata_curationstep13 %>% group_by(runexpsample_tissue_coarsecurated) %>% summarize(n = n()) %>% arrange(desc(n)) %>% print(n=45)

metadata_curationstep13 %>% group_by(runexpsample_tissue_coarsecurated_alsostudy) %>% summarize(n = n()) %>% arrange(desc(n)) %>% print(n=45)


# A tibble: 21 × 2
   runexpsample_tissue_coarsecurated     n
   <chr>                             <int>
 1 All anatomical structures         34655
 2 Nervous System                     9567
 3 Multi-system                       6009
 4 Surface Structure                  5350
 5 Cardiovascular System              3611
 6 Endocrine System                   2612
 7 Embryo Imprecise                   2455
 8 Undetermined                       1870
 9 Liver and Biliary System           1826
10 Sensory System                     1745
11 Hematopoietic System               1666
12 Reproductive System                1402
13 Cancer or Tumor                    1116
14 Renal System                       1105
15 Digestive System                    899
16 Muscular System                     605
17 Cell Line                           387
18 Respiratory System                  261
19 Skeletal Element                    119
20 Adipose Tissue                       57
21 Swim Bladder                    

In [63]:
colnames(metadata_curationstep13)

[1] "run.accession"                                 
 [2] "study.accession"                               
 [3] "bioproject"                                    
 [4] "study.title"                                   
 [5] "sample.accession"                              
 [6] "experiment.accession"                          
 [7] "earliest_date"                                 
 [8] "experiment.title"                              
 [9] "experiment.attributes"                         
[10] "sample.title"                                  
[11] "sample.attributes"                             
[12] "GEOsample.title"                               
[13] "GEOsample.characteristics"                     
[14] "experiment.title_devstage"                     
[15] "experiment.title_age"                          
[16] "experiment.title_tissue"                       
[17] "experiment.attributes_devstage"                
[18] "experiment.attributes_age"                     
[19] "experiment.attributes_tissue"                  
[20] "sample.title_devstage"                         
[21] "sample.title_age"                              
[22] "sample.title_tissue"                           
[23] "sample.attributes_devstage"                    
[24] "sample.attributes_age"                         
[25] "sample.attributes_tissue"                      
[26] "GEOsample.title_devstage"                      
[27] "GEOsample.title_age"                           
[28] "GEOsample.title_tissue"                        
[29] "GEOsample.characteristics_devstage"            
[30] "GEOsample.characteristics_age"                 
[31] "GEOsample.characteristics_tissue"              
[32] "run.title"                                     
[33] "run.attributes"                                
[34] "experiment.library_name"                       
[35] "experiment.design_description"                 
[36] "experiment.library_construction_protocol"      
[37] "sample.description"                            
[38] "GEOsample.dataprocessing"                      
[39] "GEOsample.source"                              
[40] "GEOsample.treatmentprotocol"                   
[41] "GEOsample.extractprotocol"                     
[42] "GEOsample.growthprotocol"                      
[43] "runexpsample_devstage_curated_first"           
[44] "runexpsample_devstage_curated_firstandsecond"  
[45] "runexpsample_devstage_coarsecurated"           
[46] "runexpsample_devstage_curated_second"          
[47] "runexpsample_devstage_curated_first0"          
[48] "runexpsample_devstage_curated_second0"         
[49] "runexpsample_devstage_coarsecurated0"          
[50] "runexpsample_devstage_coarselastcurated0"      
[51] "runexpsample_devstage_coarselastcurated"       
[52] "runexpsample_devstage_coarsecurated_alsostudy" 
[53] "runexpsample_devstage_curated_first2"          
[54] "runexpsample_devstage_curated_firstandsecond2" 
[55] "runexpsample_devstage_coarsecurated2"          
[56] "runexpsample_devstage_coarsecurated_alsostudy2"
[57] "study.abstract"                                
[58] "study.attributes"                              
[59] "run.alias"                                     
[60] "experiment.alias"                              
[61] "sample.alias"                                  
[62] "runexpsample_tissue_curated_first"             
[63] "runexpsample_tissue_curated_second"            
[64] "runexpsample_tissue_curated_firstandsecond"    
[65] "runexpsample_tissue_curated_first0"            
[66] "runexpsample_tissue_curated_second0"           
[67] "runexpsample_tissue_curated_firstandsecond2"   
[68] "runexpsample_tissue_curated_first2"            
[69] "runexpsample_tissue_coarsecurated"             
[70] "runexpsample_tissue_coarsecurated_alsostudy"

In [64]:
## rename a bunch of columns, save fullcolumns version, the remove many for final curation version
metadata_curationstep13 <- metadata_curationstep13 %>% rename(devstage_curation_runexperimentsampleGEOsample_sixcuratedcolumns = runexpsample_devstage_curated_first)
metadata_curationstep13 <- metadata_curationstep13 %>% rename(devstage_curation_runexperimentsampleGEOsample_sixcuratedcolumns_and_twentycolumns = runexpsample_devstage_curated_firstandsecond)
metadata_curationstep13 <- metadata_curationstep13 %>% rename(devstage_curation_runexperimentsampleGEOsample_coarse = runexpsample_devstage_coarsecurated)
metadata_curationstep13 <- metadata_curationstep13 %>% rename(devstage_curation_runexperimentsampleGEOsample_twentycolumns = runexpsample_devstage_curated_second)
metadata_curationstep13 <- metadata_curationstep13 %>% rename(devstage_curation_runexperimentsampleGEOsample_sixcuratedcolumns0 = runexpsample_devstage_curated_first0)
metadata_curationstep13 <- metadata_curationstep13 %>% rename(devstage_curation_runexperimentsampleGEOsample_twentycolumns0 = runexpsample_devstage_curated_second0)
metadata_curationstep13 <- metadata_curationstep13 %>% rename(devstage_curation_runexperimentsampleGEOsample_coarse0 = runexpsample_devstage_coarsecurated0)
metadata_curationstep13 <- metadata_curationstep13 %>% rename(devstage_curation_runexperimentsampleGEOsample_coarselast0 = runexpsample_devstage_coarselastcurated0)
metadata_curationstep13 <- metadata_curationstep13 %>% rename(devstage_curation_runexperimentsampleGEOsample_coarselast = runexpsample_devstage_coarselastcurated)
metadata_curationstep13 <- metadata_curationstep13 %>% rename(devstage_curation_runexperimentsampleGEOsample_alsostudy_coarse = runexpsample_devstage_coarsecurated_alsostudy)
metadata_curationstep13 <- metadata_curationstep13 %>% rename(devstage_curation_runexperimentsampleGEOsample_withmultistage_sixcuratedcolumns = runexpsample_devstage_curated_first2)
metadata_curationstep13 <- metadata_curationstep13 %>% rename(devstage_curation_runexperimentsampleGEOsample_withmultistage_sixcuratedcolumns_and_twentycolumns = runexpsample_devstage_curated_firstandsecond2)
metadata_curationstep13 <- metadata_curationstep13 %>% rename(devstage_curation_runexperimentsampleGEOsample_withmultistage_coarse = runexpsample_devstage_coarsecurated2)
metadata_curationstep13 <- metadata_curationstep13 %>% rename(devstage_curation_runexperimentsampleGEOsample_alsostudy_withmultistage_coarse = runexpsample_devstage_coarsecurated_alsostudy2)

metadata_curationstep13 <- metadata_curationstep13 %>% rename(tissue_curation_runexperimentsampleGEOsample_sixcuratedcolumns = runexpsample_tissue_curated_first)
metadata_curationstep13 <- metadata_curationstep13 %>% rename(tissue_curation_runexperimentsampleGEOsample_twentycolumns = runexpsample_tissue_curated_second)
metadata_curationstep13 <- metadata_curationstep13 %>% rename(tissue_curation_runexperimentsampleGEOsample_sixcuratedcolumns_and_twentycolumns = runexpsample_tissue_curated_firstandsecond)
metadata_curationstep13 <- metadata_curationstep13 %>% rename(tissue_curation_runexperimentsampleGEOsample_sixcuratedcolumns0 = runexpsample_tissue_curated_first0)
metadata_curationstep13 <- metadata_curationstep13 %>% rename(tissue_curation_runexperimentsampleGEOsample_twentycolumns0 = runexpsample_tissue_curated_second0)
metadata_curationstep13 <- metadata_curationstep13 %>% rename(tissue_curation_runexperimentsampleGEOsample_withmultistage_sixcuratedcolumns_and_twentycolumns = runexpsample_tissue_curated_firstandsecond2)
metadata_curationstep13 <- metadata_curationstep13 %>% rename(tissue_curation_runexperimentsampleGEOsample_withmultistage_sixcuratedcolumns = runexpsample_tissue_curated_first2)
metadata_curationstep13 <- metadata_curationstep13 %>% rename(tissue_curation_runexperimentsampleGEOsample_withmultistage_coarse = runexpsample_tissue_coarsecurated)
metadata_curationstep13 <- metadata_curationstep13 %>% rename(tissue_curation_runexperimentsampleGEOsample_alsostudy_withmultistage_coarse = runexpsample_tissue_coarsecurated_alsostudy)


In [65]:
## relocate( .after = )
metadata_curationstep13 <- metadata_curationstep13 %>% relocate(study.abstract, .after = last_col())
metadata_curationstep13 <- metadata_curationstep13 %>% relocate(study.attributes, .after = last_col())
metadata_curationstep13 <- metadata_curationstep13 %>% relocate(run.alias, .after = last_col())
metadata_curationstep13 <- metadata_curationstep13 %>% relocate(experiment.alias, .after = last_col())
metadata_curationstep13 <- metadata_curationstep13 %>% relocate(sample.alias, .after = last_col())
metadata_curationstep13 <- metadata_curationstep13 %>% relocate(devstage_curation_runexperimentsampleGEOsample_sixcuratedcolumns0, .after = last_col())
metadata_curationstep13 <- metadata_curationstep13 %>% relocate(devstage_curation_runexperimentsampleGEOsample_twentycolumns0, .after = last_col())
metadata_curationstep13 <- metadata_curationstep13 %>% relocate(devstage_curation_runexperimentsampleGEOsample_coarselast0, .after = last_col())
metadata_curationstep13 <- metadata_curationstep13 %>% relocate(devstage_curation_runexperimentsampleGEOsample_coarse0, .after = last_col())
metadata_curationstep13 <- metadata_curationstep13 %>% relocate(devstage_curation_runexperimentsampleGEOsample_twentycolumns, .after = last_col())
metadata_curationstep13 <- metadata_curationstep13 %>% relocate(devstage_curation_runexperimentsampleGEOsample_coarselast, .after = last_col())
metadata_curationstep13 <- metadata_curationstep13 %>% relocate(devstage_curation_runexperimentsampleGEOsample_coarse, .after = last_col())
metadata_curationstep13 <- metadata_curationstep13 %>% relocate(devstage_curation_runexperimentsampleGEOsample_alsostudy_coarse, .after = last_col())
metadata_curationstep13 <- metadata_curationstep13 %>% relocate(devstage_curation_runexperimentsampleGEOsample_sixcuratedcolumns, .after = last_col())
metadata_curationstep13 <- metadata_curationstep13 %>% relocate(devstage_curation_runexperimentsampleGEOsample_sixcuratedcolumns_and_twentycolumns, .after = last_col())
metadata_curationstep13 <- metadata_curationstep13 %>% relocate(devstage_curation_runexperimentsampleGEOsample_withmultistage_sixcuratedcolumns, .after = last_col())
metadata_curationstep13 <- metadata_curationstep13 %>% relocate(devstage_curation_runexperimentsampleGEOsample_withmultistage_coarse, .after = last_col())
metadata_curationstep13 <- metadata_curationstep13 %>% relocate(devstage_curation_runexperimentsampleGEOsample_withmultistage_sixcuratedcolumns_and_twentycolumns, .after = last_col())
metadata_curationstep13 <- metadata_curationstep13 %>% relocate(devstage_curation_runexperimentsampleGEOsample_alsostudy_withmultistage_coarse, .after = last_col())
metadata_curationstep13 <- metadata_curationstep13 %>% relocate(tissue_curation_runexperimentsampleGEOsample_sixcuratedcolumns0, .after = last_col())
metadata_curationstep13 <- metadata_curationstep13 %>% relocate(tissue_curation_runexperimentsampleGEOsample_twentycolumns0, .after = last_col())
metadata_curationstep13 <- metadata_curationstep13 %>% relocate(tissue_curation_runexperimentsampleGEOsample_sixcuratedcolumns, .after = last_col())
metadata_curationstep13 <- metadata_curationstep13 %>% relocate(tissue_curation_runexperimentsampleGEOsample_twentycolumns, .after = last_col())
metadata_curationstep13 <- metadata_curationstep13 %>% relocate(tissue_curation_runexperimentsampleGEOsample_sixcuratedcolumns_and_twentycolumns, .after = last_col())
metadata_curationstep13 <- metadata_curationstep13 %>% relocate(tissue_curation_runexperimentsampleGEOsample_withmultistage_sixcuratedcolumns, .after = last_col())
metadata_curationstep13 <- metadata_curationstep13 %>% relocate(tissue_curation_runexperimentsampleGEOsample_withmultistage_coarse, .after = last_col())
metadata_curationstep13 <- metadata_curationstep13 %>% relocate(tissue_curation_runexperimentsampleGEOsample_withmultistage_sixcuratedcolumns_and_twentycolumns, .after = last_col())
metadata_curationstep13 <- metadata_curationstep13 %>% relocate(tissue_curation_runexperimentsampleGEOsample_alsostudy_withmultistage_coarse, .after = last_col())


## Adding technology


In [66]:
## INSTRUCTIONS FOR RUNNING Y TECHNOLOGY CURATION SCRIPT
# ## copied into //hpc/projects/balla_group/sra_experiments/SRA_metadata/dec2025_75k_submitteradded/intermediate
# ## also have to run these
# ## try loading anaconda first??
# module load anaconda
# module load data.science

# # now activate single-cell-base
# #conda activate /hpc/user_apps/data.science/conda_envs/single-cell-base
# conda activate single-cell-base

# module load ugrep
# #./metadata-annotate.py combine all_zf_metadata2_run.tsv all_zf_metadata2_sample.tsv all_zf_metadata2_experiment.tsv all_zf_metadata2_study.tsv

# ## results in new file all_metadata_combined.csv

In [67]:
## then add technology
intermediatepath <- str_c(workingpath, "/intermediate")
intermediatepathmetadata_techcuration <- str_c(intermediatepath, "/all_metadata_combined.csv")
metadata_techcuration <- read_csv(intermediatepathmetadata_techcuration)
metadata_techcuration <- metadata_techcuration %>% select(run.accession,technology)
metadata_techcuration <- metadata_techcuration %>% rename(technology_curation = technology)

metadata_curationstep13b <- left_join(metadata_curationstep13,metadata_techcuration)


Rows: 77329 Columns: 38
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr  (10): run.accession, run.experiment, run.pool_member, run.read_lengths,...
dbl  (27): run.total_spots, bulk, droncseq, dropseq, fluidigm, indrops, mars...
dttm  (1): run.published_date

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Joining with `by = join_by(run.accession)`


In [68]:
# ## first saving a fullcolumns version then removing ~10 columns
# write.table(metadata_curationstep13b, file = paste0("all_zf_dates_devstage_tissue_metatech_curated_withGEO_rawcolumns.tsv"), sep = "\t", row.names = FALSE, quote = FALSE)


In [69]:

## lost pubmed ids! they are in study.PMIDs
#metadata_curationstep0 <- read_tsv("all_zf_datescurated_withGEO.tsv")

## why lost? this step!
## metadata_curationstep1 <- metadata_curationstep0 %>% select(run.accession:earliest_date,experiment.title,experiment.attributes,sample.title,sample.attributes,GEOsample.title,GEOsample.characteristics,run.title,run.alias,run.attributes,experiment.alias,experiment.library_name,experiment.design_description,experiment.library_construction_protocol,sample.description,sample.alias,GEOsample.dataprocessing,GEOsample.source,GEOsample.treatmentprotocol,GEOsample.extractprotocol,GEOsample.growthprotocol,study.abstract,study.attributes)
colnames(metadata_curationstep0)
colnames(metadata_curationstep13b)



[1] "run.accession"                           
 [2] "study.accession"                         
 [3] "bioproject"                              
 [4] "study.title"                             
 [5] "sample.accession"                        
 [6] "experiment.accession"                    
 [7] "earliest_date"                           
 [8] "run.title"                               
 [9] "run.alias"                               
[10] "run.attributes"                          
[11] "experiment.title"                        
[12] "experiment.alias"                        
[13] "experiment.library_name"                 
[14] "experiment.design_description"           
[15] "experiment.library_construction_protocol"
[16] "experiment.attributes"                   
[17] "sample.description"                      
[18] "sample.title"                            
[19] "sample.alias"                            
[20] "sample.attributes"                       
[21] "sample.centername"                       
[22] "submission.bioprojectsource"             
[23] "submission.srasource"                    
[24] "submission.accession"                    
[25] "run.filename"                            
[26] "run.semantic_name"                       
[27] "run.total_bases"                         
[28] "run.total_spots"                         
[29] "experiment.library_strategy"             
[30] "experiment.library_source"               
[31] "experiment.library_selection"            
[32] "experiment.library_layout"               
[33] "experiment.platform"                     
[34] "experiment.instrument_model"             
[35] "experiment.spot_descriptor"              
[36] "study.alias"                             
[37] "study.type"                              
[38] "study.abstract"                          
[39] "study.attributes"                        
[40] "study.PMIDs"                             
[41] "GEOsample.accession"                     
[42] "GEOsample.title"                         
[43] "GEOsample.dataprocessing"                
[44] "GEOsample.source"                        
[45] "GEOsample.treatmentprotocol"             
[46] "GEOsample.extractprotocol"               
[47] "GEOsample.growthprotocol"                
[48] "GEOsample.characteristics"               
[49] "GEO_studyID"                             
[50] "GEO_sampleID_withrun"                    
[51] "run.read_lengths"                        
[52] "run.base_counts"                         
[53] "run.r1_length"                           
[54] "run.r2_length"                           
[55] "run.r3_length"                           
[56] "run.r4_length"                           
[57] "run.Acount"                              
[58] "run.Ccount"                              
[59] "run.Gcount"                              
[60] "run.Tcount"                              
[61] "run.Ncount"                              
[62] "run.experiment"                          
[63] "run.pool_member"                         
[64] "experiment.study_ref"                    
[65] "run.enafirstpublic_date"                 
[66] "run.enalastupdate_date"                  
[67] "run.lastSRAFile_date"                    
[68] "run.published_date"                      
[69] "run.SRAFile_date"                        
[70] "sample.enafirstpublic_date"              
[71] "sample.enafirstpublic2_date"             
[72] "sample.enalastupdate_date"               
[73] "sample.enalastupdate2_date"              
[74] "sample.insdcfirstpublic_date"            
[75] "sample.insdclastupdate_date"             
[76] "study.enafirstpublic_date"               
[77] "study.enalastupdate_date"                
[78] "GEOsample.submission_date"               
[79] "GEOsample.release_date"                  
[80] "GEOsample.lastupdate_date"

[1] "run.accession"                                                                                    
 [2] "study.accession"                                                                                  
 [3] "bioproject"                                                                                       
 [4] "study.title"                                                                                      
 [5] "sample.accession"                                                                                 
 [6] "experiment.accession"                                                                             
 [7] "earliest_date"                                                                                    
 [8] "experiment.title"                                                                                 
 [9] "experiment.attributes"                                                                            
[10] "sample.title"                                                                                     
[11] "sample.attributes"                                                                                
[12] "GEOsample.title"                                                                                  
[13] "GEOsample.characteristics"                                                                        
[14] "experiment.title_devstage"                                                                        
[15] "experiment.title_age"                                                                             
[16] "experiment.title_tissue"                                                                          
[17] "experiment.attributes_devstage"                                                                   
[18] "experiment.attributes_age"                                                                        
[19] "experiment.attributes_tissue"                                                                     
[20] "sample.title_devstage"                                                                            
[21] "sample.title_age"                                                                                 
[22] "sample.title_tissue"                                                                              
[23] "sample.attributes_devstage"                                                                       
[24] "sample.attributes_age"                                                                            
[25] "sample.attributes_tissue"                                                                         
[26] "GEOsample.title_devstage"                                                                         
[27] "GEOsample.title_age"                                                                              
[28] "GEOsample.title_tissue"                                                                           
[29] "GEOsample.characteristics_devstage"                                                               
[30] "GEOsample.characteristics_age"                                                                    
[31] "GEOsample.characteristics_tissue"                                                                 
[32] "run.title"                                                                                        
[33] "run.attributes"                                                                                   
[34] "experiment.library_name"                                                                          
[35] "experiment.design_description"                                                                    
[36] "experiment.library_construction_protocol"                                                         
[37] "sample.description"                                                                               
[38] "GEOsample.dataprocessing"                                                                         
[39] "GEOsa

In [70]:
## bring them back - instead use code just below 
# metadata_pmids <- metadata_curationstep0 %>% select(run.accession,study.PMIDs)

# metadata_curationstep13fewer2 <- left_join(metadata_curationstep13fewer,metadata_pmids)
# metadata_curationstep13fewer2 <- metadata_curationstep13fewer2 %>% relocate(study.PMIDs, .after = technology_curation)


In [71]:
# ## saving easier-to-read version with 52 columns
# #write.table(metadata_curationstep13b, file = paste0("all_zf_dates_devstage_tissue_metatech_curated_withGEO_rawcolumns.tsv"), sep = "\t", row.names = FALSE, quote = FALSE)
# write.table(metadata_curationstep13fewer, file = paste0("all_zf_dates_devstage_tissue_metatech_curated_withGEO.tsv"), sep = "\t", row.names = FALSE, quote = FALSE)


In [72]:
# write.table(metadata_curationstep13fewer2, file = paste0("all_zf_dates_devstage_tissue_tech_curated_withGEO.tsv"), sep = "\t", row.names = FALSE, quote = FALSE)


In [73]:
## let's add many of these back, but not all, and re-order these ed_order
# 'sample.centername''submission.bioprojectsource''submission.srasource''submission.accession'
mt_toaddback <- c(
  "run.accession", "run.filename", "run.semantic_name", "run.total_bases", "run.total_spots",
  "run.read_lengths", "run.base_counts", "run.Acount", "run.Ccount", "run.Gcount", "run.Tcount", "run.Ncount",
  "run.r1_length", "run.r2_length", "run.r3_length", "run.r4_length", "run.experiment", "run.pool_member",
  "experiment.instrument_model", "experiment.library_layout", "experiment.library_selection",
  "experiment.library_source", "experiment.library_strategy", "experiment.platform", "experiment.spot_descriptor",
  "experiment.study_ref", "sample.centername", "GEOsample.accession", "GEO_studyID", "GEO_sampleID_withrun",
  "study.alias", "study.type", "study.PMIDs", "submission.bioprojectsource", "submission.srasource", "submission.accession"
)



In [74]:
metadata_toaddback <- metadata_curationstep0 %>%
  select(all_of(mt_toaddback))

In [75]:
metadata_curationstep14 <- left_join(metadata_toaddback,metadata_curationstep13b)

metadata_curationstep14 <- metadata_curationstep14 %>% rename(GEOsample.sampleID_withrun = GEO_sampleID_withrun) %>% rename(GEOsample.studyID = GEO_studyID)


Joining with `by = join_by(run.accession)`


In [76]:

## now we have 3 center/submitted columns
# sample.centername, submission.bioprojectsource, submission.srasource
# submission.bioprojectsource looks to be the best - except in the following case, when we should use submission.srasource

metadata_curationstep14 <- metadata_curationstep14 %>%
  mutate(submission.bioprojectsource = if_else(
    submission.bioprojectsource == "European Bioinformatics Institute",
    submission.srasource,
    submission.bioprojectsource
  ))

## also coalesce for handful of cases when submission.bioprojectsource is NA
metadata_curationstep14 <- metadata_curationstep14 %>% mutate(submission.bioprojectsource = coalesce(submission.bioprojectsource, submission.srasource))

## finally there are only 2 complete NAs - this is from the 2 runs without a known bioproject
## we used a dummy bioproject ID, but we can manually look it up and replace it:
#SRR065197
#SRR065196
## submission.bioprojectsource Justus-Liebig University Giessen
metadata_curationstep14 <- metadata_curationstep14 %>%
  mutate(submission.bioprojectsource = case_when(
    run.accession %in% c("SRR065196", "SRR065197") ~ "Justus-Liebig University Giessen",
    TRUE ~ submission.bioprojectsource
  ))

In [77]:
colnames(metadata_curationstep14)

[1] "run.accession"                                                                                    
  [2] "run.filename"                                                                                     
  [3] "run.semantic_name"                                                                                
  [4] "run.total_bases"                                                                                  
  [5] "run.total_spots"                                                                                  
  [6] "run.read_lengths"                                                                                 
  [7] "run.base_counts"                                                                                  
  [8] "run.Acount"                                                                                       
  [9] "run.Ccount"                                                                                       
 [10] "run.Gcount"                                                                                       
 [11] "run.Tcount"                                                                                       
 [12] "run.Ncount"                                                                                       
 [13] "run.r1_length"                                                                                    
 [14] "run.r2_length"                                                                                    
 [15] "run.r3_length"                                                                                    
 [16] "run.r4_length"                                                                                    
 [17] "run.experiment"                                                                                   
 [18] "run.pool_member"                                                                                  
 [19] "experiment.instrument_model"                                                                      
 [20] "experiment.library_layout"                                                                        
 [21] "experiment.library_selection"                                                                     
 [22] "experiment.library_source"                                                                        
 [23] "experiment.library_strategy"                                                                      
 [24] "experiment.platform"                                                                              
 [25] "experiment.spot_descriptor"                                                                       
 [26] "experiment.study_ref"                                                                             
 [27] "sample.centername"                                                                                
 [28] "GEOsample.accession"                                                                              
 [29] "GEOsample.studyID"                                                                                
 [30] "GEOsample.sampleID_withrun"                                                                       
 [31] "study.alias"                                                                                      
 [32] "study.type"                                                                                       
 [33] "study.PMIDs"                                                                                      
 [34] "submission.bioprojectsource"                                                                      
 [35] "submission.srasource"                                                                             
 [36] "submission.accession"                                                                             
 [37] "study.accession"                                                                                  
 [38] "bioproject"                                                              

In [78]:
#write.table(metadata_curationstep14, file = paste0("all_zf_dates_devstage_tissue_tech_curated_withGEO_morecolumns.tsv"), sep = "\t", row.names = FALSE, quote = FALSE)

## ADD YTTRIA'S CURATION OF PAIRED VS SINGLE

## getting paired vs single this way:
# from /hpc/projects/balla_group/sra_experiments/SRA_metadata
# ls /hpc/projects/balla_group/sra_experiments/versioned_zf_output/75k_unstable/host_mapping/unmapped_reads/Single > zf75k_singleruns.txt
# ls /hpc/projects/balla_group/sra_experiments/versioned_zf_output/75k_unstable/host_mapping/unmapped_reads/Paired > zf75k_pairedruns.txt
# Then making new column called seqdetective.library_layout_curation


metadata_curationstep_seqdetpairedvssingle <- read_tsv("zf_runs_pairedvssingle.tsv")
metadata_curationstep14b <- left_join(metadata_curationstep14,metadata_curationstep_seqdetpairedvssingle)
metadata_curationstep14b <- metadata_curationstep14b %>% relocate(seqdetective.library_layout_curation, .after = technology_curation)

metadata_curationstep14b

Rows: 73955 Columns: 2
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): run.accession, seqdetective.library_layout_curation

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Joining with `by = join_by(run.accession)`


run.accession,run.filename,run.semantic_name,run.total_bases,run.total_spots,run.read_lengths,run.base_counts,run.Acount,run.Ccount,run.Gcount,⋯,tissue_curation_runexperimentsampleGEOsample_twentycolumns0,tissue_curation_runexperimentsampleGEOsample_sixcuratedcolumns,tissue_curation_runexperimentsampleGEOsample_twentycolumns,tissue_curation_runexperimentsampleGEOsample_sixcuratedcolumns_and_twentycolumns,tissue_curation_runexperimentsampleGEOsample_withmultistage_sixcuratedcolumns,tissue_curation_runexperimentsampleGEOsample_withmultistage_coarse,tissue_curation_runexperimentsampleGEOsample_withmultistage_sixcuratedcolumns_and_twentycolumns,tissue_curation_runexperimentsampleGEOsample_alsostudy_withmultistage_coarse,technology_curation,seqdetective.library_layout_curation
<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
DRR314108,NA,NA,23076492885,76679376,0:150.51 1:150.44,A:6146969533;C:5373690527;G:5458576301;T:6095818756;N:1437768,6146969533,5373690527,5458576301,⋯,"Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined",Undetermined,Undetermined,Undetermined,Undetermined,Embryo Imprecise,Undetermined,Embryo Imprecise,unknown,PAIRED
DRR314107,NA,NA,26091623771,86694066,0:150.51 1:150.45,A:6955374552;C:6050013285;G:6169385096;T:6915281392;N:1569446,6955374552,6050013285,6169385096,⋯,"Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined",Undetermined,Undetermined,Undetermined,Undetermined,Embryo Imprecise,Undetermined,Embryo Imprecise,unknown,PAIRED
DRR314106,NA,NA,23833525756,79191795,0:150.51 1:150.44,A:6324521565;C:5556755459;G:5694025045;T:6256748423;N:1475264,6324521565,5556755459,5694025045,⋯,"Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined",Undetermined,Undetermined,Undetermined,Undetermined,Embryo Imprecise,Undetermined,Embryo Imprecise,unknown,PAIRED
DRR314105,NA,NA,27688386114,92009317,0:150.49 1:150.44,A:7681451080;C:6071566134;G:6242782106;T:7690830655;N:1756139,7681451080,6071566134,6242782106,⋯,"Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined",Undetermined,Undetermined,Undetermined,Undetermined,Embryo Imprecise,Undetermined,Embryo Imprecise,unknown,PAIRED
DRR314104,NA,NA,22970994572,76322352,0:150.52 1:150.46,A:6142535654;C:5310453740;G:5430124468;T:6086516676;N:1364034,6142535654,5310453740,5430124468,⋯,"Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined",Undetermined,Undetermined,Undetermined,Undetermined,Embryo Imprecise,Undetermined,Embryo Imprecise,unknown,PAIRED
DRR314103,NA,NA,23637901630,78541449,0:150.51 1:150.45,A:6359178134;C:5420730443;G:5530700546;T:6325864263;N:1428244,6359178134,5420730443,5530700546,⋯,"Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined,Undetermined",Undetermined,Undetermined,Undetermined,Un

In [79]:
## let's check column names and rename especially
# devstage_curation_runexperimentsampleGEOsample_withmultistage_sixcuratedcolumns_and_twentycolumns
#devstage_curation_runexperimentsampleGEOsample_alsostudy_withmultistage_coarse
#tissue_curation_runexperimentsampleGEOsample_withmultistage_sixcuratedcolumns_and_twentycolumns
#tissue_curation_runexperimentsampleGEOsample_alsostudy_withmultistage_coarse
colnames(metadata_curationstep14b)
## change to
## devstage_curation
## devstage_curation_coarse
## tissue_curation
## tissue_curation_coarse


metadata_curationstep14b <- metadata_curationstep14b %>% rename(devstage_curation = devstage_curation_runexperimentsampleGEOsample_withmultistage_sixcuratedcolumns_and_twentycolumns)
metadata_curationstep14b <- metadata_curationstep14b %>% rename(devstage_curation_coarse = devstage_curation_runexperimentsampleGEOsample_alsostudy_withmultistage_coarse)
metadata_curationstep14b <- metadata_curationstep14b %>% rename(tissue_curation = tissue_curation_runexperimentsampleGEOsample_withmultistage_sixcuratedcolumns_and_twentycolumns)
metadata_curationstep14b <- metadata_curationstep14b %>% rename(tissue_curation_coarse = tissue_curation_runexperimentsampleGEOsample_alsostudy_withmultistage_coarse)


[1] "run.accession"                                                                                    
  [2] "run.filename"                                                                                     
  [3] "run.semantic_name"                                                                                
  [4] "run.total_bases"                                                                                  
  [5] "run.total_spots"                                                                                  
  [6] "run.read_lengths"                                                                                 
  [7] "run.base_counts"                                                                                  
  [8] "run.Acount"                                                                                       
  [9] "run.Ccount"                                                                                       
 [10] "run.Gcount"                                                                                       
 [11] "run.Tcount"                                                                                       
 [12] "run.Ncount"                                                                                       
 [13] "run.r1_length"                                                                                    
 [14] "run.r2_length"                                                                                    
 [15] "run.r3_length"                                                                                    
 [16] "run.r4_length"                                                                                    
 [17] "run.experiment"                                                                                   
 [18] "run.pool_member"                                                                                  
 [19] "experiment.instrument_model"                                                                      
 [20] "experiment.library_layout"                                                                        
 [21] "experiment.library_selection"                                                                     
 [22] "experiment.library_source"                                                                        
 [23] "experiment.library_strategy"                                                                      
 [24] "experiment.platform"                                                                              
 [25] "experiment.spot_descriptor"                                                                       
 [26] "experiment.study_ref"                                                                             
 [27] "sample.centername"                                                                                
 [28] "GEOsample.accession"                                                                              
 [29] "GEOsample.studyID"                                                                                
 [30] "GEOsample.sampleID_withrun"                                                                       
 [31] "study.alias"                                                                                      
 [32] "study.type"                                                                                       
 [33] "study.PMIDs"                                                                                      
 [34] "submission.bioprojectsource"                                                                      
 [35] "submission.srasource"                                                                             
 [36] "submission.accession"                                                                             
 [37] "study.accession"                                                                                  
 [38] "bioproject"                                                              

In [94]:
## late December - add country!

metadata_curationstep_country <- read_tsv("all_zf_submissioncenters2_with_condensed_and_country_v4.tsv")
metadata_curationstep_country <- metadata_curationstep_country %>% select(bioproject,submission.bioprojectsource.country)





Rows: 77329 Columns: 5
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (5): submission.bioprojectsource, submission.accession, bioproject, subm...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [96]:
# 1) Check how many unique countries per bioproject
country_check <- metadata_curationstep_country %>%
  group_by(bioproject) %>%
  summarise(
    n_countries = n_distinct(submission.bioprojectsource.country),
    countries = paste(unique(submission.bioprojectsource.country), collapse = "|")
  )

# View bioprojects with multiple countries (if any)
country_check %>% filter(n_countries > 1)

bioproject,n_countries,countries
<chr>,<int>,<chr>


In [98]:
metadata_curationstep_country_collapsed <- metadata_curationstep_country %>%
  distinct(bioproject, submission.bioprojectsource.country)
head(metadata_curationstep_country_collapsed)

bioproject,submission.bioprojectsource.country
<chr>,<chr>
PRJNA382557,Australia
PRJNA382558,Australia
PRJNA382561,Australia
PRJNA505329,Australia
PRJNA529645,Australia
PRJEB42059,Australia


In [99]:
metadata_curationstep14c <- left_join(metadata_curationstep14b,metadata_curationstep_country_collapsed)
metadata_curationstep14c <- metadata_curationstep14c %>% relocate(submission.bioprojectsource.country, .after = submission.bioprojectsource)

ncol(metadata_curationstep14b)
ncol(metadata_curationstep14c)

Joining with `by = join_by(bioproject)`


[1] 107

[1] 108

In [100]:
colnames(metadata_curationstep14c)


[1] "run.accession"                                                                     
  [2] "run.filename"                                                                      
  [3] "run.semantic_name"                                                                 
  [4] "run.total_bases"                                                                   
  [5] "run.total_spots"                                                                   
  [6] "run.read_lengths"                                                                  
  [7] "run.base_counts"                                                                   
  [8] "run.Acount"                                                                        
  [9] "run.Ccount"                                                                        
 [10] "run.Gcount"                                                                        
 [11] "run.Tcount"                                                                        
 [12] "run.Ncount"                                                                        
 [13] "run.r1_length"                                                                     
 [14] "run.r2_length"                                                                     
 [15] "run.r3_length"                                                                     
 [16] "run.r4_length"                                                                     
 [17] "run.experiment"                                                                    
 [18] "run.pool_member"                                                                   
 [19] "experiment.instrument_model"                                                       
 [20] "experiment.library_layout"                                                         
 [21] "experiment.library_selection"                                                      
 [22] "experiment.library_source"                                                         
 [23] "experiment.library_strategy"                                                       
 [24] "experiment.platform"                                                               
 [25] "experiment.spot_descriptor"                                                        
 [26] "experiment.study_ref"                                                              
 [27] "sample.centername"                                                                 
 [28] "GEOsample.accession"                                                               
 [29] "GEOsample.studyID"                                                                 
 [30] "GEOsample.sampleID_withrun"                                                        
 [31] "study.alias"                                                                       
 [32] "study.type"                                                                        
 [33] "study.PMIDs"                                                                       
 [34] "submission.bioprojectsource"                                                       
 [35] "submission.bioprojectsource.country"                                               
 [36] "submission.srasource"                                                              
 [37] "submission.accession"                                                              
 [38] "study.accession"                                                                   
 [39] "bioproject"                                                                        
 [40] "study.title"                                                                       
 [41] "sample.accession"                                                                  
 [42] "experiment.accession"                                                              
 [43] "earliest_date"                                                                     
 [44] "experiment.title"                                                                 

In [117]:
## rename 14c to 14b to preserve code below
metadata_curationstep14b <- metadata_curationstep14c

In [118]:
## finally remove columns only now for fewercolumns version
metadata_curationstep14fewer <- metadata_curationstep14b %>% select(-devstage_curation_runexperimentsampleGEOsample_sixcuratedcolumns0)
metadata_curationstep14fewer <- metadata_curationstep14fewer %>% select(-devstage_curation_runexperimentsampleGEOsample_twentycolumns0)
metadata_curationstep14fewer <- metadata_curationstep14fewer %>% select(-devstage_curation_runexperimentsampleGEOsample_coarselast0)
metadata_curationstep14fewer <- metadata_curationstep14fewer %>% select(-devstage_curation_runexperimentsampleGEOsample_coarse0)
metadata_curationstep14fewer <- metadata_curationstep14fewer %>% select(-devstage_curation_runexperimentsampleGEOsample_twentycolumns)
metadata_curationstep14fewer <- metadata_curationstep14fewer %>% select(-devstage_curation_runexperimentsampleGEOsample_coarselast)
metadata_curationstep14fewer <- metadata_curationstep14fewer %>% select(-devstage_curation_runexperimentsampleGEOsample_coarse)
metadata_curationstep14fewer <- metadata_curationstep14fewer %>% select(-devstage_curation_runexperimentsampleGEOsample_alsostudy_coarse)
metadata_curationstep14fewer <- metadata_curationstep14fewer %>% select(-devstage_curation_runexperimentsampleGEOsample_sixcuratedcolumns)
metadata_curationstep14fewer <- metadata_curationstep14fewer %>% select(-devstage_curation_runexperimentsampleGEOsample_sixcuratedcolumns_and_twentycolumns)
metadata_curationstep14fewer <- metadata_curationstep14fewer %>% select(-devstage_curation_runexperimentsampleGEOsample_withmultistage_sixcuratedcolumns)
metadata_curationstep14fewer <- metadata_curationstep14fewer %>% select(-devstage_curation_runexperimentsampleGEOsample_withmultistage_coarse)

metadata_curationstep14fewer <- metadata_curationstep14fewer %>% select(-tissue_curation_runexperimentsampleGEOsample_sixcuratedcolumns0)
metadata_curationstep14fewer <- metadata_curationstep14fewer %>% select(-tissue_curation_runexperimentsampleGEOsample_twentycolumns0)
metadata_curationstep14fewer <- metadata_curationstep14fewer %>% select(-tissue_curation_runexperimentsampleGEOsample_sixcuratedcolumns)
metadata_curationstep14fewer <- metadata_curationstep14fewer %>% select(-tissue_curation_runexperimentsampleGEOsample_twentycolumns)
metadata_curationstep14fewer <- metadata_curationstep14fewer %>% select(-tissue_curation_runexperimentsampleGEOsample_sixcuratedcolumns_and_twentycolumns)
metadata_curationstep14fewer <- metadata_curationstep14fewer %>% select(-tissue_curation_runexperimentsampleGEOsample_withmultistage_sixcuratedcolumns)
metadata_curationstep14fewer <- metadata_curationstep14fewer %>% select(-tissue_curation_runexperimentsampleGEOsample_withmultistage_coarse)


# last move 2 study columns to 51,52 & technology after date
metadata_curationstep14fewer <- metadata_curationstep14fewer %>% relocate(technology_curation, .after = earliest_date)
metadata_curationstep14fewer <- metadata_curationstep14fewer %>% relocate(study.attributes, .after = last_col())
metadata_curationstep14fewer <- metadata_curationstep14fewer %>% relocate(study.abstract, .after = last_col())


In [119]:
head(metadata_curationstep14fewer)
colnames(metadata_curationstep14fewer)


run.accession,run.filename,run.semantic_name,run.total_bases,run.total_spots,run.read_lengths,run.base_counts,run.Acount,run.Ccount,run.Gcount,⋯,run.alias,experiment.alias,sample.alias,devstage_curation,devstage_curation_coarse,tissue_curation,tissue_curation_coarse,seqdetective.library_layout_curation,study.attributes,study.abstract
<chr>,<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,⋯,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
DRR314108,NA,NA,23076492885,76679376,0:150.51 1:150.44,A:6146969533;C:5373690527;G:5458576301;T:6095818756;N:1437768,6146969533,5373690527,5458576301,⋯,DRR314108,DRX303511,SAMD00399013,Hatching,Embryo,Undetermined,Embryo Imprecise,PAIRED,NA,To characterize the physiological function of Banp the expression profile of banp mutant and wild type sibling was obtained by ATAC sequencing and RNA sequencing. All data were obtained using embryo heads at 48 hpf.
DRR314107,NA,NA,26091623771,86694066,0:150.51 1:150.45,A:6955374552;C:6050013285;G:6169385096;T:6915281392;N:1569446,6955374552,6050013285,6169385096,⋯,DRR314107,DRX303510,SAMD00399012,Hatching,Embryo,Undetermined,Embryo Imprecise,PAIRED,NA,To characterize the physiological function of Banp the expression profile of banp mutant and wild type sibling was obtained by ATAC sequencing and RNA sequencing. All data were obtained using embryo heads at 48 hpf.
DRR314106,NA,NA,23833525756,79191795,0:150.51 1:150.44,A:6324521565;C:5556755459;G:5694025045;T:6256748423;N:1475264,6324521565,5556755459,5694025045,⋯,DRR314106,DRX303509,SAMD00399011,Hatching,Embryo,Undetermined,Embryo Imprecise,PAIRED,NA,To characterize the physiological function of Banp the expression profile of banp mutant and wild type sibling was obtained by ATAC sequencing and RNA sequencing. All data were obtained using embryo heads at 48 hpf.
DRR314105,NA,NA,27688386114,92009317,0:150.49 1:150.44,A:7681451080;C:6071566134;G:6242782106;T:7690830655;N:1756139,7681451080,6071566134,6242782106,⋯,DRR314105,DRX303508,SAMD00399010,Hatching,Embryo,Undetermined,Embryo Imprecise,PAIRED,NA,To characterize the physiological function of Banp the expression profile of banp mutant and wild type sibling was obtained by ATAC sequencing and RNA sequencing. All data were obtained using embryo heads at 48 hpf.
DRR314104,NA,NA,22970994572,76322352,0:150.52 1:150.46,A:6142535654;C:5310453740;G:5430124468;T:6086516676;N:1364034,6142535654,5310453740,5430124468,⋯,DRR314104,DRX303507,SAMD00399009,Hatching,Embryo,Undetermined,Embryo Imprecise,PAIRED,NA,To characterize the physiological function of Banp the expression profile of banp mutant and wild type sibling was obtained by ATAC sequencing and RNA sequencing. All data were obtained using embryo heads at 48 hpf.
DRR314103,NA,NA,23637901630,78541449,0:150.51 1:150.45,A:6359178134;C:5420730443;G:5530700546;T:6325864263;N:1428244,6359178134,5420730443,5530700546,⋯,DRR314103,DRX303506,SAMD00399008,Hatching,Embryo,Undetermined,Embryo Imprecise,PAIRED,NA,To characterize the physiological function of Banp the expression profile of banp mutant and wild type sibling was obtained by ATAC sequencing and RNA sequencing. All data were obtained using embryo heads at 48 hpf.


[1] "run.accession"                           
 [2] "run.filename"                            
 [3] "run.semantic_name"                       
 [4] "run.total_bases"                         
 [5] "run.total_spots"                         
 [6] "run.read_lengths"                        
 [7] "run.base_counts"                         
 [8] "run.Acount"                              
 [9] "run.Ccount"                              
[10] "run.Gcount"                              
[11] "run.Tcount"                              
[12] "run.Ncount"                              
[13] "run.r1_length"                           
[14] "run.r2_length"                           
[15] "run.r3_length"                           
[16] "run.r4_length"                           
[17] "run.experiment"                          
[18] "run.pool_member"                         
[19] "experiment.instrument_model"             
[20] "experiment.library_layout"               
[21] "experiment.library_selection"            
[22] "experiment.library_source"               
[23] "experiment.library_strategy"             
[24] "experiment.platform"                     
[25] "experiment.spot_descriptor"              
[26] "experiment.study_ref"                    
[27] "sample.centername"                       
[28] "GEOsample.accession"                     
[29] "GEOsample.studyID"                       
[30] "GEOsample.sampleID_withrun"              
[31] "study.alias"                             
[32] "study.type"                              
[33] "study.PMIDs"                             
[34] "submission.bioprojectsource"             
[35] "submission.bioprojectsource.country"     
[36] "submission.srasource"                    
[37] "submission.accession"                    
[38] "study.accession"                         
[39] "bioproject"                              
[40] "study.title"                             
[41] "sample.accession"                        
[42] "experiment.accession"                    
[43] "earliest_date"                           
[44] "technology_curation"                     
[45] "experiment.title"                        
[46] "experiment.attributes"                   
[47] "sample.title"                            
[48] "sample.attributes"                       
[49] "GEOsample.title"                         
[50] "GEOsample.characteristics"               
[51] "experiment.title_devstage"               
[52] "experiment.title_age"                    
[53] "experiment.title_tissue"                 
[54] "experiment.attributes_devstage"          
[55] "experiment.attributes_age"               
[56] "experiment.attributes_tissue"            
[57] "sample.title_devstage"                   
[58] "sample.title_age"                        
[59] "sample.title_tissue"                     
[60] "sample.attributes_devstage"              
[61] "sample.attributes_age"                   
[62] "sample.attributes_tissue"                
[63] "GEOsample.title_devstage"                
[64] "GEOsample.title_age"                     
[65] "GEOsample.title_tissue"                  
[66] "GEOsample.characteristics_devstage"      
[67] "GEOsample.characteristics_age"           
[68] "GEOsample.characteristics_tissue"        
[69] "run.title"                               
[70] "run.attributes"                          
[71] "experiment.library_name"                 
[72] "experiment.design_description"           
[73] "experiment.library_construction_protocol"
[74] "sample.description"                      
[75] "GEOsample.dataprocessing"                
[76] "GEOsample.source"                        
[77] "GEOsample.treatmentprotocol"             
[78] "GEOsample.extractprotocol"               
[79] "GEOsample.growthprotocol"                
[80] "run.alias"                               
[81] "experiment.alias"                        
[82] "sample.alias"                            
[83] "devstage_curation"                       
[84] "devstage_cu

In [120]:
## then use this order - adding submission.bioprojectsource.country last
desired_order14 <- c(
    "run.accession", "study.accession", "bioproject", "study.title", "sample.accession",
    "experiment.accession", "earliest_date", 
    "devstage_curation", "devstage_curation_coarse", "tissue_curation", "tissue_curation_coarse",
    "seqdetective.library_layout_curation", "technology_curation", "run.title", "run.attributes",
    "experiment.title", "experiment.alias", "experiment.library_name", "experiment.design_description",
    "experiment.library_construction_protocol", "experiment.attributes", "sample.description",
    "sample.title", "sample.alias", "sample.attributes", "submission.bioprojectsource", "submission.bioprojectsource.country",
    "GEOsample.title", "GEOsample.dataprocessing", "GEOsample.source", "GEOsample.treatmentprotocol",
    "GEOsample.extractprotocol", "GEOsample.growthprotocol", "GEOsample.characteristics", "GEOsample.accession", "run.filename",
    "run.semantic_name", "run.total_bases", "run.total_spots", "experiment.library_strategy",
    "experiment.library_source", "experiment.library_selection", "experiment.library_layout",
    "experiment.platform", "experiment.instrument_model", "experiment.spot_descriptor", "study.alias",
    "study.type", "study.abstract", "study.attributes", "study.PMIDs", 
    "run.alias", "run.read_lengths", "run.base_counts", "run.r1_length", "run.r2_length",
    "run.r3_length", "run.r4_length", "run.Acount", "run.Ccount", "run.Gcount", "run.Tcount", "run.Ncount",
    "run.experiment", "run.pool_member", "experiment.study_ref", "sample.centername", 
    "submission.accession", "submission.srasource"
)


## close but need to move GEOsample fields up
# desired_order14 <- c(
#     "run.accession", "study.accession", "bioproject", "study.title", "sample.accession",
#     "experiment.accession", "earliest_date", 
#     "devstage_curation", "devstage_curation_coarse", "tissue_curation", "tissue_curation_coarse",
#     "seqdetective.library_layout_curation", "technology_curation", "run.title", "run.attributes",
#     "experiment.title", "experiment.alias", "experiment.library_name", "experiment.design_description",
#     "experiment.library_construction_protocol", "experiment.attributes", "sample.description",
#     "sample.title", "sample.alias", "sample.attributes", "submission.bioprojectsource", "run.filename",
#     "run.semantic_name", "run.total_bases", "run.total_spots", "experiment.library_strategy",
#     "experiment.library_source", "experiment.library_selection", "experiment.library_layout",
#     "experiment.platform", "experiment.instrument_model", "experiment.spot_descriptor", "study.alias",
#     "study.type", "study.abstract", "study.attributes", "study.PMIDs", "GEOsample.accession",
#     "GEOsample.title", "GEOsample.dataprocessing", "GEOsample.source", "GEOsample.treatmentprotocol",
#     "GEOsample.extractprotocol", "GEOsample.growthprotocol", "GEOsample.characteristics",
#     "run.alias", "run.read_lengths", "run.base_counts", "run.r1_length", "run.r2_length",
#     "run.r3_length", "run.r4_length", "run.Acount", "run.Ccount", "run.Gcount", "run.Tcount", "run.Ncount",
#     "run.experiment", "run.pool_member", "experiment.study_ref", "sample.centername", 
#     "submission.accession", "submission.srasource"
# )


### move these earlier to before run.filename
#GEOsample.accession	GEOsample.title	GEOsample.dataprocessing	GEOsample.source	GEOsample.treatmentprotocol	GEOsample.extractprotocol	GEOsample.growthprotocol	GEOsample.characteristics

# (seqdetective.library_layout_curation, .after = technology_curation)


# desired_order14 <- c(
#     "run.accession", "study.accession", "bioproject", "study.title", "sample.accession",
#     "experiment.accession", "earliest_date", "run.title", "run.alias", "run.attributes",
#     "experiment.title", "experiment.alias", "experiment.library_name", "experiment.design_description",
#     "experiment.library_construction_protocol", "experiment.attributes", "sample.description",
#     "sample.title", "sample.alias", "sample.attributes", "submission.bioprojectsource", "run.filename",
#     "run.semantic_name", "run.total_bases", "run.total_spots", "experiment.library_strategy",
#     "experiment.library_source", "experiment.library_selection", "experiment.library_layout",
#     "experiment.platform", "experiment.instrument_model", "experiment.spot_descriptor", "study.alias",
#     "study.type", "study.abstract", "study.attributes", "study.PMIDs", "GEOsample.accession",
#     "GEOsample.title", "GEOsample.dataprocessing", "GEOsample.source", "GEOsample.treatmentprotocol",
#     "GEOsample.extractprotocol", "GEOsample.growthprotocol", "GEOsample.characteristics", "GEO_studyID",
#     "GEO_sampleID_withrun", "run.read_lengths", "run.base_counts", "run.r1_length", "run.r2_length",
#     "run.r3_length", "run.r4_length", "run.Acount", "run.Ccount", "run.Gcount", "run.Tcount", "run.Ncount",
#     "run.experiment", "run.pool_member", "experiment.study_ref", "sample.centername", 
#     "submission.accession", "submission.srasource", "run.enafirstpublic_date",
#     "run.enalastupdate_date", "run.lastSRAFile_date", "run.published_date", "run.SRAFile_date",
#     "sample.enafirstpublic_date", "sample.enafirstpublic2_date", "sample.enalastupdate_date",
#     "sample.enalastupdate2_date", "sample.insdcfirstpublic_date", "sample.insdclastupdate_date",
#     "study.enafirstpublic_date", "study.enalastupdate_date", "GEOsample.submission_date",
#     "GEOsample.release_date", "GEOsample.lastupdate_date"
# )

In [121]:
## move by select

metadata_curationstep14fewer <- metadata_curationstep14fewer %>%
  select(all_of(desired_order14))


In [122]:

# ## also 9 more to get everything up to study.abstract into first 50 columns???
# metadata_curationstep14fewer <- metadata_curationstep14fewer %>% relocate(run.title, .before = experiment.title_devstage)
# metadata_curationstep14fewer <- metadata_curationstep14fewer %>% relocate(run.attributes, .before = experiment.title_devstage)
# metadata_curationstep14fewer <- metadata_curationstep14fewer %>% relocate(run.filename, .before = experiment.title_devstage)
# metadata_curationstep14fewer <- metadata_curationstep14fewer %>% relocate(run.semantic_name, .before = experiment.title_devstage)
# metadata_curationstep14fewer <- metadata_curationstep14fewer %>% relocate(run.base_counts, .before = experiment.title_devstage)
# metadata_curationstep14fewer <- metadata_curationstep14fewer %>% relocate(run.Acount, .before = experiment.title_devstage)
# metadata_curationstep14fewer <- metadata_curationstep14fewer %>% relocate(run.Ccount, .before = experiment.title_devstage)
# metadata_curationstep14fewer <- metadata_curationstep14fewer %>% relocate(run.Gcount, .before = experiment.title_devstage)
# metadata_curationstep14fewer <- metadata_curationstep14fewer %>% relocate(run.Tcount, .before = experiment.title_devstage)
# metadata_curationstep14fewer <- metadata_curationstep14fewer %>% relocate(run.Ncount, .before = experiment.title_devstage)


# ## lastly move these to near end (after study.abstract)
# metadata_curationstep14fewer <- metadata_curationstep14fewer %>% relocate(run.experiment, .before = experiment.title_devstage)
# metadata_curationstep14fewer <- metadata_curationstep14fewer %>% relocate(run.pool_member, .before = experiment.title_devstage)
# metadata_curationstep14fewer <- metadata_curationstep14fewer %>% relocate(run.alias, .before = experiment.title_devstage)
# metadata_curationstep14fewer <- metadata_curationstep14fewer %>% relocate(experiment.alias, .before = experiment.title_devstage)
# metadata_curationstep14fewer <- metadata_curationstep14fewer %>% relocate(experiment.study_ref, .before = experiment.title_devstage)
# metadata_curationstep14fewer <- metadata_curationstep14fewer %>% relocate(sample.alias, .before = experiment.title_devstage)
# metadata_curationstep14fewer <- metadata_curationstep14fewer %>% relocate(study.alias, .before = experiment.title_devstage)


In [123]:
colnames(metadata_curationstep14fewer)

[1] "run.accession"                           
 [2] "study.accession"                         
 [3] "bioproject"                              
 [4] "study.title"                             
 [5] "sample.accession"                        
 [6] "experiment.accession"                    
 [7] "earliest_date"                           
 [8] "devstage_curation"                       
 [9] "devstage_curation_coarse"                
[10] "tissue_curation"                         
[11] "tissue_curation_coarse"                  
[12] "seqdetective.library_layout_curation"    
[13] "technology_curation"                     
[14] "run.title"                               
[15] "run.attributes"                          
[16] "experiment.title"                        
[17] "experiment.alias"                        
[18] "experiment.library_name"                 
[19] "experiment.design_description"           
[20] "experiment.library_construction_protocol"
[21] "experiment.attributes"                   
[22] "sample.description"                      
[23] "sample.title"                            
[24] "sample.alias"                            
[25] "sample.attributes"                       
[26] "submission.bioprojectsource"             
[27] "submission.bioprojectsource.country"     
[28] "GEOsample.title"                         
[29] "GEOsample.dataprocessing"                
[30] "GEOsample.source"                        
[31] "GEOsample.treatmentprotocol"             
[32] "GEOsample.extractprotocol"               
[33] "GEOsample.growthprotocol"                
[34] "GEOsample.characteristics"               
[35] "GEOsample.accession"                     
[36] "run.filename"                            
[37] "run.semantic_name"                       
[38] "run.total_bases"                         
[39] "run.total_spots"                         
[40] "experiment.library_strategy"             
[41] "experiment.library_source"               
[42] "experiment.library_selection"            
[43] "experiment.library_layout"               
[44] "experiment.platform"                     
[45] "experiment.instrument_model"             
[46] "experiment.spot_descriptor"              
[47] "study.alias"                             
[48] "study.type"                              
[49] "study.abstract"                          
[50] "study.attributes"                        
[51] "study.PMIDs"                             
[52] "run.alias"                               
[53] "run.read_lengths"                        
[54] "run.base_counts"                         
[55] "run.r1_length"                           
[56] "run.r2_length"                           
[57] "run.r3_length"                           
[58] "run.r4_length"                           
[59] "run.Acount"                              
[60] "run.Ccount"                              
[61] "run.Gcount"                              
[62] "run.Tcount"                              
[63] "run.Ncount"                              
[64] "run.experiment"                          
[65] "run.pool_member"                         
[66] "experiment.study_ref"                    
[67] "sample.centername"                       
[68] "submission.accession"                    
[69] "submission.srasource"

In [124]:
## get summaries of categories
metadata_curationstep14fewer %>% group_by(devstage_curation) %>% summarize(count=n()) %>% arrange(desc(count))
metadata_curationstep14fewer %>% group_by(devstage_curation_coarse) %>% summarize(count=n()) %>% arrange(desc(count))
metadata_curationstep14fewer %>% group_by(tissue_curation) %>% summarize(count=n()) %>% arrange(desc(count))
metadata_curationstep14fewer %>% group_by(tissue_curation_coarse) %>% summarize(count=n()) %>% arrange(desc(count))


devstage_curation,count
<chr>,<int>
Larval,19709
Adult,15670
Undetermined,10064
Pharyngula,7444
Hatching,5782
Gastrula,4676
Multi-stage,4114
Segmentation,3626
Juvenile,2710


devstage_curation_coarse,count
<chr>,<int>
Embryo,30056
Larval,21006
Adult,16849
Undetermined,4327
Juvenile,2716
Multi-stage,2375


tissue_curation,count
<chr>,<int>
Whole Organism,25682
Embryo Imprecise,8973
Brain,6454
Multi-tissue,5171
Undetermined,4325
Trunk,3555
Heart,3173
Pancreas,2522
Spinal Cord,2275


tissue_curation_coarse,count
<chr>,<int>
All anatomical structures,34655
Nervous System,9567
Multi-system,6009
Surface Structure,5350
Cardiovascular System,3611
Endocrine System,2612
Embryo Imprecise,2580
Liver and Biliary System,1826
Sensory System,1745


In [125]:
write.table(metadata_curationstep14fewer, file = paste0("all_zf_dates_devstage_tissue_tech_curated.tsv"), sep = "\t", row.names = FALSE, quote = FALSE)


In [126]:
unique(metadata_curationstep14fewer$tissue_curation)
unique(metadata_curationstep14fewer$tissue_curation_coarse)

[1] "Undetermined"       "Gut"                "Brain"             
 [4] "Spinal Cord"        "Gonad"              "Multi-tissue"      
 [7] "Whole Organism"     "Fin"                "Trunk"             
[10] "Liver"              "Muscle"             "Heart"             
[13] "Gill"               "Pancreas"           "Eye"               
[16] "Embryo Imprecise"   "Oocyte"             "Skin"              
[19] "Spleen"             "Head"               "Swim Bladder"      
[22] "Kidney"             "Blood"              "Scale"             
[25] "Cell Line"          "Tail"               "Pharyngeal Arch"   
[28] "Bone or Cartilage"  "Endothelium"        "Lateral Line"      
[31] "Cancer or Tumor"    "Thymus"             "Pituitary Gland"   
[34] "Jaw"                "Adipose Tissue"     "Pineal Gland"      
[37] "Ear"                "Thyroid"            "Lymphatic System"  
[40] "Marrow"             "BCR TCR repertoire"

[1] "Embryo Imprecise"          "Undetermined"             
 [3] "Digestive System"          "Nervous System"           
 [5] "Reproductive System"       "Multi-system"             
 [7] "All anatomical structures" "Surface Structure"        
 [9] "Liver and Biliary System"  "Muscular System"          
[11] "Cardiovascular System"     "Respiratory System"       
[13] "Endocrine System"          "Sensory System"           
[15] "Hematopoietic System"      "Swim Bladder"             
[17] "Renal System"              "Cell Line"                
[19] "Skeletal Element"          "Cancer or Tumor"          
[21] "Adipose Tissue"

In [127]:
## also make full set of columns for LLMs...instead use 'rawcolumns' from above
# ## pulling intermediate columns
# metadata_curationstep14toadd <- metadata_curationstep9 %>% select(run.accession,43:50)

# ## join then move to before experiment.title
# metadata_curationstep14withintermediates <- left_join(metadata_curationstep14,metadata_curationstep14toadd)

# metadata_curationstep14withintermediates <- metadata_curationstep14withintermediates %>% relocate(85:92, .before = experiment.title)

# metadata_curationstep14withintermediates
# write.table(metadata_curationstep14withintermediates, file = paste0("all_zf_dates_devstage_tissue_tech_curated_withGEO_morecolumnsandintermediates.tsv"), sep = "\t", row.names = FALSE, quote = FALSE)


# ## first saving a fullcolumns version then removing ~10 columns
write.table(metadata_curationstep14b, file = paste0("all_zf_dates_devstage_tissue_tech_curated_rawcolumns.tsv"), sep = "\t", row.names = FALSE, quote = FALSE)


In [128]:

# ## pulling unknown in either coarse devstage or tissue
# ## latest 2025 'unknown' set - where either  devstage_curation_runexperimentsampleGEOsample_alsostudy_withmultistage_coarse or tissue_curation_runexperimentsampleGEOsample_alsostudy_withmultistage_coarse
# ## are undetermined

# ## filter 
# metadata_curationstep13_unknown <- metadata_curationstep13fewer2 %>% filter(devstage_curation_runexperimentsampleGEOsample_alsostudy_withmultistage_coarse == "Undetermined" | tissue_curation_runexperimentsampleGEOsample_alsostudy_withmultistage_coarse == "Undetermined")
# ## 2409 of 5444 have pubmed pmids!
# write.table(metadata_curationstep13_unknown, file = paste0("all_zf_dates_devstage_tissue_tech_curated_withGEO_unknowndevstageortissue.tsv"), sep = "\t", row.names = FALSE, quote = FALSE)

## separate devstage & tissue unknowns
metadata_curationstep14fewer_unknowndev <- metadata_curationstep14fewer %>% filter(devstage_curation_coarse == "Undetermined")
metadata_curationstep14fewer_unknowntis <- metadata_curationstep14fewer %>% filter(tissue_curation_coarse == "Undetermined")
write.table(metadata_curationstep14fewer_unknowndev, file = paste0("all_zf_dates_devstage_tissue_tech_curated_unknowndevstage.tsv"), sep = "\t", row.names = FALSE, quote = FALSE)
write.table(metadata_curationstep14fewer_unknowntis, file = paste0("all_zf_dates_devstage_tissue_tech_curated_unknowntissue.tsv"), sep = "\t", row.names = FALSE, quote = FALSE)
metadata_curationstep14fewer_unknowndev
metadata_curationstep14fewer_unknowntis

run.accession,study.accession,bioproject,study.title,sample.accession,experiment.accession,earliest_date,devstage_curation,devstage_curation_coarse,tissue_curation,⋯,run.Ccount,run.Gcount,run.Tcount,run.Ncount,run.experiment,run.pool_member,experiment.study_ref,sample.centername,submission.accession,submission.srasource
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<chr>,<chr>,<chr>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
DRR050167,DRP003043,PRJDB4416,Transcriptome analysis in ES1 expressing rods of zebrafish,DRS025834,DRX045209,2016-03-09,Undetermined,Undetermined,Fin,⋯,79475572,83520360,101561779,0,DRX045209,DRS025834,DRP003043,NA,DRA004229,"OSAKA_FB|Kawamura lab., Frontier Biology, Osaka univ."
DRR050166,DRP003043,PRJDB4416,Transcriptome analysis in ES1 expressing rods of zebrafish,DRS025833,DRX045208,2016-03-09,Undetermined,Undetermined,Fin,⋯,86415371,90822163,121670773,0,DRX045208,DRS025833,DRP003043,NA,DRA004229,"OSAKA_FB|Kawamura lab., Frontier Biology, Osaka univ."
DRR050165,DRP003043,PRJDB4416,Transcriptome analysis in ES1 expressing rods of zebrafish,DRS025832,DRX045207,2016-03-09,Undetermined,Undetermined,Fin,⋯,129885620,136727379,153657736,0,DRX045207,DRS025832,DRP003043,NA,DRA004229,"OSAKA_FB|Kawamura lab., Frontier Biology, Osaka univ."
DRR050164,DRP003043,PRJDB4416,Transcriptome analysis in ES1 expressing rods of zebrafish,DRS025831,DRX045206,2016-03-09,Undetermined,Undetermined,Fin,⋯,73070267,77738821,92783592,0,DRX045206,DRS025831,DRP003043,NA,DRA004229,"OSAKA_FB|Kawamura lab., Frontier Biology, Osaka univ."
DRR050163,DRP003043,PRJDB4416,Transcriptome analysis in ES1 expressing rods of zebrafish,DRS025830,DRX045205,2016-03-09,Undetermined,Undetermined,Fin,⋯,113928273,120051739,146481242,0,DRX045205,DRS025830,DRP003043,NA,DRA004229,"OSAKA_FB|Kawamura lab., Frontier Biology, Osaka univ."
DRR050162,DRP003043,PRJDB4416,Transcriptome analysis in ES1 expressing rods of zebrafish,DRS025829,DRX045204,2016-03-09,Undetermined,Undetermined,Fin,⋯,104872341,113114502,123927858,0,DRX045204,DRS025829,DRP003043,NA,DRA004229,"OSAKA_FB|Kawamura lab., Frontier Biology, Osaka univ."
DRR051067,DRP003977,PRJDB4470,Gene expression analysis of the zebrafish brain,DRS057267,DRX045959,2018-01-06,Undetermined,Undetermined,Brain,⋯,3957379257,3797879751,5144366288,4618438,DRX045959,DRS057267,DRP003977,NA,DRA004277,NIG|National Institute of Genetics (Japan)
DRR051063,DRP003977,PRJDB4470,Gene expression analysis of the zebrafish brain,DRS057266,DRX045955,2018-01-06,Undetermined,Undetermined,Brain,⋯,3082552426,3134669086,4606055524,5037798,DRX045955,DRS057266,DRP003977,NA,DRA004273,NIG|National Institute of Genetics (Japan)
ERR1125090,ERP013160,PRJEB11748,Analysis of single CD41 cell transcriptomes from Zebrafish,ERS959300,ERX1204289,2016-01-08,Undetermined,Undetermined,Multi-tissue,⋯,119943911,107010121,144702244,58425,ERX1204289,ERS959300,ERP013160,EMBL-EBI,ERA533479,EMBL-EBI|European Nucleotide Archive


run.accession,study.accession,bioproject,study.title,sample.accession,experiment.accession,earliest_date,devstage_curation,devstage_curation_coarse,tissue_curation,⋯,run.Ccount,run.Gcount,run.Tcount,run.Ncount,run.experiment,run.pool_member,experiment.study_ref,sample.centername,submission.accession,submission.srasource
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<chr>,<chr>,<chr>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
DRR334977,DRP008001,PRJDB12578,Allometric scaling of RNA abundance from genes to communities,DRS217313,DRX323973,2021-12-23,Adult,Adult,Undetermined,⋯,926887571,898697642,998107868,174,DRX323973,DRS217313,DRP008001,NA,DRA013226,"SINICA|Machida Laboratory Biodiversity Research Center, Academia Sinica"
DRR334976,DRP008001,PRJDB12578,Allometric scaling of RNA abundance from genes to communities,DRS217312,DRX323972,2021-12-23,Adult,Adult,Undetermined,⋯,702936549,679920984,718646067,599,DRX323972,DRS217312,DRP008001,NA,DRA013226,"SINICA|Machida Laboratory Biodiversity Research Center, Academia Sinica"
DRR334975,DRP008001,PRJDB12578,Allometric scaling of RNA abundance from genes to communities,DRS217311,DRX323971,2021-12-23,Adult,Adult,Undetermined,⋯,791315678,766594599,805780465,458,DRX323971,DRS217311,DRP008001,NA,DRA013226,"SINICA|Machida Laboratory Biodiversity Research Center, Academia Sinica"
DRR334974,DRP008001,PRJDB12578,Allometric scaling of RNA abundance from genes to communities,DRS217310,DRX323970,2021-12-23,Adult,Adult,Undetermined,⋯,733754277,715270279,711475544,386,DRX323970,DRS217310,DRP008001,NA,DRA013226,"SINICA|Machida Laboratory Biodiversity Research Center, Academia Sinica"
DRR334973,DRP008001,PRJDB12578,Allometric scaling of RNA abundance from genes to communities,DRS217309,DRX323969,2021-12-23,Adult,Adult,Undetermined,⋯,825478743,804835148,792466124,199,DRX323969,DRS217309,DRP008001,NA,DRA013226,"SINICA|Machida Laboratory Biodiversity Research Center, Academia Sinica"
DRR334972,DRP008001,PRJDB12578,Allometric scaling of RNA abundance from genes to communities,DRS217308,DRX323968,2021-12-23,Adult,Adult,Undetermined,⋯,954687406,929537615,895107890,417,DRX323968,DRS217308,DRP008001,NA,DRA013226,"SINICA|Machida Laboratory Biodiversity Research Center, Academia Sinica"
DRR334971,DRP008001,PRJDB12578,Allometric scaling of RNA abundance from genes to communities,DRS217307,DRX323967,2021-12-23,Adult,Adult,Undetermined,⋯,738814269,712866126,793876235,343,DRX323967,DRS217307,DRP008001,NA,DRA013226,"SINICA|Machida Laboratory Biodiversity Research Center, Academia Sinica"
DRR334970,DRP008001,PRJDB12578,Allometric scaling of RNA abundance from genes to communities,DRS217306,DRX323966,2021-12-23,Adult,Adult,Undetermined,⋯,794741415,771082024,784363609,170,DRX323966,DRS217306,DRP008001,NA,DRA013226,"SINICA|Machida Laboratory Biodiversity Research Center, Academia Sinica"
DRR334969,DRP008001,PRJDB12578,Allometric scaling of RNA abundance from genes to communities,DRS217305,DRX323965,2021-12-23,Adult,Adult,Undetermined,⋯,814578518,787469614,811734581,210,DRX323965,DRS217305,DRP008001,NA,DRA013226,"SINICA|Machida Laboratory Biodiversity Research Center, Academia Sinica"


In [129]:
metadata_curationstep14fewer_unknown <- metadata_curationstep14fewer %>% filter(devstage_curation_coarse == "Undetermined" | tissue_curation_coarse == "Undetermined")
metadata_curationstep14fewer_unknown
write.table(metadata_curationstep14fewer_unknown, file = paste0("all_zf_dates_devstage_tissue_tech_curated_unknown.tsv"), sep = "\t", row.names = FALSE, quote = FALSE)


run.accession,study.accession,bioproject,study.title,sample.accession,experiment.accession,earliest_date,devstage_curation,devstage_curation_coarse,tissue_curation,⋯,run.Ccount,run.Gcount,run.Tcount,run.Ncount,run.experiment,run.pool_member,experiment.study_ref,sample.centername,submission.accession,submission.srasource
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<date>,<chr>,<chr>,<chr>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
DRR334977,DRP008001,PRJDB12578,Allometric scaling of RNA abundance from genes to communities,DRS217313,DRX323973,2021-12-23,Adult,Adult,Undetermined,⋯,926887571,898697642,998107868,174,DRX323973,DRS217313,DRP008001,NA,DRA013226,"SINICA|Machida Laboratory Biodiversity Research Center, Academia Sinica"
DRR334976,DRP008001,PRJDB12578,Allometric scaling of RNA abundance from genes to communities,DRS217312,DRX323972,2021-12-23,Adult,Adult,Undetermined,⋯,702936549,679920984,718646067,599,DRX323972,DRS217312,DRP008001,NA,DRA013226,"SINICA|Machida Laboratory Biodiversity Research Center, Academia Sinica"
DRR334975,DRP008001,PRJDB12578,Allometric scaling of RNA abundance from genes to communities,DRS217311,DRX323971,2021-12-23,Adult,Adult,Undetermined,⋯,791315678,766594599,805780465,458,DRX323971,DRS217311,DRP008001,NA,DRA013226,"SINICA|Machida Laboratory Biodiversity Research Center, Academia Sinica"
DRR334974,DRP008001,PRJDB12578,Allometric scaling of RNA abundance from genes to communities,DRS217310,DRX323970,2021-12-23,Adult,Adult,Undetermined,⋯,733754277,715270279,711475544,386,DRX323970,DRS217310,DRP008001,NA,DRA013226,"SINICA|Machida Laboratory Biodiversity Research Center, Academia Sinica"
DRR334973,DRP008001,PRJDB12578,Allometric scaling of RNA abundance from genes to communities,DRS217309,DRX323969,2021-12-23,Adult,Adult,Undetermined,⋯,825478743,804835148,792466124,199,DRX323969,DRS217309,DRP008001,NA,DRA013226,"SINICA|Machida Laboratory Biodiversity Research Center, Academia Sinica"
DRR334972,DRP008001,PRJDB12578,Allometric scaling of RNA abundance from genes to communities,DRS217308,DRX323968,2021-12-23,Adult,Adult,Undetermined,⋯,954687406,929537615,895107890,417,DRX323968,DRS217308,DRP008001,NA,DRA013226,"SINICA|Machida Laboratory Biodiversity Research Center, Academia Sinica"
DRR334971,DRP008001,PRJDB12578,Allometric scaling of RNA abundance from genes to communities,DRS217307,DRX323967,2021-12-23,Adult,Adult,Undetermined,⋯,738814269,712866126,793876235,343,DRX323967,DRS217307,DRP008001,NA,DRA013226,"SINICA|Machida Laboratory Biodiversity Research Center, Academia Sinica"
DRR334970,DRP008001,PRJDB12578,Allometric scaling of RNA abundance from genes to communities,DRS217306,DRX323966,2021-12-23,Adult,Adult,Undetermined,⋯,794741415,771082024,784363609,170,DRX323966,DRS217306,DRP008001,NA,DRA013226,"SINICA|Machida Laboratory Biodiversity Research Center, Academia Sinica"
DRR334969,DRP008001,PRJDB12578,Allometric scaling of RNA abundance from genes to communities,DRS217305,DRX323965,2021-12-23,Adult,Adult,Undetermined,⋯,814578518,787469614,811734581,210,DRX323965,DRS217305,DRP008001,NA,DRA013226,"SINICA|Machida Laboratory Biodiversity Research Center, Academia Sinica"


In [130]:
n_distinct(metadata_curationstep14fewer$bioproject)
n_distinct(metadata_curationstep14fewer$submission.accession)

[1] 2469

[1] 3032